<a href="https://colab.research.google.com/github/amzad-786githumb/Privacy-Preserving-Synthetic-Tabular-Data-Generation-Using-Generative-Adversarial-Networks/blob/main/08_Comparative_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# 8.1 Environment Setup
# Block 1 : Install Required Packages
# =============================================================================

print("=" * 80)
print("Installing Required Packages")
print("=" * 80)

# Core
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn tqdm pyyaml joblib

# Deep Learning
!pip -q install torch torchvision torchaudio

# Synthetic Data
!pip -q install sdv ctgan copulas sdmetrics

# Privacy
!pip -q install opacus diffprivlib

# Statistical Analysis
!pip -q install statsmodels pingouin

# Gradient Boosting
!pip -q install xgboost lightgbm

# Dimensionality Reduction
!pip -q install umap-learn

print("\nAll packages installed successfully.")

In [ ]:
# =============================================================================
# Block 2 : Imports
# =============================================================================

# Standard Library
import os
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

# Numerical Computing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Machine Learning
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler
)

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score

)

from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC

from sklearn.decomposition import PCA

from sklearn.manifold import TSNE

# Gradient Boosting
from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

# Deep Learning
import torch
import torch.nn as nn

# Synthetic Data Evaluation
from sdmetrics.reports.single_table import QualityReport

# Progress Bar
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

In [ ]:
# =============================================================================
# Block 3 : Mount Google Drive
# =============================================================================

from google.colab import drive
from pathlib import Path

print("=" * 80)
print("Mounting Google Drive")
print("=" * 80)

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Project"
)

print("\nGoogle Drive Mounted Successfully")

print(f"\nProject Root : {PROJECT_ROOT}")

if PROJECT_ROOT.exists():

    print("Project Directory Found")

else:

    raise FileNotFoundError(

        f"Project directory not found:\n{PROJECT_ROOT}"

    )

In [ ]:
# =============================================================================
# Block 4 : Random Seed
# =============================================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

print("=" * 80)
print("Random Seed Initialized")
print("=" * 80)

print(f"Seed : {SEED}")

In [ ]:
# =============================================================================
# Block 5 : GPU Detection
# =============================================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print("=" * 80)
print("GPU Detection")
print("=" * 80)

print(f"Device : {DEVICE}")

if DEVICE.type == "cuda":

    print(f"GPU : {torch.cuda.get_device_name(0)}")

    print(f"CUDA Version : {torch.version.cuda}")

    print(f"GPU Count : {torch.cuda.device_count()}")

else:

    print("Running on CPU")

In [ ]:
# =============================================================================
# Block 6 : Project Paths
# =============================================================================

CONFIG_PATH = PROJECT_ROOT / "config.yaml"

DATASET_DIR = PROJECT_ROOT / "datasets"

MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR = PROJECT_ROOT / "results"

EVALUATION_DIR = RESULTS_DIR / "evaluation"

FIGURE_DIR = RESULTS_DIR / "figures"

TABLE_DIR = RESULTS_DIR / "tables"

REPORT_DIR = RESULTS_DIR / "reports"

for directory in [

    EVALUATION_DIR,

    FIGURE_DIR,

    TABLE_DIR,

    REPORT_DIR

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

print("=" * 80)
print("Project Paths Configured")
print("=" * 80)

print(f"Config Path      : {CONFIG_PATH}")
print(f"Dataset Path     : {DATASET_DIR}")
print(f"Models Path      : {MODELS_DIR}")
print(f"Results Path     : {RESULTS_DIR}")
print(f"Evaluation Path  : {EVALUATION_DIR}")
print(f"Figures Path     : {FIGURE_DIR}")
print(f"Tables Path      : {TABLE_DIR}")
print(f"Reports Path     : {REPORT_DIR}")

In [ ]:
# =============================================================================
# Block 6 : Environment Verification
# =============================================================================

print("=" * 80)
print("Environment Verification")
print("=" * 80)

print(f"Python Version        : {os.sys.version.split()[0]}")

print(f"PyTorch Version       : {torch.__version__}")

print(f"NumPy Version         : {np.__version__}")

print(f"Pandas Version        : {pd.__version__}")

print(f"Project Root Exists   : {PROJECT_ROOT.exists()}")

print(f"Config File Exists    : {CONFIG_PATH.exists()}")

print(f"Dataset Directory     : {DATASET_DIR.exists()}")

print(f"Results Directory     : {RESULTS_DIR.exists()}")

print(f"Models Directory      : {MODELS_DIR.exists()}")

print(f"Evaluation Directory  : {EVALUATION_DIR.exists()}")

print(f"Figures Directory     : {FIGURE_DIR.exists()}")

print(f"Tables Directory      : {TABLE_DIR.exists()}")

print(f"Reports Directory     : {REPORT_DIR.exists()}")

print(f"Device                : {DEVICE}")

print("=" * 80)
print("Environment Ready for Comparative Evaluation")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.2 Load Project Configuration
# Block 1 : Load config.yaml
# =============================================================================

import yaml

print("=" * 80)
print("Loading Project Configuration")
print("=" * 80)

if not CONFIG_PATH.exists():

    raise FileNotFoundError(

        f"Configuration file not found:\n{CONFIG_PATH}"

    )

with open(CONFIG_PATH, "r") as file:

    CONFIG = yaml.safe_load(file)

print("Configuration Loaded Successfully")

In [ ]:
# =============================================================================
# Block 2 : Experiment Configuration
# =============================================================================

print("=" * 80)
print("Experiment Configuration")
print("=" * 80)

SEED = CONFIG.get("seed", 42)

DEVICE = CONFIG.get(

    "device",

    "cuda" if torch.cuda.is_available() else "cpu"

)

BATCH_SIZE = CONFIG.get("batch_size", 256)

EPOCHS = CONFIG.get("epochs", 300)

LATENT_DIM = CONFIG.get("latent_dimension", 128)

LEARNING_RATE = CONFIG.get("learning_rate", 0.0002)

BETA1 = CONFIG.get("beta1", 0.5)

BETA2 = CONFIG.get("beta2", 0.999)

print(f"Seed               : {SEED}")

print(f"Device             : {DEVICE}")

print(f"Batch Size         : {BATCH_SIZE}")

print(f"Epochs             : {EPOCHS}")

print(f"Latent Dimension   : {LATENT_DIM}")

print(f"Learning Rate      : {LEARNING_RATE}")

print(f"Beta1              : {BETA1}")

print(f"Beta2              : {BETA2}")

In [ ]:
# =============================================================================
# Block 3 : Privacy Configuration
# =============================================================================

print("=" * 80)
print("Privacy Configuration")
print("=" * 80)

PRIVACY_BUDGET = CONFIG.get(

    "privacy_budget",

    4.0

)

DELTA = CONFIG.get(

    "delta",

    1e-5

)

NOISE_MULTIPLIER = CONFIG.get(

    "noise_multiplier",

    1.1

)

MAX_GRAD_NORM = CONFIG.get(

    "gradient_clip",

    1.0

)

print(f"Privacy Budget (ε) : {PRIVACY_BUDGET}")

print(f"Delta              : {DELTA}")

print(f"Noise Multiplier   : {NOISE_MULTIPLIER}")

print(f"Gradient Clip      : {MAX_GRAD_NORM}")

In [ ]:
# =============================================================================
# Block 4 : Evaluation Configuration
# =============================================================================

print("=" * 80)
print("Evaluation Configuration")
print("=" * 80)

MODEL_NAMES = [

    "Gaussian Multivariate",

    "Gaussian Copula",

    "CTGAN",

    "TVAE",

    "DP-CTGAN",

    "SPP-GAN"

]

DATASET_NAMES = [

    "Adult Income",

    "Bank Marketing",

    "Breast Cancer"

]

CLASSIFIERS = {

    "Logistic Regression",

    "Random Forest",

    "Support Vector Machine",

    "XGBoost",

    "LightGBM"

}

FIDELITY_METRICS = [

    "KS Test",

    "Wasserstein Distance",

    "Jensen-Shannon Divergence",

    "KL Divergence",

    "Correlation Difference"

]

UTILITY_METRICS = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC-AUC"

]

PRIVACY_METRICS = [

    "Membership Inference",

    "Re-identification Risk",

    "Attribute Disclosure"

]

print(f"Models                 : {len(MODEL_NAMES)}")

print(f"Datasets               : {len(DATASET_NAMES)}")

print(f"Classification Models  : {len(CLASSIFIERS)}")

print(f"Fidelity Metrics       : {len(FIDELITY_METRICS)}")

print(f"Utility Metrics        : {len(UTILITY_METRICS)}")

print(f"Privacy Metrics        : {len(PRIVACY_METRICS)}")

In [ ]:
# =============================================================================
# Block 5 : Project Paths
# =============================================================================

DATASET_DIR = PROJECT_ROOT / "datasets"

MODEL_DIR = PROJECT_ROOT / "models"

RESULTS_DIR = PROJECT_ROOT / "results"

SYNTHETIC_DATA_DIR = RESULTS_DIR / "synthetic_data"

EVALUATION_DIR = RESULTS_DIR / "evaluation"

FIGURE_DIR = RESULTS_DIR / "figures"

TABLE_DIR = RESULTS_DIR / "tables"

REPORT_DIR = RESULTS_DIR / "reports"

directories = {

    "Datasets": DATASET_DIR,

    "Models": MODEL_DIR,

    "Results": RESULTS_DIR,

    "Synthetic Data": SYNTHETIC_DATA_DIR,

    "Evaluation": EVALUATION_DIR,

    "Figures": FIGURE_DIR,

    "Tables": TABLE_DIR,

    "Reports": REPORT_DIR

}

for directory in directories.values():

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

print("=" * 80)
print("Project Paths")
print("=" * 80)

for name, path in directories.items():

    print(f"{name:<18} : {path}")

In [ ]:
# =============================================================================
# Block 6 : Configuration Summary
# =============================================================================

configuration_summary = {

    "Seed": SEED,

    "Device": DEVICE,

    "Batch Size": BATCH_SIZE,

    "Epochs": EPOCHS,

    "Latent Dimension": LATENT_DIM,

    "Learning Rate": LEARNING_RATE,

    "Privacy Budget": PRIVACY_BUDGET,

    "Noise Multiplier": NOISE_MULTIPLIER,

    "Gradient Clip": MAX_GRAD_NORM,

    "Number of Models": len(MODEL_NAMES),

    "Number of Datasets": len(DATASET_NAMES)

}

print("=" * 80)
print("Configuration Summary")
print("=" * 80)

for key, value in configuration_summary.items():

    print(f"{key:<22} : {value}")

print("\nProject Configuration Loaded Successfully.")

In [ ]:
# =============================================================================
# 8.3 Load Original Datasets
# Block 1 : Locate Dataset Split Files
# =============================================================================

print("=" * 80)
print("Locating Original Dataset Splits")
print("=" * 80)

DATASET_SPLITS = {}

dataset_folders = {

    "Adult Income": [
        DATASET_DIR / "splits" / "adult_income",
        DATASET_DIR / "splits" / "Adult Income"
    ],

    "Bank Marketing": [
        DATASET_DIR / "splits" / "bank_marketing",
        DATASET_DIR / "splits" / "Bank Marketing"
    ],

    "Breast Cancer": [
        DATASET_DIR / "splits" / "breast_cancer",
        DATASET_DIR / "splits" / "Breast Cancer"
    ]

}

for dataset_name, candidates in dataset_folders.items():

    selected_folder = None

    for folder in candidates:

        if folder.exists():

            selected_folder = folder

            break

    if selected_folder is None:

        raise FileNotFoundError(

            f"Dataset folder not found for {dataset_name}"

        )

    DATASET_SPLITS[dataset_name] = {

        "train": selected_folder / "train.csv",

        "validation": selected_folder / "validation.csv",

        "test": selected_folder / "test.csv"

    }

    print(f"{dataset_name:<20} {selected_folder}")

print("\nDataset folders located successfully.")

In [ ]:
# =============================================================================
# Block 2 : Load Original Datasets
# =============================================================================

print("=" * 80)
print("Loading Original Datasets")
print("=" * 80)

original_datasets = {}

for dataset_name, paths in DATASET_SPLITS.items():

    print(f"\nLoading : {dataset_name}")

    train_df = pd.read_csv(paths["train"])

    validation_df = pd.read_csv(paths["validation"])

    test_df = pd.read_csv(paths["test"])

    original_datasets[dataset_name] = {

        "train": train_df,

        "validation": validation_df,

        "test": test_df

    }

    print(f"Train       : {train_df.shape}")

    print(f"Validation  : {validation_df.shape}")

    print(f"Test        : {test_df.shape}")

print("\nAll datasets loaded successfully.")

In [ ]:
# =============================================================================
# Block 3 : Dataset Verification
# =============================================================================

print("=" * 80)
print("Verifying Loaded Datasets")
print("=" * 80)

verification_rows = []

for dataset_name, dataset in original_datasets.items():

    train_df = dataset["train"]

    validation_df = dataset["validation"]

    test_df = dataset["test"]

    verification_rows.append({

        "Dataset": dataset_name,

        "Train Rows": len(train_df),

        "Validation Rows": len(validation_df),

        "Test Rows": len(test_df),

        "Features": train_df.shape[1],

        "Missing Values":

            train_df.isna().sum().sum()

            +

            validation_df.isna().sum().sum()

            +

            test_df.isna().sum().sum()

    })

verification_df = pd.DataFrame(

    verification_rows

)

display(verification_df)

In [ ]:
# =============================================================================
# Block 4 : Dataset Information
# =============================================================================

print("=" * 80)
print("Dataset Summary")
print("=" * 80)

for dataset_name, dataset in original_datasets.items():

    train_df = dataset["train"]

    print(f"\n{dataset_name}")

    print("-" * 60)

    print(f"Features           : {train_df.shape[1]}")

    print(f"Training Samples   : {len(dataset['train'])}")

    print(f"Validation Samples : {len(dataset['validation'])}")

    print(f"Testing Samples    : {len(dataset['test'])}")

    print(f"Column Names")

    print(list(train_df.columns))

In [ ]:
# =============================================================================
# Block 5 : Final Verification
# =============================================================================

print("=" * 80)
print("Original Dataset Loading Completed")
print("=" * 80)

for dataset_name in original_datasets.keys():

    train_shape = original_datasets[dataset_name]["train"].shape

    validation_shape = original_datasets[dataset_name]["validation"].shape

    test_shape = original_datasets[dataset_name]["test"].shape

    print(

        f"{dataset_name:<20}"

        f" Train {train_shape}"

        f" | Validation {validation_shape}"

        f" | Test {test_shape}"

    )

print("\nOriginal datasets are ready for comparative evaluation.")

In [ ]:
# =============================================================================
# 8.4 Load Synthetic Datasets
# Block 1 : Define Synthetic Dataset Locations
# =============================================================================

print("=" * 80)
print("Locating Synthetic Datasets")
print("=" * 80)

MODEL_DIRECTORIES = {

    "Gaussian Multivariate":
        SYNTHETIC_DATA_DIR / "Gaussian_Multivariate",

    "Gaussian Copula":
        SYNTHETIC_DATA_DIR / "Gaussian_Copula",

    "CTGAN":
        SYNTHETIC_DATA_DIR / "CTGAN",

    "TVAE":
        SYNTHETIC_DATA_DIR / "TVAE",

    "DP-CTGAN":
        SYNTHETIC_DATA_DIR / "DP_CTGAN",

    "SPP-GAN":
        SYNTHETIC_DATA_DIR / "SPP_GAN"

}

for model, path in MODEL_DIRECTORIES.items():

    print(f"{model:<25} {path}")

In [ ]:
# =============================================================================
# Block 2 : Load Synthetic Datasets
# =============================================================================

print("=" * 80)
print("Loading Synthetic Datasets")
print("=" * 80)

synthetic_datasets = {}

dataset_aliases = {

    "Adult Income": [

        "adult_income",

        "Adult_Income",

        "Adult Income"

    ],

    "Bank Marketing": [

        "bank_marketing",

        "Bank_Marketing",

        "Bank Marketing"

    ],

    "Breast Cancer": [

        "breast_cancer",

        "Breast_Cancer",

        "Breast Cancer"

    ]

}

for model_name, model_folder in MODEL_DIRECTORIES.items():

    print(f"\nModel : {model_name}")

    synthetic_datasets[model_name] = {}

    if not model_folder.exists():

        print("Directory not found.")

        continue

    for dataset_name, aliases in dataset_aliases.items():

        csv_file = None

        for alias in aliases:

            files = list(model_folder.glob(f"*{alias}*.csv"))

            if len(files) > 0:

                csv_file = files[0]

                break

        if csv_file is None:

            print(f"{dataset_name:<20} Not Found")

            continue

        dataframe = pd.read_csv(csv_file)

        synthetic_datasets[model_name][dataset_name] = dataframe

        print(

            f"{dataset_name:<20}"

            f"{dataframe.shape}"

        )

print("\nSynthetic datasets loaded.")

In [ ]:
# =============================================================================
# Block 3 : Verify Synthetic Datasets
# =============================================================================

print("=" * 80)
print("Verifying Synthetic Datasets")
print("=" * 80)

verification_results = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, dataframe in datasets.items():

        verification_results.append({

            "Model":

                model_name,

            "Dataset":

                dataset_name,

            "Rows":

                dataframe.shape[0],

            "Columns":

                dataframe.shape[1],

            "Missing Values":

                dataframe.isna().sum().sum(),

            "Duplicate Rows":

                dataframe.duplicated().sum()

        })

verification_df = pd.DataFrame(

    verification_results

)

display(verification_df)

In [ ]:
# =============================================================================
# Block 4 : Feature Name Verification
# =============================================================================

print("=" * 80)
print("Verifying Feature Names")
print("=" * 80)

feature_summary = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, synthetic_df in datasets.items():

        original_columns = list(

            original_datasets[dataset_name]["train"].columns

        )

        synthetic_columns = list(

            synthetic_df.columns

        )

        feature_summary.append({

            "Model":

                model_name,

            "Dataset":

                dataset_name,

            "Feature Match":

                original_columns == synthetic_columns,

            "Original Features":

                len(original_columns),

            "Synthetic Features":

                len(synthetic_columns)

        })

feature_summary = pd.DataFrame(

    feature_summary

)

display(feature_summary)

In [ ]:
# =============================================================================
# Block 5 : Verify Data Types
# =============================================================================

print("=" * 80)
print("Verifying Data Types")
print("=" * 80)

datatype_summary = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, dataframe in datasets.items():

        datatype_summary.append({

            "Model":

                model_name,

            "Dataset":

                dataset_name,

            "Numeric Features":

                dataframe.select_dtypes(

                    include=np.number

                ).shape[1],

            "Categorical Features":

                dataframe.select_dtypes(

                    exclude=np.number

                ).shape[1]

        })

datatype_summary = pd.DataFrame(

    datatype_summary

)

display(datatype_summary)

In [ ]:
# =============================================================================
# Block 6 : Synthetic Dataset Summary
# =============================================================================

print("=" * 80)
print("Synthetic Dataset Summary")
print("=" * 80)

for model_name, datasets in synthetic_datasets.items():

    print(f"\n{model_name}")

    print("-" * 60)

    for dataset_name, dataframe in datasets.items():

        print(

            f"{dataset_name:<20}"

            f"Rows={dataframe.shape[0]:<8}"

            f"Columns={dataframe.shape[1]:<4}"

        )

print("\nAll synthetic datasets are ready for evaluation.")

In [ ]:
# =============================================================================
# 8.4 Load Synthetic Datasets
# Block 1 : Synthetic Dataset Paths
# =============================================================================

print("=" * 80)
print("Synthetic Dataset Directories")
print("=" * 80)

MODEL_PATHS = {

    "Gaussian Multivariate":
        RESULTS_DIR / "synthetic_data" / "Gaussian_Multivariate",

    "Gaussian Copula":
        RESULTS_DIR / "synthetic_data" / "Gaussian_Copula",

    "CTGAN":
        RESULTS_DIR / "synthetic_data" / "CTGAN",

    "TVAE":
        RESULTS_DIR / "synthetic_data" / "TVAE",

    "DP-CTGAN":
        RESULTS_DIR / "synthetic_data" / "DP_CTGAN",

    "SPP-GAN":
        RESULTS_DIR / "synthetic_data" / "SPP_GAN"

}

for model, path in MODEL_PATHS.items():

    print(f"{model:<25} {path}")

In [ ]:
# =============================================================================
# Block 2 : Load Synthetic Datasets
# =============================================================================

print("=" * 80)
print("Loading Synthetic Datasets")
print("=" * 80)

synthetic_datasets = {}

dataset_keywords = {

    "Adult Income":
        ["adult_income", "Adult_Income", "Adult Income"],

    "Bank Marketing":
        ["bank_marketing", "Bank_Marketing", "Bank Marketing"],

    "Breast Cancer":
        ["breast_cancer", "Breast_Cancer", "Breast Cancer"]

}

for model_name, model_dir in MODEL_PATHS.items():

    print(f"\nModel : {model_name}")

    synthetic_datasets[model_name] = {}

    if not model_dir.exists():

        print("Directory does not exist.")

        continue

    csv_files = list(model_dir.glob("*.csv"))

    if len(csv_files) == 0:

        print("No CSV files found.")

        continue

    for dataset_name, keywords in dataset_keywords.items():

        matched_file = None

        for keyword in keywords:

            for file in csv_files:

                if keyword.lower() in file.stem.lower():

                    matched_file = file

                    break

            if matched_file is not None:

                break

        if matched_file is None:

            print(f"{dataset_name:<20} Not Found")

            continue

        df = pd.read_csv(matched_file)

        synthetic_datasets[model_name][dataset_name] = df

        print(

            f"{dataset_name:<20}"

            f"{df.shape}"

        )

print("\nSynthetic datasets loaded successfully.")

In [ ]:
# =============================================================================
# Block 3 : Verify Shape
# =============================================================================

print("=" * 80)
print("Verifying Dataset Shapes")
print("=" * 80)

shape_results = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, synthetic_df in datasets.items():

        original_df = original_datasets[dataset_name]["train"]

        shape_results.append({

            "Model": model_name,

            "Dataset": dataset_name,

            "Original Shape": original_df.shape,

            "Synthetic Shape": synthetic_df.shape,

            "Shape Match":

                original_df.shape[1] == synthetic_df.shape[1]

        })

shape_results = pd.DataFrame(shape_results)

display(shape_results)

In [ ]:
# =============================================================================
# Block 4 : Verify Missing Values
# =============================================================================

print("=" * 80)
print("Checking Missing Values")
print("=" * 80)

missing_results = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, synthetic_df in datasets.items():

        missing_results.append({

            "Model": model_name,

            "Dataset": dataset_name,

            "Missing Values":

                synthetic_df.isnull().sum().sum(),

            "Duplicate Rows":

                synthetic_df.duplicated().sum()

        })

missing_results = pd.DataFrame(

    missing_results

)

display(missing_results)

In [ ]:
# =============================================================================
# Block 5 : Verify Data Types
# =============================================================================

print("=" * 80)
print("Checking Data Types")
print("=" * 80)

datatype_results = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, synthetic_df in datasets.items():

        original_df = original_datasets[dataset_name]["train"]

        datatype_results.append({

            "Model": model_name,

            "Dataset": dataset_name,

            "Original Numeric":

                original_df.select_dtypes(

                    include=np.number

                ).shape[1],

            "Synthetic Numeric":

                synthetic_df.select_dtypes(

                    include=np.number

                ).shape[1],

            "Original Categorical":

                original_df.select_dtypes(

                    exclude=np.number

                ).shape[1],

            "Synthetic Categorical":

                synthetic_df.select_dtypes(

                    exclude=np.number

                ).shape[1]

        })

datatype_results = pd.DataFrame(

    datatype_results

)

display(datatype_results)

In [ ]:
# =============================================================================
# Block 6 : Verify Feature Names
# =============================================================================

print("=" * 80)
print("Checking Feature Names")
print("=" * 80)

feature_results = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, synthetic_df in datasets.items():

        original_columns = list(

            original_datasets[dataset_name]["train"].columns

        )

        synthetic_columns = list(

            synthetic_df.columns

        )

        feature_results.append({

            "Model": model_name,

            "Dataset": dataset_name,

            "Feature Match":

                original_columns == synthetic_columns,

            "Original Features":

                len(original_columns),

            "Synthetic Features":

                len(synthetic_columns)

        })

feature_results = pd.DataFrame(

    feature_results

)

display(feature_results)

In [ ]:
# =============================================================================
# Block 7 : Final Verification Summary
# =============================================================================

print("=" * 80)
print("Synthetic Dataset Verification Summary")
print("=" * 80)

summary = []

for model_name, datasets in synthetic_datasets.items():

    for dataset_name, synthetic_df in datasets.items():

        original_df = original_datasets[dataset_name]["train"]

        summary.append({

            "Model":

                model_name,

            "Dataset":

                dataset_name,

            "Rows":

                synthetic_df.shape[0],

            "Columns":

                synthetic_df.shape[1],

            "Missing":

                synthetic_df.isnull().sum().sum(),

            "Duplicates":

                synthetic_df.duplicated().sum(),

            "Feature Match":

                list(original_df.columns) == list(synthetic_df.columns),

            "Column Count Match":

                original_df.shape[1] == synthetic_df.shape[1]

        })

summary = pd.DataFrame(summary)

display(summary)

print("\nAll synthetic datasets have been successfully loaded and verified.")

In [ ]:
# =============================================================================
# 8.5 Descriptive Statistics Comparison
# Block 1 : Compute Statistics
# =============================================================================

print("=" * 80)
print("Computing Descriptive Statistics")
print("=" * 80)

descriptive_statistics = {}

statistics_list = [

    "Mean",
    "Standard Deviation",
    "Variance",
    "Median",
    "Minimum",
    "Maximum",
    "Skewness",
    "Kurtosis"

]

for dataset_name in original_datasets.keys():

    print(f"\nDataset : {dataset_name}")

    original = original_datasets[dataset_name]["train"]

    numeric_columns = original.select_dtypes(include=np.number).columns

    descriptive_statistics[dataset_name] = {}

    # ----------------------------------------------------------
    # Original Dataset
    # ----------------------------------------------------------

    original_stats = pd.DataFrame(index=numeric_columns)

    original_stats["Mean"] = original[numeric_columns].mean()

    original_stats["Standard Deviation"] = original[numeric_columns].std()

    original_stats["Variance"] = original[numeric_columns].var()

    original_stats["Median"] = original[numeric_columns].median()

    original_stats["Minimum"] = original[numeric_columns].min()

    original_stats["Maximum"] = original[numeric_columns].max()

    original_stats["Skewness"] = original[numeric_columns].skew()

    original_stats["Kurtosis"] = original[numeric_columns].kurtosis()

    descriptive_statistics[dataset_name]["Original"] = original_stats

    print("Original Dataset Completed")

    # ----------------------------------------------------------
    # Synthetic Models
    # ----------------------------------------------------------

    for model_name, datasets in synthetic_datasets.items():

        if dataset_name not in datasets:

            continue

        synthetic = datasets[dataset_name]

        synthetic = synthetic[numeric_columns]

        stats = pd.DataFrame(index=numeric_columns)

        stats["Mean"] = synthetic.mean()

        stats["Standard Deviation"] = synthetic.std()

        stats["Variance"] = synthetic.var()

        stats["Median"] = synthetic.median()

        stats["Minimum"] = synthetic.min()

        stats["Maximum"] = synthetic.max()

        stats["Skewness"] = synthetic.skew()

        stats["Kurtosis"] = synthetic.kurtosis()

        descriptive_statistics[dataset_name][model_name] = stats

        print(f"{model_name:<25} Completed")

print("\nDescriptive statistics computed successfully.")

In [ ]:
# =============================================================================
# Block 2 : Comparison Tables
# =============================================================================

print("=" * 80)
print("Creating Comparison Tables")
print("=" * 80)

comparison_tables = {}

for dataset_name in descriptive_statistics.keys():

    comparison_tables[dataset_name] = {}

    original_stats = descriptive_statistics[dataset_name]["Original"]

    for metric in statistics_list:

        comparison = pd.DataFrame()

        comparison["Original"] = original_stats[metric]

        for model_name in MODEL_NAMES:

            if model_name not in descriptive_statistics[dataset_name]:

                continue

            comparison[model_name] = (

                descriptive_statistics

                [dataset_name]

                [model_name]

                [metric]

            )

        comparison_tables[dataset_name][metric] = comparison

print("Comparison tables created.")

In [ ]:
# =============================================================================
# Block 3 : Display Tables
# =============================================================================

print("=" * 80)
print("Displaying Descriptive Statistics")
print("=" * 80)

for dataset_name in comparison_tables.keys():

    print("\n")

    print("=" * 80)

    print(dataset_name)

    print("=" * 80)

    for metric in statistics_list:

        print(f"\n{metric}")

        display(

            comparison_tables

            [dataset_name]

            [metric]

            .round(4)

        )

In [ ]:
# =============================================================================
# Block 4 : Save CSV Files
# =============================================================================

print("=" * 80)
print("Saving CSV Files")
print("=" * 80)

statistics_directory = (

    EVALUATION_DIR /

    "descriptive_statistics"

)

statistics_directory.mkdir(

    parents=True,

    exist_ok=True

)

for dataset_name in comparison_tables.keys():

    dataset_directory = (

        statistics_directory /

        dataset_name.replace(" ", "_")

    )

    dataset_directory.mkdir(

        exist_ok=True

    )

    for metric in statistics_list:

        output_file = (

            dataset_directory /

            f"{metric.replace(' ','_')}.csv"

        )

        comparison_tables[dataset_name][metric].to_csv(

            output_file

        )

print("CSV files saved successfully.")

In [ ]:
# =============================================================================
# Block 5 : Master Statistics Table
# =============================================================================

print("=" * 80)
print("Creating Master Statistics Table")
print("=" * 80)

master_statistics = []

for dataset_name in descriptive_statistics.keys():

    for model_name, stats in descriptive_statistics[dataset_name].items():

        temp = stats.copy()

        temp["Feature"] = temp.index

        temp["Dataset"] = dataset_name

        temp["Model"] = model_name

        master_statistics.append(

            temp.reset_index(drop=True)

        )

master_statistics = pd.concat(

    master_statistics,

    ignore_index=True

)

display(master_statistics.head())

In [ ]:
# =============================================================================
# Block 6 : Save Master Table
# =============================================================================

master_file = (

    statistics_directory /

    "master_descriptive_statistics.csv"

)

master_statistics.to_csv(

    master_file,

    index=False

)

print("=" * 80)
print("Master Statistics Saved")
print("=" * 80)

print(master_file)

In [ ]:
# =============================================================================
# 8.6 Distribution Similarity
# Block 1 : Imports
# =============================================================================

from scipy.stats import (
    ks_2samp,
    anderson_ksamp,
    wasserstein_distance,
    entropy
)

from scipy.spatial.distance import jensenshannon

print("=" * 80)
print("Distribution Similarity Metrics")
print("=" * 80)

In [ ]:
# =============================================================================
# Block 2 : Helper Functions
# =============================================================================

def compute_js_divergence(real, synthetic, bins=50):

    minimum = min(real.min(), synthetic.min())
    maximum = max(real.max(), synthetic.max())

    hist_real, edges = np.histogram(
        real,
        bins=bins,
        range=(minimum, maximum),
        density=True
    )

    hist_syn, _ = np.histogram(
        synthetic,
        bins=edges,
        density=True
    )

    hist_real += 1e-10
    hist_syn += 1e-10

    return jensenshannon(
        hist_real,
        hist_syn
    ) ** 2


def compute_kl_divergence(real, synthetic, bins=50):

    minimum = min(real.min(), synthetic.min())
    maximum = max(real.max(), synthetic.max())

    hist_real, edges = np.histogram(
        real,
        bins=bins,
        range=(minimum, maximum),
        density=True
    )

    hist_syn, _ = np.histogram(
        synthetic,
        bins=edges,
        density=True
    )

    hist_real += 1e-10
    hist_syn += 1e-10

    return entropy(
        hist_real,
        hist_syn
    )

In [ ]:
# =============================================================================
# Block 3 : Per-Feature Distribution Comparison
# =============================================================================

print("=" * 80)
print("Per-Feature Distribution Comparison")
print("=" * 80)

distribution_results = {}

for dataset_name in original_datasets.keys():

    print(f"\nDataset : {dataset_name}")

    real = original_datasets[dataset_name]["train"]

    numeric_columns = real.select_dtypes(include=np.number).columns

    distribution_results[dataset_name] = {}

    for model_name in MODEL_NAMES:

        if dataset_name not in synthetic_datasets[model_name]:
            continue

        synthetic = synthetic_datasets[model_name][dataset_name]

        feature_results = []

        for feature in numeric_columns:

            real_values = real[feature].dropna().values

            syn_values = synthetic[feature].dropna().values

            ks = ks_2samp(
                real_values,
                syn_values
            )

            ad = anderson_ksamp(
                [real_values, syn_values]
            )

            wasserstein = wasserstein_distance(
                real_values,
                syn_values
            )

            js = compute_js_divergence(
                real_values,
                syn_values
            )

            kl = compute_kl_divergence(
                real_values,
                syn_values
            )

            feature_results.append({

                "Feature": feature,

                "KS Statistic": ks.statistic,

                "KS p-value": ks.pvalue,

                "Anderson Statistic": ad.statistic,

                "Anderson p-value": ad.significance_level,

                "Wasserstein": wasserstein,

                "Jensen-Shannon": js,

                "KL Divergence": kl

            })

        distribution_results[dataset_name][model_name] = pd.DataFrame(
            feature_results
        )

        print(f"{model_name:<25} Completed")

In [ ]:
# =============================================================================
# Block 4 : Overall Dataset Scores
# =============================================================================

print("=" * 80)
print("Overall Distribution Similarity")
print("=" * 80)

overall_distribution_scores = []

for dataset_name in distribution_results.keys():

    for model_name, df in distribution_results[dataset_name].items():

        overall_distribution_scores.append({

            "Dataset": dataset_name,

            "Model": model_name,

            "Mean KS":

                df["KS Statistic"].mean(),

            "Mean Anderson":

                df["Anderson Statistic"].mean(),

            "Mean Wasserstein":

                df["Wasserstein"].mean(),

            "Mean Jensen-Shannon":

                df["Jensen-Shannon"].mean(),

            "Mean KL":

                df["KL Divergence"].mean()

        })

overall_distribution_scores = pd.DataFrame(
    overall_distribution_scores
)

display(
    overall_distribution_scores.round(4)
)

In [ ]:
# =============================================================================
# Block 5 : Save Per-Feature Results
# =============================================================================

distribution_directory = (
    EVALUATION_DIR /
    "distribution_similarity"
)

distribution_directory.mkdir(
    parents=True,
    exist_ok=True
)

for dataset_name in distribution_results.keys():

    dataset_folder = (
        distribution_directory /
        dataset_name.replace(" ", "_")
    )

    dataset_folder.mkdir(exist_ok=True)

    for model_name, df in distribution_results[dataset_name].items():

        output_file = (

            dataset_folder /

            f"{model_name.replace(' ','_')}.csv"

        )

        df.to_csv(
            output_file,
            index=False
        )

print("Per-feature CSV files saved.")

In [ ]:
# =============================================================================
# Block 6 : Save Overall Scores
# =============================================================================

overall_distribution_scores.to_csv(

    distribution_directory /

    "overall_distribution_scores.csv",

    index=False

)

print("=" * 80)
print("Distribution Similarity Saved")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# Block 1 : Compute Correlation Matrices
# =============================================================================

print("=" * 80)
print("Computing Correlation Matrices")
print("=" * 80)

correlation_results = {}

correlation_methods = [

    "pearson",

    "spearman",

    "kendall"

]

for dataset_name in original_datasets.keys():

    print(f"\nDataset : {dataset_name}")

    correlation_results[dataset_name] = {}

    original = original_datasets[dataset_name]["train"]

    numeric_columns = original.select_dtypes(
        include=np.number
    ).columns

    original = original[numeric_columns]

    correlation_results[dataset_name]["Original"] = {}

    for method in correlation_methods:

        correlation_results[dataset_name]["Original"][method] = (

            original.corr(method=method)

        )

    for model_name in MODEL_NAMES:

        if dataset_name not in synthetic_datasets[model_name]:

            continue

        synthetic = synthetic_datasets[model_name][dataset_name]

        synthetic = synthetic[numeric_columns]

        correlation_results[dataset_name][model_name] = {}

        for method in correlation_methods:

            correlation_results[dataset_name][model_name][method] = (

                synthetic.corr(method=method)

            )

        print(f"{model_name:<25} Completed")

In [ ]:
# =============================================================================
# Block 2 : Difference Matrices
# =============================================================================

print("=" * 80)
print("Computing Correlation Difference Matrices")
print("=" * 80)

difference_results = {}

for dataset_name in correlation_results.keys():

    difference_results[dataset_name] = {}

    for model_name in MODEL_NAMES:

        if model_name not in correlation_results[dataset_name]:

            continue

        difference_results[dataset_name][model_name] = {}

        for method in correlation_methods:

            original_corr = (

                correlation_results

                [dataset_name]

                ["Original"]

                [method]

            )

            synthetic_corr = (

                correlation_results

                [dataset_name]

                [model_name]

                [method]

            )

            difference_results[dataset_name][model_name][method] = (

                synthetic_corr - original_corr

            )

In [ ]:
# =============================================================================
# Block 3 : Overall Correlation Error
# =============================================================================

print("=" * 80)
print("Computing Overall Correlation Errors")
print("=" * 80)

correlation_summary = []

for dataset_name in difference_results.keys():

    for model_name in MODEL_NAMES:

        if model_name not in difference_results[dataset_name]:

            continue

        row = {

            "Dataset": dataset_name,

            "Model": model_name

        }

        for method in correlation_methods:

            diff = difference_results[dataset_name][model_name][method]

            row[f"{method.title()} MAE"] = np.abs(diff.values).mean()

            row[f"{method.title()} RMSE"] = np.sqrt(

                np.mean(diff.values ** 2)

            )

            row[f"{method.title()} Max Error"] = np.abs(

                diff.values

            ).max()

        correlation_summary.append(row)

correlation_summary = pd.DataFrame(

    correlation_summary

)

display(

    correlation_summary.round(4)

)

In [ ]:
# =============================================================================
# Block 4 : Save Correlation Matrices
# =============================================================================

print("=" * 80)
print("Saving Correlation Matrices")
print("=" * 80)

correlation_directory = (

    EVALUATION_DIR /

    "correlation_preservation"

)

correlation_directory.mkdir(

    parents=True,

    exist_ok=True

)

for dataset_name in correlation_results.keys():

    dataset_folder = (

        correlation_directory /

        dataset_name.replace(" ", "_")

    )

    dataset_folder.mkdir(exist_ok=True)

    for model_name, methods in correlation_results[dataset_name].items():

        model_folder = (

            dataset_folder /

            model_name.replace(" ", "_")

        )

        model_folder.mkdir(exist_ok=True)

        for method, matrix in methods.items():

            matrix.to_csv(

                model_folder /

                f"{method}.csv"

            )

print("Correlation matrices saved.")

In [ ]:
# =============================================================================
# Block 5 : Save Difference Matrices
# =============================================================================

for dataset_name in difference_results.keys():

    dataset_folder = (

        correlation_directory /

        dataset_name.replace(" ", "_")

    )

    for model_name, methods in difference_results[dataset_name].items():

        model_folder = (

            dataset_folder /

            model_name.replace(" ", "_") /

            "difference"

        )

        model_folder.mkdir(

            parents=True,

            exist_ok=True

        )

        for method, matrix in methods.items():

            matrix.to_csv(

                model_folder /

                f"{method}_difference.csv"

            )

print("Difference matrices saved.")

In [ ]:
# =============================================================================
# Block 6 : Correlation Heatmaps
# =============================================================================

print("=" * 80)
print("Generating Heatmaps")
print("=" * 80)

heatmap_directory = (

    correlation_directory /

    "heatmaps"

)

heatmap_directory.mkdir(

    exist_ok=True

)

for dataset_name in correlation_results.keys():

    for model_name, methods in correlation_results[dataset_name].items():

        for method, matrix in methods.items():

            plt.figure(figsize=(8,7))

            plt.imshow(

                matrix,

                interpolation="nearest",

                aspect="auto"

            )

            plt.colorbar()

            plt.title(

                f"{dataset_name}\n"

                f"{model_name}\n"

                f"{method.title()}"

            )

            plt.xticks(

                range(len(matrix.columns)),

                matrix.columns,

                rotation=90,

                fontsize=8

            )

            plt.yticks(

                range(len(matrix.columns)),

                matrix.columns,

                fontsize=8

            )

            plt.tight_layout()

            plt.savefig(

                heatmap_directory /

                f"{dataset_name}_{model_name}_{method}.png",

                dpi=300,

                bbox_inches="tight"

            )

            plt.close()

print("Heatmaps generated.")

In [ ]:
# =============================================================================
# Block 7 : Save Summary
# =============================================================================

summary_file = (

    correlation_directory /

    "correlation_summary.csv"

)

correlation_summary.to_csv(

    summary_file,

    index=False

)

print("=" * 80)
print("Correlation Preservation Completed")
print("=" * 80)

print(summary_file)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.1 Imports & Configuration
# =============================================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)

from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

# =============================================================================
# Configuration
# =============================================================================

print("=" * 80)
print("Correlation Preservation Configuration")
print("=" * 80)

# ------------------------------------------------------------
# Correlation Methods
# ------------------------------------------------------------

CORRELATION_METHODS = [

    "pearson",
    "spearman",
    "kendall"

]

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------

MODEL_NAMES = [

    "Gaussian Multivariate",
    "Gaussian Copula",
    "CTGAN",
    "TVAE",
    "DP-CTGAN",
    "SPP-GAN"

]

# ------------------------------------------------------------
# Output Directories
# ------------------------------------------------------------

CORRELATION_DIR = (

    EVALUATION_DIR /
    "correlation_preservation"

)

CORRELATION_MATRIX_DIR = (

    CORRELATION_DIR /
    "correlation_matrices"

)

DIFFERENCE_MATRIX_DIR = (

    CORRELATION_DIR /
    "difference_matrices"

)

HEATMAP_DIR = (

    CORRELATION_DIR /
    "heatmaps"

)

SUMMARY_DIR = (

    CORRELATION_DIR /
    "summary"

)

# ------------------------------------------------------------
# Create Directories
# ------------------------------------------------------------

for directory in [

    CORRELATION_DIR,
    CORRELATION_MATRIX_DIR,
    DIFFERENCE_MATRIX_DIR,
    HEATMAP_DIR,
    SUMMARY_DIR

]:

    directory.mkdir(

        parents=True,
        exist_ok=True

    )

# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

correlation_results = {}

difference_results = {}

correlation_summary = []

heatmap_paths = {}

# ------------------------------------------------------------
# Figure Settings
# ------------------------------------------------------------

FIGURE_SIZE = (10, 8)

HEATMAP_CMAP = "coolwarm"

HEATMAP_DPI = 300

# ------------------------------------------------------------
# Numeric Precision
# ------------------------------------------------------------

pd.set_option(

    "display.precision",
    4

)

np.set_printoptions(

    precision=4,
    suppress=True

)

# =============================================================================
# Environment Verification
# =============================================================================

print(f"{'Correlation Methods':<30}: {CORRELATION_METHODS}")

print(f"{'Number of Models':<30}: {len(MODEL_NAMES)}")

print(f"{'Evaluation Directory':<30}: {CORRELATION_DIR}")

print(f"{'Heatmap Directory':<30}: {HEATMAP_DIR}")

print(f"{'Figure Size':<30}: {FIGURE_SIZE}")

print(f"{'Heatmap DPI':<30}: {HEATMAP_DPI}")

print("=" * 80)
print("Correlation Preservation Configuration Complete")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.2 Helper Functions
# =============================================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

# =============================================================================
# Compute Correlation Matrix
# =============================================================================

def compute_correlation_matrix(
    dataframe,
    method="pearson"
):
    """
    Computes correlation matrix for a dataframe.

    Parameters
    ----------
    dataframe : pandas.DataFrame

    method : str
        pearson
        spearman
        kendall

    Returns
    -------
    pandas.DataFrame
    """

    numeric_df = dataframe.select_dtypes(
        include=np.number
    )

    return numeric_df.corr(
        method=method
    )


# =============================================================================
# Compute Difference Matrix
# =============================================================================

def compute_difference_matrix(

    original_matrix,
    synthetic_matrix

):
    """
    Synthetic - Original
    """

    return synthetic_matrix - original_matrix


# =============================================================================
# Mean Absolute Error
# =============================================================================

def matrix_mae(

    original_matrix,
    synthetic_matrix

):

    difference = np.abs(

        original_matrix.values -

        synthetic_matrix.values

    )

    return difference.mean()


# =============================================================================
# Root Mean Square Error
# =============================================================================

def matrix_rmse(

    original_matrix,
    synthetic_matrix

):

    return np.sqrt(

        mean_squared_error(

            original_matrix.values.flatten(),

            synthetic_matrix.values.flatten()

        )

    )


# =============================================================================
# Frobenius Norm
# =============================================================================

def matrix_frobenius(

    original_matrix,
    synthetic_matrix

):

    return np.linalg.norm(

        original_matrix.values -

        synthetic_matrix.values,

        ord="fro"

    )


# =============================================================================
# Maximum Absolute Difference
# =============================================================================

def matrix_max_error(

    original_matrix,
    synthetic_matrix

):

    difference = np.abs(

        original_matrix.values -

        synthetic_matrix.values

    )

    return difference.max()


# =============================================================================
# Flatten Correlation Matrix
# =============================================================================

def flatten_matrix(matrix):

    """
    Returns upper triangle only
    """

    mask = np.triu(

        np.ones(

            matrix.shape,

            dtype=bool

        ),

        k=1

    )

    return matrix.where(mask).stack().values


# =============================================================================
# Publication Quality Heatmap
# =============================================================================

def plot_heatmap(

    matrix,

    title,

    save_path,

    cmap="coolwarm",

    figsize=(10,8),

    vmin=-1,

    vmax=1

):

    plt.figure(

        figsize=figsize

    )

    plt.imshow(

        matrix,

        cmap=cmap,

        interpolation="nearest",

        aspect="auto",

        vmin=vmin,

        vmax=vmax

    )

    plt.colorbar()

    plt.xticks(

        range(len(matrix.columns)),

        matrix.columns,

        rotation=90,

        fontsize=8

    )

    plt.yticks(

        range(len(matrix.columns)),

        matrix.columns,

        fontsize=8

    )

    plt.title(

        title,

        fontsize=14,

        fontweight="bold"

    )

    plt.tight_layout()

    plt.savefig(

        save_path,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()


# =============================================================================
# Difference Heatmap
# =============================================================================

def plot_difference_heatmap(

    matrix,

    title,

    save_path,

    cmap="bwr",

    figsize=(10,8)

):

    maximum = np.abs(

        matrix.values

    ).max()

    plt.figure(

        figsize=figsize

    )

    plt.imshow(

        matrix,

        cmap=cmap,

        interpolation="nearest",

        aspect="auto",

        vmin=-maximum,

        vmax=maximum

    )

    plt.colorbar()

    plt.xticks(

        range(len(matrix.columns)),

        matrix.columns,

        rotation=90,

        fontsize=8

    )

    plt.yticks(

        range(len(matrix.columns)),

        matrix.columns,

        fontsize=8

    )

    plt.title(

        title,

        fontsize=14,

        fontweight="bold"

    )

    plt.tight_layout()

    plt.savefig(

        save_path,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()


# =============================================================================
# Correlation Preservation Metrics
# =============================================================================

def correlation_preservation_metrics(

    original_matrix,

    synthetic_matrix

):

    return {

        "MAE":

            matrix_mae(

                original_matrix,

                synthetic_matrix

            ),

        "RMSE":

            matrix_rmse(

                original_matrix,

                synthetic_matrix

            ),

        "Frobenius":

            matrix_frobenius(

                original_matrix,

                synthetic_matrix

            ),

        "Maximum Error":

            matrix_max_error(

                original_matrix,

                synthetic_matrix

            )

    }


# =============================================================================
# Print Matrix Information
# =============================================================================

def print_matrix_information(

    matrix,

    name

):

    print("-" * 60)

    print(name)

    print("-" * 60)

    print(

        f"Shape : {matrix.shape}"

    )

    print(

        f"Minimum : {matrix.values.min():.4f}"

    )

    print(

        f"Maximum : {matrix.values.max():.4f}"

    )

    print(

        f"Mean : {matrix.values.mean():.4f}"

    )

    print(

        f"Std : {matrix.values.std():.4f}"

    )


# =============================================================================
# Helper Function Verification
# =============================================================================

print("=" * 80)
print("Helper Functions Loaded Successfully")
print("=" * 80)

print("Available Functions")

print("- compute_correlation_matrix()")

print("- compute_difference_matrix()")

print("- matrix_mae()")

print("- matrix_rmse()")

print("- matrix_frobenius()")

print("- matrix_max_error()")

print("- flatten_matrix()")

print("- plot_heatmap()")

print("- plot_difference_heatmap()")

print("- correlation_preservation_metrics()")

print("- print_matrix_information()")

print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.3 Compute Pearson, Spearman & Kendall Correlation Matrices
# =============================================================================

print("=" * 80)
print("Computing Correlation Matrices")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

correlation_results = {}

# =============================================================================
# Compute Correlations
# =============================================================================

for dataset_name in original_datasets.keys():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    correlation_results[dataset_name] = {}

    # -------------------------------------------------------------------------
    # Original Dataset
    # -------------------------------------------------------------------------

    original_df = original_datasets[dataset_name]["train"]

    original_numeric = original_df.select_dtypes(
        include=np.number
    )

    correlation_results[dataset_name]["Original"] = {}

    print("\nOriginal Dataset")

    for method in CORRELATION_METHODS:

        correlation_matrix = compute_correlation_matrix(

            dataframe=original_numeric,

            method=method

        )

        correlation_results[dataset_name]["Original"][method] = (

            correlation_matrix

        )

        print(

            f"{method.capitalize():<12}"

            f"{correlation_matrix.shape}"

        )

    # -------------------------------------------------------------------------
    # Synthetic Datasets
    # -------------------------------------------------------------------------

    for model_name in MODEL_NAMES:

        if dataset_name not in synthetic_datasets.get(model_name, {}):

            continue

        synthetic_df = synthetic_datasets[model_name][dataset_name]

        # -------------------------------------------------------------
        # Keep only numeric columns
        # -------------------------------------------------------------

        synthetic_numeric = synthetic_df.select_dtypes(
            include=np.number
        )

        # -------------------------------------------------------------
        # Align Feature Order
        # -------------------------------------------------------------

        common_columns = [

            column

            for column in original_numeric.columns

            if column in synthetic_numeric.columns

        ]

        original_aligned = original_numeric[common_columns]

        synthetic_aligned = synthetic_numeric[common_columns]

        correlation_results[dataset_name][model_name] = {}

        print(f"\n{model_name}")

        for method in CORRELATION_METHODS:

            correlation_matrix = compute_correlation_matrix(

                dataframe=synthetic_aligned,

                method=method

            )

            correlation_results[dataset_name][model_name][method] = (

                correlation_matrix

            )

            print(

                f"{method.capitalize():<12}"

                f"{correlation_matrix.shape}"

            )

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Correlation Matrix Verification")
print("=" * 80)

for dataset_name in correlation_results.keys():

    print(f"\nDataset : {dataset_name}")

    for model_name in correlation_results[dataset_name]:

        print(f"\n{model_name}")

        for method in CORRELATION_METHODS:

            matrix = correlation_results[dataset_name][model_name][method]

            print(

                f"{method.capitalize():<12}"

                f"Shape : {matrix.shape}"

            )

# =============================================================================
# Summary
# =============================================================================

print("\n" + "=" * 80)
print("Correlation Matrices Successfully Computed")
print("=" * 80)

print(f"Datasets Evaluated : {len(correlation_results)}")

print(f"Correlation Methods : {len(CORRELATION_METHODS)}")

print(f"Models Compared : {len(MODEL_NAMES)}")

print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.4 Compute Difference Matrices
# =============================================================================

print("=" * 80)
print("Computing Correlation Difference Matrices")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

difference_results = {}

# =============================================================================
# Compute Difference Matrices
# =============================================================================

for dataset_name in correlation_results.keys():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    difference_results[dataset_name] = {}

    # -------------------------------------------------------------------------
    # Original Correlation Matrices
    # -------------------------------------------------------------------------

    original_correlations = correlation_results[dataset_name]["Original"]

    # -------------------------------------------------------------------------
    # Compare Every Synthetic Model
    # -------------------------------------------------------------------------

    for model_name in MODEL_NAMES:

        if model_name not in correlation_results[dataset_name]:

            continue

        print(f"\nModel : {model_name}")

        difference_results[dataset_name][model_name] = {}

        for method in CORRELATION_METHODS:

            original_matrix = original_correlations[method]

            synthetic_matrix = (

                correlation_results

                [dataset_name]

                [model_name]

                [method]

            )

            # -------------------------------------------------------------
            # Align matrices (safety check)
            # -------------------------------------------------------------

            common_columns = [

                column

                for column in original_matrix.columns

                if column in synthetic_matrix.columns

            ]

            original_matrix = (

                original_matrix

                .loc[common_columns, common_columns]

            )

            synthetic_matrix = (

                synthetic_matrix

                .loc[common_columns, common_columns]

            )

            # -------------------------------------------------------------
            # Difference Matrix
            # -------------------------------------------------------------

            difference_matrix = compute_difference_matrix(

                original_matrix,

                synthetic_matrix

            )

            difference_results[dataset_name][model_name][method] = (

                difference_matrix

            )

            print(

                f"{method.capitalize():<12}"

                f"{difference_matrix.shape}"

            )

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Difference Matrix Verification")
print("=" * 80)

for dataset_name in difference_results.keys():

    print(f"\nDataset : {dataset_name}")

    for model_name in difference_results[dataset_name]:

        print(f"\n{model_name}")

        for method in CORRELATION_METHODS:

            matrix = (

                difference_results

                [dataset_name]

                [model_name]

                [method]

            )

            print(

                f"{method.capitalize():<12}"

                f"Shape : {matrix.shape}"

                f" | "

                f"Min : {matrix.values.min():.4f}"

                f" | "

                f"Max : {matrix.values.max():.4f}"

            )

# =============================================================================
# Display Example Difference Matrix
# =============================================================================

example_dataset = list(difference_results.keys())[0]

example_model = list(

    difference_results[example_dataset].keys()

)[0]

example_method = CORRELATION_METHODS[0]

print("\n" + "=" * 80)
print("Example Difference Matrix")
print("=" * 80)

print(f"Dataset : {example_dataset}")

print(f"Model   : {example_model}")

print(f"Method  : {example_method.capitalize()}")

display(

    difference_results

    [example_dataset]

    [example_model]

    [example_method]

    .round(4)

)

# =============================================================================
# Summary
# =============================================================================

print("\n" + "=" * 80)
print("Difference Matrices Successfully Computed")
print("=" * 80)

print(f"Datasets Processed : {len(difference_results)}")

print(f"Correlation Methods : {len(CORRELATION_METHODS)}")

print(f"Models Compared : {len(MODEL_NAMES)}")

print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.5 Correlation Preservation Metrics
# =============================================================================

print("=" * 80)
print("Computing Correlation Preservation Metrics")
print("=" * 80)

# =============================================================================
# Container
# =============================================================================

correlation_summary = []

# =============================================================================
# Compute Metrics
# =============================================================================

for dataset_name in difference_results.keys():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    for model_name in MODEL_NAMES:

        if model_name not in difference_results[dataset_name]:
            continue

        print(f"\nModel : {model_name}")

        row = {

            "Dataset": dataset_name,
            "Model": model_name

        }

        # ------------------------------------------------------------
        # Compute metrics for every correlation method
        # ------------------------------------------------------------

        for method in CORRELATION_METHODS:

            original_matrix = (

                correlation_results
                [dataset_name]
                ["Original"]
                [method]

            )

            synthetic_matrix = (

                correlation_results
                [dataset_name]
                [model_name]
                [method]

            )

            metrics = correlation_preservation_metrics(

                original_matrix,

                synthetic_matrix

            )

            row[f"{method.title()} MAE"] = metrics["MAE"]

            row[f"{method.title()} RMSE"] = metrics["RMSE"]

            row[f"{method.title()} Frobenius"] = metrics["Frobenius"]

            row[f"{method.title()} Max Error"] = metrics["Maximum Error"]

            print(

                f"{method.capitalize():<12}"

                f"MAE={metrics['MAE']:.6f} | "

                f"RMSE={metrics['RMSE']:.6f} | "

                f"Frobenius={metrics['Frobenius']:.6f}"

            )

        # ------------------------------------------------------------
        # Overall Scores
        # ------------------------------------------------------------

        row["Average MAE"] = np.mean([

            row["Pearson MAE"],

            row["Spearman MAE"],

            row["Kendall MAE"]

        ])

        row["Average RMSE"] = np.mean([

            row["Pearson RMSE"],

            row["Spearman RMSE"],

            row["Kendall RMSE"]

        ])

        row["Average Frobenius"] = np.mean([

            row["Pearson Frobenius"],

            row["Spearman Frobenius"],

            row["Kendall Frobenius"]

        ])

        row["Average Maximum Error"] = np.mean([

            row["Pearson Max Error"],

            row["Spearman Max Error"],

            row["Kendall Max Error"]

        ])

        correlation_summary.append(row)

# =============================================================================
# Convert to DataFrame
# =============================================================================

correlation_summary = pd.DataFrame(

    correlation_summary

)

# =============================================================================
# Rank Models
# =============================================================================

correlation_summary = correlation_summary.sort_values(

    by=[

        "Dataset",

        "Average MAE",

        "Average RMSE"

    ],

    ascending=True

).reset_index(drop=True)

# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 80)
print("Correlation Preservation Summary")
print("=" * 80)

display(

    correlation_summary.round(6)

)

# =============================================================================
# Best Model Per Dataset
# =============================================================================

print("\n" + "=" * 80)
print("Best Correlation Preservation Model")
print("=" * 80)

for dataset_name in correlation_summary["Dataset"].unique():

    best = (

        correlation_summary

        [

            correlation_summary["Dataset"]

            == dataset_name

        ]

        .iloc[0]

    )

    print(

        f"{dataset_name:<20}"

        f"{best['Model']:<25}"

        f"Average MAE = {best['Average MAE']:.6f}"

    )

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(f"Datasets Evaluated : {correlation_summary['Dataset'].nunique()}")

print(f"Models Evaluated   : {correlation_summary['Model'].nunique()}")

print(f"Rows Generated     : {len(correlation_summary)}")

print(f"Columns Generated  : {len(correlation_summary.columns)}")

# =============================================================================
# Summary
# =============================================================================

print("\n" + "=" * 80)
print("Correlation Preservation Metrics Computed Successfully")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.6 Save Correlation Matrices
# =============================================================================

print("=" * 80)
print("Saving Correlation Matrices")
print("=" * 80)

# =============================================================================
# Directory
# =============================================================================

CORRELATION_MATRIX_DIR = (

    CORRELATION_DIR /

    "correlation_matrices"

)

CORRELATION_MATRIX_DIR.mkdir(

    parents=True,

    exist_ok=True

)

# =============================================================================
# Save Matrices
# =============================================================================

saved_files = 0

for dataset_name in correlation_results.keys():

    dataset_directory = (

        CORRELATION_MATRIX_DIR /

        dataset_name.replace(" ", "_")

    )

    dataset_directory.mkdir(

        parents=True,

        exist_ok=True

    )

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Original Dataset
    # -------------------------------------------------------------------------

    original_directory = (

        dataset_directory /

        "Original"

    )

    original_directory.mkdir(

        exist_ok=True

    )

    for method in CORRELATION_METHODS:

        output_file = (

            original_directory /

            f"{method}_correlation.csv"

        )

        correlation_results[

            dataset_name

        ][

            "Original"

        ][

            method

        ].round(6).to_csv(

            output_file,

            index=True

        )

        saved_files += 1

        print(

            f"Saved : {output_file.name}"

        )

    # -------------------------------------------------------------------------
    # Synthetic Models
    # -------------------------------------------------------------------------

    for model_name in MODEL_NAMES:

        if model_name not in correlation_results[dataset_name]:

            continue

        model_directory = (

            dataset_directory /

            model_name.replace(" ", "_")

        )

        model_directory.mkdir(

            exist_ok=True

        )

        print(f"\nModel : {model_name}")

        for method in CORRELATION_METHODS:

            output_file = (

                model_directory /

                f"{method}_correlation.csv"

            )

            correlation_results[

                dataset_name

            ][

                model_name

            ][

                method

            ].round(6).to_csv(

                output_file,

                index=True

            )

            saved_files += 1

            print(

                f"Saved : {output_file.name}"

            )

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(

    f"Datasets Saved : {len(correlation_results)}"

)

print(

    f"Correlation Methods : {len(CORRELATION_METHODS)}"

)

print(

    f"Models : {len(MODEL_NAMES)+1} (including Original)"

)

print(

    f"CSV Files Saved : {saved_files}"

)

print(

    f"Output Directory : {CORRELATION_MATRIX_DIR}"

)

# =============================================================================
# Directory Structure
# =============================================================================

print("\n" + "=" * 80)
print("Directory Structure")
print("=" * 80)

for dataset_name in correlation_results.keys():

    print(

        f"\n{dataset_name}"

    )

    print(

        "├── Original"

    )

    for model in MODEL_NAMES:

        if model in correlation_results[dataset_name]:

            print(

                f"├── {model}"

            )

# =============================================================================
# Summary
# =============================================================================

print("\n" + "=" * 80)
print("Correlation Matrices Successfully Saved")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.7 Save Difference Matrices
# =============================================================================

print("=" * 80)
print("Saving Correlation Difference Matrices")
print("=" * 80)

# =============================================================================
# Directory
# =============================================================================

DIFFERENCE_MATRIX_DIR = (

    CORRELATION_DIR /

    "difference_matrices"

)

DIFFERENCE_MATRIX_DIR.mkdir(

    parents=True,

    exist_ok=True

)

# =============================================================================
# Save Difference Matrices
# =============================================================================

saved_files = 0

for dataset_name in difference_results.keys():

    dataset_directory = (

        DIFFERENCE_MATRIX_DIR /

        dataset_name.replace(" ", "_")

    )

    dataset_directory.mkdir(

        parents=True,

        exist_ok=True

    )

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    for model_name in MODEL_NAMES:

        if model_name not in difference_results[dataset_name]:

            continue

        model_directory = (

            dataset_directory /

            model_name.replace(" ", "_")

        )

        model_directory.mkdir(

            exist_ok=True

        )

        print(f"\nModel : {model_name}")

        for method in CORRELATION_METHODS:

            difference_matrix = (

                difference_results

                [dataset_name]

                [model_name]

                [method]

            )

            output_file = (

                model_directory /

                f"{method}_difference.csv"

            )

            difference_matrix.round(6).to_csv(

                output_file,

                index=True

            )

            saved_files += 1

            print(

                f"Saved : {output_file.name}"

            )

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(

    f"Datasets Saved : {len(difference_results)}"

)

print(

    f"Models Evaluated : {len(MODEL_NAMES)}"

)

print(

    f"Correlation Methods : {len(CORRELATION_METHODS)}"

)

print(

    f"Difference CSV Files : {saved_files}"

)

print(

    f"Output Directory : {DIFFERENCE_MATRIX_DIR}"

)

# =============================================================================
# Example Matrix
# =============================================================================

example_dataset = list(difference_results.keys())[0]

example_model = list(

    difference_results[example_dataset].keys()

)[0]

example_method = CORRELATION_METHODS[0]

print("\n" + "=" * 80)
print("Example Difference Matrix")
print("=" * 80)

print(f"Dataset : {example_dataset}")

print(f"Model   : {example_model}")

print(f"Method  : {example_method}")

display(

    difference_results

    [example_dataset]

    [example_model]

    [example_method]

    .round(4)

)

# =============================================================================
# Directory Structure
# =============================================================================

print("\n" + "=" * 80)
print("Directory Structure")
print("=" * 80)

for dataset_name in difference_results.keys():

    print(f"\n{dataset_name}")

    for model_name in MODEL_NAMES:

        if model_name in difference_results[dataset_name]:

            print(

                f"├── {model_name}"

            )

            for method in CORRELATION_METHODS:

                print(

                    f"│   ├── {method}_difference.csv"

                )

# =============================================================================
# Summary
# =============================================================================

print("\n" + "=" * 80)
print("Difference Matrices Successfully Saved")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.8 Generate Publication-Quality Heatmaps
# =============================================================================

print("=" * 80)
print("Generating Publication Quality Heatmaps")
print("=" * 80)

# =============================================================================
# Heatmap Directory
# =============================================================================

HEATMAP_DIR = (

    CORRELATION_DIR /

    "heatmaps"

)

ORIGINAL_HEATMAP_DIR = (

    HEATMAP_DIR /

    "original"

)

SYNTHETIC_HEATMAP_DIR = (

    HEATMAP_DIR /

    "synthetic"

)

DIFFERENCE_HEATMAP_DIR = (

    HEATMAP_DIR /

    "difference"

)

for directory in [

    HEATMAP_DIR,

    ORIGINAL_HEATMAP_DIR,

    SYNTHETIC_HEATMAP_DIR,

    DIFFERENCE_HEATMAP_DIR

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

# =============================================================================
# Heatmap Function
# =============================================================================

def save_heatmap(

    matrix,

    title,

    output_path,

    cmap,

    vmin,

    vmax

):

    fig, ax = plt.subplots(

        figsize=(10,8)

    )

    image = ax.imshow(

        matrix.values,

        cmap=cmap,

        interpolation="nearest",

        aspect="equal",

        vmin=vmin,

        vmax=vmax

    )

    ax.set_xticks(

        np.arange(

            len(matrix.columns)

        )

    )

    ax.set_yticks(

        np.arange(

            len(matrix.columns)

        )

    )

    ax.set_xticklabels(

        matrix.columns,

        rotation=90,

        fontsize=8

    )

    ax.set_yticklabels(

        matrix.columns,

        fontsize=8

    )

    ax.set_title(

        title,

        fontsize=14,

        fontweight="bold"

    )

    cbar = plt.colorbar(

        image,

        ax=ax,

        shrink=0.80

    )

    cbar.ax.tick_params(

        labelsize=8

    )

    plt.tight_layout()

    plt.savefig(

        output_path,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close(fig)

# =============================================================================
# Generate Heatmaps
# =============================================================================

total_heatmaps = 0

for dataset_name in correlation_results.keys():

    print("\n" + "=" * 80)

    print(dataset_name)

    print("=" * 80)

    # -------------------------------------------------------------------------
    # Original
    # -------------------------------------------------------------------------

    for method in CORRELATION_METHODS:

        matrix = (

            correlation_results

            [dataset_name]

            ["Original"]

            [method]

        )

        filename = (

            ORIGINAL_HEATMAP_DIR /

            f"{dataset_name.replace(' ','_')}_{method}.png"

        )

        save_heatmap(

            matrix=matrix,

            title=f"{dataset_name}\nOriginal\n{method.title()}",

            output_path=filename,

            cmap="coolwarm",

            vmin=-1,

            vmax=1

        )

        total_heatmaps += 1

    # -------------------------------------------------------------------------
    # Synthetic Models
    # -------------------------------------------------------------------------

    for model_name in MODEL_NAMES:

        if model_name not in correlation_results[dataset_name]:

            continue

        print(model_name)

        for method in CORRELATION_METHODS:

            synthetic_matrix = (

                correlation_results

                [dataset_name]

                [model_name]

                [method]

            )

            difference_matrix = (

                difference_results

                [dataset_name]

                [model_name]

                [method]

            )

            # -------------------------------------------------------------
            # Synthetic Heatmap
            # -------------------------------------------------------------

            synthetic_output = (

                SYNTHETIC_HEATMAP_DIR /

                f"{dataset_name.replace(' ','_')}_"

                f"{model_name.replace(' ','_')}_"

                f"{method}.png"

            )

            save_heatmap(

                matrix=synthetic_matrix,

                title=(

                    f"{dataset_name}\n"

                    f"{model_name}\n"

                    f"{method.title()}"

                ),

                output_path=synthetic_output,

                cmap="coolwarm",

                vmin=-1,

                vmax=1

            )

            total_heatmaps += 1

            # -------------------------------------------------------------
            # Difference Heatmap
            # -------------------------------------------------------------

            difference_output = (

                DIFFERENCE_HEATMAP_DIR /

                f"{dataset_name.replace(' ','_')}_"

                f"{model_name.replace(' ','_')}_"

                f"{method}_difference.png"

            )

            maximum = np.abs(

                difference_matrix.values

            ).max()

            save_heatmap(

                matrix=difference_matrix,

                title=(

                    f"{dataset_name}\n"

                    f"{model_name}\n"

                    f"{method.title()} Difference"

                ),

                output_path=difference_output,

                cmap="bwr",

                vmin=-maximum,

                vmax=maximum

            )

            total_heatmaps += 1

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)

print("Heatmap Generation Completed")

print("=" * 80)

print(f"Datasets            : {len(correlation_results)}")

print(f"Models              : {len(MODEL_NAMES)}")

print(f"Correlation Methods : {len(CORRELATION_METHODS)}")

print(f"Heatmaps Generated  : {total_heatmaps}")

print(f"Output Directory    : {HEATMAP_DIR}")

print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.9 Create Summary Tables
# =============================================================================

print("=" * 80)
print("Creating Publication Summary Tables")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

dataset_summary_tables = {}

overall_summary_tables = {}

# =============================================================================
# Dataset-wise Summary Tables
# =============================================================================

for dataset_name in correlation_summary["Dataset"].unique():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    summary = (

        correlation_summary

        [

            correlation_summary["Dataset"] == dataset_name

        ]

        .copy()

    )

    summary = summary.sort_values(

        by=[

            "Average MAE",

            "Average RMSE"

        ],

        ascending=True

    )

    summary.insert(

        0,

        "Rank",

        range(

            1,

            len(summary)+1

        )

    )

    dataset_summary_tables[dataset_name] = summary

    display(

        summary.round(6)

    )

# =============================================================================
# Overall Model Performance
# =============================================================================

overall_summary = (

    correlation_summary

    .groupby("Model")

    .agg({

        "Pearson MAE":"mean",
        "Spearman MAE":"mean",
        "Kendall MAE":"mean",

        "Pearson RMSE":"mean",
        "Spearman RMSE":"mean",
        "Kendall RMSE":"mean",

        "Pearson Frobenius":"mean",
        "Spearman Frobenius":"mean",
        "Kendall Frobenius":"mean",

        "Average MAE":"mean",
        "Average RMSE":"mean",
        "Average Frobenius":"mean"

    })

)

overall_summary = overall_summary.sort_values(

    by=[

        "Average MAE",

        "Average RMSE"

    ],

    ascending=True

)

overall_summary.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_summary)+1

    )

)

overall_summary_tables["Overall"] = overall_summary

print("\n" + "=" * 80)
print("Overall Correlation Preservation Ranking")
print("=" * 80)

display(

    overall_summary.round(6)

)

# =============================================================================
# Best Model Per Dataset
# =============================================================================

best_models = []

for dataset_name in dataset_summary_tables.keys():

    best = dataset_summary_tables[dataset_name].iloc[0]

    best_models.append({

        "Dataset": dataset_name,

        "Best Model": best["Model"],

        "Average MAE": best["Average MAE"],

        "Average RMSE": best["Average RMSE"],

        "Average Frobenius": best["Average Frobenius"]

    })

best_models = pd.DataFrame(

    best_models

)

print("\n" + "=" * 80)
print("Best Model for Each Dataset")
print("=" * 80)

display(

    best_models.round(6)

)

# =============================================================================
# Mean Rank Across Datasets
# =============================================================================

ranking_table = []

for dataset_name in dataset_summary_tables.keys():

    temp = dataset_summary_tables[dataset_name][

        [

            "Rank",

            "Model"

        ]

    ]

    ranking_table.append(temp)

ranking_table = pd.concat(

    ranking_table,

    ignore_index=True

)

mean_rank = (

    ranking_table

    .groupby("Model")

    ["Rank"]

    .mean()

    .reset_index()

)

mean_rank.columns = [

    "Model",

    "Average Rank"

]

mean_rank = mean_rank.sort_values(

    "Average Rank"

)

print("\n" + "=" * 80)
print("Average Model Ranking")
print("=" * 80)

display(

    mean_rank.round(4)

)

# =============================================================================
# Publication Table
# =============================================================================

publication_table = overall_summary.reset_index()

publication_table = publication_table[

    [

        "Overall Rank",

        "Model",

        "Average MAE",

        "Average RMSE",

        "Average Frobenius"

    ]

]

print("\n" + "=" * 80)
print("Publication Table")
print("=" * 80)

display(

    publication_table.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(

    f"Dataset Tables : {len(dataset_summary_tables)}"

)

print(

    f"Overall Models : {len(overall_summary)}"

)

print(

    f"Publication Rows : {len(publication_table)}"

)

print("=" * 80)
print("Summary Tables Created Successfully")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.10 Save CSV Files
# =============================================================================

print("=" * 80)
print("Saving Correlation Preservation Results")
print("=" * 80)

# =============================================================================
# Create Output Directory
# =============================================================================

SUMMARY_DIR = (

    CORRELATION_DIR /

    "summary"

)

SUMMARY_DIR.mkdir(

    parents=True,

    exist_ok=True

)

# =============================================================================
# Container
# =============================================================================

saved_files = []

# =============================================================================
# 1. Correlation Summary
# =============================================================================

summary_file = (

    SUMMARY_DIR /

    "correlation_summary.csv"

)

correlation_summary.round(6).to_csv(

    summary_file,

    index=False

)

saved_files.append(summary_file)

# =============================================================================
# 2. Overall Summary
# =============================================================================

overall_file = (

    SUMMARY_DIR /

    "overall_model_ranking.csv"

)

overall_summary.round(6).to_csv(

    overall_file

)

saved_files.append(overall_file)

# =============================================================================
# 3. Publication Table
# =============================================================================

publication_file = (

    SUMMARY_DIR /

    "publication_table.csv"

)

publication_table.round(6).to_csv(

    publication_file,

    index=False

)

saved_files.append(publication_file)

# =============================================================================
# 4. Best Models
# =============================================================================

best_model_file = (

    SUMMARY_DIR /

    "best_models.csv"

)

best_models.round(6).to_csv(

    best_model_file,

    index=False

)

saved_files.append(best_model_file)

# =============================================================================
# 5. Average Rank
# =============================================================================

average_rank_file = (

    SUMMARY_DIR /

    "average_model_rank.csv"

)

mean_rank.round(6).to_csv(

    average_rank_file,

    index=False

)

saved_files.append(average_rank_file)

# =============================================================================
# 6. Dataset-wise Summary Tables
# =============================================================================

DATASET_SUMMARY_DIR = (

    SUMMARY_DIR /

    "dataset_tables"

)

DATASET_SUMMARY_DIR.mkdir(

    exist_ok=True

)

for dataset_name, table in dataset_summary_tables.items():

    filename = (

        DATASET_SUMMARY_DIR /

        f"{dataset_name.replace(' ','_')}_summary.csv"

    )

    table.round(6).to_csv(

        filename,

        index=False

    )

    saved_files.append(filename)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Saved Files")
print("=" * 80)

for file in saved_files:

    print(file.name)

# =============================================================================
# Summary Statistics
# =============================================================================

print("\n" + "=" * 80)
print("Summary")
print("=" * 80)

print(

    f"CSV Files Saved : {len(saved_files)}"

)

print(

    f"Output Directory : {SUMMARY_DIR}"

)

print(

    f"Dataset Tables : {len(dataset_summary_tables)}"

)

print("=" * 80)
print("All Correlation Preservation Results Saved Successfully")
print("=" * 80)

In [ ]:
# =============================================================================
# 8.7 Correlation Preservation
# 8.7.11 Final Summary
# =============================================================================

print("=" * 90)
print("FINAL SUMMARY : CORRELATION PRESERVATION EVALUATION")
print("=" * 90)

# =============================================================================
# Experiment Information
# =============================================================================

print("\nExperiment Information")
print("-" * 90)

print(f"Datasets Evaluated            : {len(correlation_results)}")
print(f"Synthetic Models Evaluated    : {len(MODEL_NAMES)}")
print(f"Correlation Methods           : {len(CORRELATION_METHODS)}")
print(f"Correlation Methods Used      : {', '.join([m.title() for m in CORRELATION_METHODS])}")

# =============================================================================
# Files Generated
# =============================================================================

print("\nGenerated Outputs")
print("-" * 90)

print("✓ Correlation Matrices")
print("✓ Difference Matrices")
print("✓ Publication Heatmaps")
print("✓ Dataset Summary Tables")
print("✓ Overall Ranking Table")
print("✓ Publication Table")
print("✓ CSV Reports")

# =============================================================================
# Best Model Per Dataset
# =============================================================================

print("\nBest Model Per Dataset")
print("-" * 90)

display(

    best_models.round(6)

)

# =============================================================================
# Overall Ranking
# =============================================================================

print("\nOverall Model Ranking")
print("-" * 90)

display(

    publication_table.round(6)

)

# =============================================================================
# Overall Best Model
# =============================================================================

overall_best = publication_table.iloc[0]

print("\nOverall Best Performing Model")
print("-" * 90)

print(f"Rank               : {overall_best['Overall Rank']}")
print(f"Model              : {overall_best['Model']}")
print(f"Average MAE        : {overall_best['Average MAE']:.6f}")
print(f"Average RMSE       : {overall_best['Average RMSE']:.6f}")
print(f"Average Frobenius  : {overall_best['Average Frobenius']:.6f}")

# =============================================================================
# Average Model Ranking
# =============================================================================

print("\nAverage Rank Across All Datasets")
print("-" * 90)

display(

    mean_rank.round(4)

)

# =============================================================================
# Evaluation Metrics Used
# =============================================================================

print("\nEvaluation Metrics")
print("-" * 90)

metrics = pd.DataFrame({

    "Category":[

        "Correlation",

        "Correlation",

        "Correlation",

        "Correlation",

        "Correlation",

        "Correlation"

    ],

    "Metric":[

        "Pearson",

        "Spearman",

        "Kendall",

        "MAE",

        "RMSE",

        "Frobenius Norm"

    ],

    "Purpose":[

        "Linear Relationship Preservation",

        "Rank Relationship Preservation",

        "Ordinal Relationship Preservation",

        "Average Correlation Error",

        "Squared Correlation Error",

        "Overall Matrix Difference"

    ]

})

display(metrics)

# =============================================================================
# Output Locations
# =============================================================================

print("\nSaved Results")
print("-" * 90)

print(f"Correlation Matrices : {CORRELATION_MATRIX_DIR}")

print(f"Difference Matrices  : {DIFFERENCE_MATRIX_DIR}")

print(f"Heatmaps             : {HEATMAP_DIR}")

print(f"Summary Tables       : {SUMMARY_DIR}")

# =============================================================================
# Publication Readiness
# =============================================================================

print("\nPublication Checklist")
print("-" * 90)

checklist = pd.DataFrame({

    "Task":[

        "Correlation Matrices",

        "Difference Matrices",

        "Heatmaps",

        "Summary Tables",

        "CSV Reports",

        "Overall Ranking",

        "Publication Table",

        "Best Model Selection"

    ],

    "Status":[

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed"

    ]

})

display(checklist)

# =============================================================================
# Research Interpretation
# =============================================================================

print("\nResearch Interpretation")
print("-" * 90)

print(
    "Lower MAE, RMSE, and Frobenius Norm values indicate better preservation "
    "of the original feature relationships. Models ranked at the top preserve "
    "the statistical dependency structure of the original datasets more "
    "effectively. These results will be integrated with the remaining "
    "evaluation metrics (distribution similarity, utility, privacy, and "
    "downstream machine learning performance) in the final comparative "
    "analysis."
)

# =============================================================================
# Completion
# =============================================================================

print("\n" + "=" * 90)
print("SECTION 8.7 COMPLETED SUCCESSFULLY")
print("=" * 90)

print("The following outputs are ready for Notebook 09:")

print(" • Correlation matrices")
print(" • Difference matrices")
print(" • Publication-quality heatmaps")
print(" • Summary CSV files")
print(" • Model ranking tables")
print(" • Publication-ready tables")

print("=" * 90)
print("Proceed to Section 8.8")
print("=" * 90)

In [ ]:
# =============================================================================
# Section 8.8 : Multivariate Similarity
# Block 8.8.1 : Imports & Configuration
# =============================================================================

print("=" * 80)
print("SECTION 8.8 : MULTIVARIATE SIMILARITY")
print("Block 8.8.1 : Imports & Configuration")
print("=" * 80)

# =============================================================================
# Standard Library Imports
# =============================================================================

import os
import gc
import random
import warnings
from pathlib import Path

# =============================================================================
# Scientific Computing
# =============================================================================

import numpy as np
import pandas as pd

# =============================================================================
# Visualization
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# Publication Style
plt.style.use("default")
sns.set_theme(style="whitegrid")

# =============================================================================
# Machine Learning
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import pairwise_distances

from sklearn.feature_selection import mutual_info_regression

# =============================================================================
# Statistical Analysis
# =============================================================================

from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import kendalltau

from scipy.linalg import norm

# =============================================================================
# Ignore Warnings
# =============================================================================

warnings.filterwarnings("ignore")

# =============================================================================
# Random Seed
# =============================================================================

SEED = CONFIG["seed"]

random.seed(SEED)

np.random.seed(SEED)

# =============================================================================
# Publication Figure Settings
# =============================================================================

FIGURE_DPI = 300

FIGURE_SIZE = (10, 8)

HEATMAP_SIZE = (9, 8)

SCATTER_SIZE = (9, 7)

FONT_SIZE = 12

TITLE_SIZE = 14

plt.rcParams["figure.dpi"] = FIGURE_DPI
plt.rcParams["savefig.dpi"] = FIGURE_DPI

plt.rcParams["font.size"] = FONT_SIZE
plt.rcParams["axes.titlesize"] = TITLE_SIZE
plt.rcParams["axes.labelsize"] = FONT_SIZE

# =============================================================================
# PCA Configuration
# =============================================================================

PCA_COMPONENTS = 3

# =============================================================================
# Directory Structure
# =============================================================================

MULTIVARIATE_DIR = (

    RESULTS_DIR /

    "evaluation" /

    "multivariate_similarity"

)

MULTIVARIATE_DIR.mkdir(

    parents=True,

    exist_ok=True

)

# -------------------------------------------------------------------------

COVARIANCE_DIR = (

    MULTIVARIATE_DIR /

    "covariance"

)

PCA_DIR = (

    MULTIVARIATE_DIR /

    "pca"

)

MUTUAL_INFORMATION_DIR = (

    MULTIVARIATE_DIR /

    "mutual_information"

)

HEATMAP_DIR = (

    MULTIVARIATE_DIR /

    "heatmaps"

)

PCA_FIGURE_DIR = (

    MULTIVARIATE_DIR /

    "pca_figures"

)

SUMMARY_DIR = (

    MULTIVARIATE_DIR /

    "summary"

)

# =============================================================================
# Create Directories
# =============================================================================

directories = [

    COVARIANCE_DIR,

    PCA_DIR,

    MUTUAL_INFORMATION_DIR,

    HEATMAP_DIR,

    PCA_FIGURE_DIR,

    SUMMARY_DIR

]

for directory in directories:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

# =============================================================================
# Evaluation Models
# =============================================================================

MODELS = [

    "Gaussian Multivariate",

    "Gaussian Copula",

    "CTGAN",

    "TVAE",

    "DP-CTGAN",

    "SPP-GAN"

]

# =============================================================================
# Containers
# =============================================================================

covariance_results = {}

pca_results = {}

mutual_information_results = {}

multivariate_summary = []

# =============================================================================
# Environment Verification
# =============================================================================

print("\nEnvironment Verification")
print("-" * 80)

print(f"Random Seed              : {SEED}")

print(f"PCA Components           : {PCA_COMPONENTS}")

print(f"Datasets                 : {len(original_datasets)}")

print(f"Synthetic Models         : {len(MODELS)}")

print(f"Publication DPI          : {FIGURE_DPI}")

print(f"Figure Size              : {FIGURE_SIZE}")

print("\nOutput Directories")

print("-" * 80)

print(f"Main Directory           : {MULTIVARIATE_DIR}")

print(f"Covariance               : {COVARIANCE_DIR}")

print(f"PCA                      : {PCA_DIR}")

print(f"Mutual Information       : {MUTUAL_INFORMATION_DIR}")

print(f"Heatmaps                 : {HEATMAP_DIR}")

print(f"PCA Figures              : {PCA_FIGURE_DIR}")

print(f"Summary                  : {SUMMARY_DIR}")

print("\n" + "=" * 80)
print("8.8.1 IMPORTS & CONFIGURATION COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================================
# Section 8.8 : Multivariate Similarity
# Block 8.8.2 : Helper Functions
# =============================================================================

print("=" * 80)
print("SECTION 8.8 : HELPER FUNCTIONS")
print("=" * 80)

# =============================================================================
# Extract Numeric Features
# =============================================================================

def get_numeric_dataframe(df):

    """
    Return only numeric columns.
    """

    numeric_df = df.select_dtypes(include=[np.number]).copy()

    return numeric_df


# =============================================================================
# Standardize Dataset
# =============================================================================

def standardize_dataset(df):

    """
    Standardize numeric features.
    """

    scaler = StandardScaler()

    scaled = scaler.fit_transform(df)

    scaled = pd.DataFrame(

        scaled,

        columns=df.columns,

        index=df.index

    )

    return scaled


# =============================================================================
# Covariance Matrix
# =============================================================================

def compute_covariance_matrix(df):

    """
    Compute covariance matrix.
    """

    return df.cov()


# =============================================================================
# Covariance Difference Metrics
# =============================================================================

def covariance_metrics(original_cov, synthetic_cov):

    """
    Compute covariance preservation metrics.
    """

    difference = (

        synthetic_cov.values -

        original_cov.values

    )

    mae = np.mean(

        np.abs(difference)

    )

    rmse = np.sqrt(

        np.mean(

            difference ** 2

        )

    )

    frobenius = np.linalg.norm(

        difference,

        ord="fro"

    )

    maximum = np.max(

        np.abs(difference)

    )

    return {

        "MAE": mae,

        "RMSE": rmse,

        "Frobenius": frobenius,

        "Maximum Error": maximum

    }


# =============================================================================
# PCA Analysis
# =============================================================================

def compute_pca(df, n_components=3):

    """
    Fit PCA.
    """

    pca = PCA(

        n_components=n_components,

        random_state=SEED

    )

    projection = pca.fit_transform(df)

    projection = pd.DataFrame(

        projection,

        columns=[

            f"PC{i+1}"

            for i in range(n_components)

        ]

    )

    return pca, projection


# =============================================================================
# PCA Similarity Metrics
# =============================================================================

def pca_similarity(original_pca, synthetic_pca):

    """
    Compare explained variance.
    """

    variance_difference = np.abs(

        original_pca.explained_variance_ratio_

        -

        synthetic_pca.explained_variance_ratio_

    )

    mae = np.mean(

        variance_difference

    )

    rmse = np.sqrt(

        np.mean(

            variance_difference ** 2

        )

    )

    cosine_similarity = (

        np.dot(

            original_pca.explained_variance_ratio_,

            synthetic_pca.explained_variance_ratio_

        )

        /

        (

            np.linalg.norm(

                original_pca.explained_variance_ratio_

            )

            *

            np.linalg.norm(

                synthetic_pca.explained_variance_ratio_

            )

        )

    )

    return {

        "MAE": mae,

        "RMSE": rmse,

        "Cosine Similarity": cosine_similarity

    }


# =============================================================================
# Mutual Information
# =============================================================================

def compute_mutual_information(df):

    """
    Average mutual information between features.
    """

    df = df.copy()

    X = df.iloc[:, :-1]

    y = df.iloc[:, -1]

    mi = mutual_info_regression(

        X,

        y,

        random_state=SEED

    )

    return pd.Series(

        mi,

        index=X.columns

    )


# =============================================================================
# 2D PCA Visualization
# =============================================================================

def plot_pca_2d(

    original,

    synthetic,

    title,

    save_path=None

):

    plt.figure(

        figsize=SCATTER_SIZE

    )

    plt.scatter(

        original.iloc[:,0],

        original.iloc[:,1],

        s=20,

        alpha=0.60,

        label="Original"

    )

    plt.scatter(

        synthetic.iloc[:,0],

        synthetic.iloc[:,1],

        s=20,

        alpha=0.60,

        label="Synthetic"

    )

    plt.xlabel("PC1")

    plt.ylabel("PC2")

    plt.title(title)

    plt.legend()

    plt.tight_layout()

    if save_path is not None:

        plt.savefig(

            save_path,

            dpi=FIGURE_DPI,

            bbox_inches="tight"

        )

    plt.close()


# =============================================================================
# 3D PCA Visualization
# =============================================================================

def plot_pca_3d(

    original,

    synthetic,

    title,

    save_path=None

):

    fig = plt.figure(

        figsize=FIGURE_SIZE

    )

    ax = fig.add_subplot(

        111,

        projection="3d"

    )

    ax.scatter(

        original.iloc[:,0],

        original.iloc[:,1],

        original.iloc[:,2],

        alpha=0.60,

        s=15,

        label="Original"

    )

    ax.scatter(

        synthetic.iloc[:,0],

        synthetic.iloc[:,1],

        synthetic.iloc[:,2],

        alpha=0.60,

        s=15,

        label="Synthetic"

    )

    ax.set_xlabel("PC1")

    ax.set_ylabel("PC2")

    ax.set_zlabel("PC3")

    ax.set_title(title)

    ax.legend()

    plt.tight_layout()

    if save_path is not None:

        plt.savefig(

            save_path,

            dpi=FIGURE_DPI,

            bbox_inches="tight"

        )

    plt.close()


# =============================================================================
# Covariance Heatmap
# =============================================================================

def covariance_heatmap(

    matrix,

    title,

    save_path=None

):

    plt.figure(

        figsize=HEATMAP_SIZE

    )

    sns.heatmap(

        matrix,

        cmap="coolwarm",

        center=0,

        square=True,

        cbar=True

    )

    plt.title(title)

    plt.tight_layout()

    if save_path is not None:

        plt.savefig(

            save_path,

            dpi=FIGURE_DPI,

            bbox_inches="tight"

        )

    plt.close()


# =============================================================================
# Verification
# =============================================================================

print("\nAvailable Helper Functions")
print("-" * 80)

functions = [

    "get_numeric_dataframe",

    "standardize_dataset",

    "compute_covariance_matrix",

    "covariance_metrics",

    "compute_pca",

    "pca_similarity",

    "compute_mutual_information",

    "plot_pca_2d",

    "plot_pca_3d",

    "covariance_heatmap"

]

for i, function in enumerate(functions, start=1):

    print(f"{i:2d}. {function}")

print("\n" + "=" * 80)
print("8.8.2 HELPER FUNCTIONS READY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.3 : Covariance Matrix Comparison
# =============================================================================

print("=" * 80)
print("SECTION 8.8.3 : COVARIANCE MATRIX COMPARISON")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

covariance_results = {}

# =============================================================================
# Compute Covariance Matrices
# =============================================================================

for dataset_name in original_datasets.keys():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    covariance_results[dataset_name] = {}

    # -------------------------------------------------------------------------
    # Original Dataset
    # -------------------------------------------------------------------------

    original_df = get_numeric_dataframe(

        original_datasets[dataset_name]["train"]

    )

    original_df = standardize_dataset(original_df)

    original_covariance = compute_covariance_matrix(original_df)

    covariance_results[dataset_name]["Original"] = original_covariance

    print(
        f"Original Covariance Shape : {original_covariance.shape}"
    )

    # -------------------------------------------------------------------------
    # Synthetic Models
    # -------------------------------------------------------------------------

    for model_name in MODELS:

        if model_name not in synthetic_datasets[dataset_name]:

            print(f"Skipping {model_name}")

            continue

        synthetic_df = get_numeric_dataframe(

            synthetic_datasets[dataset_name][model_name]

        )

        synthetic_df = standardize_dataset(

            synthetic_df

        )

        synthetic_covariance = compute_covariance_matrix(

            synthetic_df

        )

        covariance_results[dataset_name][model_name] = (

            synthetic_covariance

        )

        print(

            f"{model_name:<25}"

            f"{synthetic_covariance.shape}"

        )

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

for dataset_name in covariance_results.keys():

    print(f"\n{dataset_name}")

    for model_name in covariance_results[dataset_name].keys():

        matrix = covariance_results[dataset_name][model_name]

        print(

            f"{model_name:<25}"

            f"{matrix.shape}"

        )

print("\n" + "=" * 80)
print("Covariance Matrices Successfully Computed")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.4 : Covariance Difference Metrics
# =============================================================================

print("=" * 80)
print("SECTION 8.8.4 : COVARIANCE DIFFERENCE METRICS")
print("=" * 80)

# =============================================================================
# Container
# =============================================================================

covariance_summary = []

covariance_difference_results = {}

# =============================================================================
# Compute Metrics
# =============================================================================

for dataset_name in covariance_results.keys():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    covariance_difference_results[dataset_name] = {}

    original_cov = covariance_results[dataset_name]["Original"]

    for model_name in MODELS:

        if model_name not in covariance_results[dataset_name]:

            continue

        synthetic_cov = covariance_results[dataset_name][model_name]

        # --------------------------------------------------------------
        # Difference Matrix
        # --------------------------------------------------------------

        difference_matrix = synthetic_cov - original_cov

        covariance_difference_results[dataset_name][model_name] = (

            difference_matrix

        )

        # --------------------------------------------------------------
        # Metrics
        # --------------------------------------------------------------

        metrics = covariance_metrics(

            original_cov,

            synthetic_cov

        )

        covariance_summary.append({

            "Dataset": dataset_name,

            "Model": model_name,

            "MAE": metrics["MAE"],

            "RMSE": metrics["RMSE"],

            "Frobenius": metrics["Frobenius"],

            "Maximum Error": metrics["Maximum Error"]

        })

        print(

            f"{model_name:<25}"

            f"MAE={metrics['MAE']:.6f}   "

            f"RMSE={metrics['RMSE']:.6f}   "

            f"Frobenius={metrics['Frobenius']:.6f}"

        )

# =============================================================================
# Create Summary DataFrame
# =============================================================================

covariance_summary = pd.DataFrame(

    covariance_summary

)

# =============================================================================
# Rank Models
# =============================================================================

covariance_summary = covariance_summary.sort_values(

    by=[

        "Dataset",

        "MAE",

        "RMSE"

    ],

    ascending=True

).reset_index(

    drop=True

)

# =============================================================================
# Overall Performance
# =============================================================================

overall_covariance_summary = (

    covariance_summary

    .groupby("Model")

    .agg({

        "MAE":"mean",

        "RMSE":"mean",

        "Frobenius":"mean",

        "Maximum Error":"mean"

    })

)

overall_covariance_summary = (

    overall_covariance_summary

    .sort_values(

        by=[

            "MAE",

            "RMSE"

        ]

    )

)

overall_covariance_summary.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_covariance_summary)+1

    )

)

# =============================================================================
# Display Dataset Summary
# =============================================================================

print("\n" + "=" * 80)
print("Dataset-wise Covariance Metrics")
print("=" * 80)

display(

    covariance_summary.round(6)

)

# =============================================================================
# Display Overall Ranking
# =============================================================================

print("\n" + "=" * 80)
print("Overall Covariance Preservation Ranking")
print("=" * 80)

display(

    overall_covariance_summary.round(6)

)

# =============================================================================
# Best Model Per Dataset
# =============================================================================

best_covariance_models = []

for dataset_name in covariance_summary["Dataset"].unique():

    best = (

        covariance_summary

        [

            covariance_summary["Dataset"]

            == dataset_name

        ]

        .iloc[0]

    )

    best_covariance_models.append({

        "Dataset": dataset_name,

        "Best Model": best["Model"],

        "MAE": best["MAE"],

        "RMSE": best["RMSE"],

        "Frobenius": best["Frobenius"]

    })

best_covariance_models = pd.DataFrame(

    best_covariance_models

)

print("\n" + "=" * 80)
print("Best Covariance Preservation Model")
print("=" * 80)

display(

    best_covariance_models.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(

    f"Datasets Evaluated : {covariance_summary['Dataset'].nunique()}"

)

print(

    f"Models Evaluated   : {covariance_summary['Model'].nunique()}"

)

print(

    f"Difference Matrices: {sum(len(v) for v in covariance_difference_results.values())}"

)

print(

    f"Rows Generated     : {len(covariance_summary)}"

)

print("\n" + "=" * 80)
print("Covariance Difference Metrics Completed")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.5 : PCA Similarity Analysis
# =============================================================================

print("=" * 80)
print("SECTION 8.8.5 : PCA SIMILARITY ANALYSIS")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

pca_results = {}

pca_summary = []

# =============================================================================
# PCA Analysis
# =============================================================================

for dataset_name in original_datasets.keys():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    pca_results[dataset_name] = {}

    # -------------------------------------------------------------------------
    # Original Dataset
    # -------------------------------------------------------------------------

    original_df = get_numeric_dataframe(
        original_datasets[dataset_name]["train"]
    )

    scaler = StandardScaler()

    original_scaled = pd.DataFrame(
        scaler.fit_transform(original_df),
        columns=original_df.columns
    )

    original_pca = PCA(
        n_components=PCA_COMPONENTS,
        random_state=SEED
    )

    original_projection = original_pca.fit_transform(original_scaled)

    pca_results[dataset_name]["Original"] = {

        "pca": original_pca,

        "projection": original_projection,

        "explained_variance":

            original_pca.explained_variance_,

        "explained_variance_ratio":

            original_pca.explained_variance_ratio_,

        "cumulative_variance":

            np.cumsum(

                original_pca.explained_variance_ratio_

            )

    }

    # -------------------------------------------------------------------------
    # Synthetic Models
    # -------------------------------------------------------------------------

    for model_name in MODELS:

        if model_name not in synthetic_datasets[dataset_name]:

            continue

        synthetic_df = get_numeric_dataframe(

            synthetic_datasets[dataset_name][model_name]

        )

        synthetic_scaled = pd.DataFrame(

            scaler.transform(synthetic_df),

            columns=synthetic_df.columns

        )

        synthetic_projection = original_pca.transform(

            synthetic_scaled

        )

        synthetic_pca = PCA(

            n_components=PCA_COMPONENTS,

            random_state=SEED

        )

        synthetic_pca.fit(synthetic_scaled)

        original_ratio = (

            original_pca.explained_variance_ratio_

        )

        synthetic_ratio = (

            synthetic_pca.explained_variance_ratio_

        )

        mae = np.mean(

            np.abs(

                original_ratio -

                synthetic_ratio

            )

        )

        rmse = np.sqrt(

            np.mean(

                (

                    original_ratio -

                    synthetic_ratio

                ) ** 2

            )

        )

        cosine = (

            np.dot(

                original_ratio,

                synthetic_ratio

            )

            /

            (

                np.linalg.norm(original_ratio)

                *

                np.linalg.norm(synthetic_ratio)

            )

        )

        pearson = pearsonr(

            original_ratio,

            synthetic_ratio

        )[0]

        pca_results[dataset_name][model_name] = {

            "pca": synthetic_pca,

            "projection": synthetic_projection,

            "explained_variance":

                synthetic_pca.explained_variance_,

            "explained_variance_ratio":

                synthetic_ratio,

            "cumulative_variance":

                np.cumsum(synthetic_ratio)

        }

        pca_summary.append({

            "Dataset": dataset_name,

            "Model": model_name,

            "MAE": mae,

            "RMSE": rmse,

            "Cosine Similarity": cosine,

            "Pearson Correlation": pearson,

            "PC1 Variance":

                synthetic_ratio[0],

            "PC2 Variance":

                synthetic_ratio[1],

            "PC3 Variance":

                synthetic_ratio[2]

        })

        print(

            f"{model_name:<25}"

            f"Cosine={cosine:.4f}   "

            f"Pearson={pearson:.4f}"

        )

# =============================================================================
# Summary Table
# =============================================================================

pca_summary = pd.DataFrame(

    pca_summary

)

pca_summary = pca_summary.sort_values(

    [

        "Dataset",

        "MAE",

        "RMSE"

    ]

).reset_index(

    drop=True

)

# =============================================================================
# Overall Ranking
# =============================================================================

overall_pca_summary = (

    pca_summary

    .groupby("Model")

    .agg({

        "MAE":"mean",

        "RMSE":"mean",

        "Cosine Similarity":"mean",

        "Pearson Correlation":"mean"

    })

)

overall_pca_summary = (

    overall_pca_summary

    .sort_values(

        [

            "MAE",

            "RMSE"

        ]

    )

)

overall_pca_summary.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_pca_summary)+1

    )

)

# =============================================================================
# Display Results
# =============================================================================

print("\nDataset-wise PCA Similarity")
print("-" * 80)

display(

    pca_summary.round(6)

)

print("\nOverall PCA Ranking")
print("-" * 80)

display(

    overall_pca_summary.round(6)

)

# =============================================================================
# Best Model Per Dataset
# =============================================================================

best_pca_models = (

    pca_summary

    .sort_values(

        [

            "Dataset",

            "MAE"

        ]

    )

    .groupby("Dataset")

    .first()

    .reset_index()

)

print("\nBest PCA Preservation Model")
print("-" * 80)

display(

    best_pca_models.round(6)

)

print("\n" + "=" * 80)
print("PCA Similarity Analysis Completed")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.6 : PCA Projection (2D & 3D)
# =============================================================================

print("=" * 80)
print("SECTION 8.8.6 : PCA PROJECTION VISUALIZATION")
print("=" * 80)

from mpl_toolkits.mplot3d import Axes3D

# =============================================================================
# Create Output Directories
# =============================================================================

PCA_2D_DIR = PCA_FIGURE_DIR / "2D"

PCA_3D_DIR = PCA_FIGURE_DIR / "3D"

PCA_2D_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PCA_3D_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# Generate Figures
# =============================================================================

figure_count = 0

for dataset_name in original_datasets.keys():

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    original_projection = pd.DataFrame(

        pca_results[dataset_name]["Original"]["projection"],

        columns=["PC1","PC2","PC3"]

    )

    for model_name in MODELS:

        if model_name not in pca_results[dataset_name]:

            continue

        synthetic_projection = pd.DataFrame(

            pca_results[dataset_name][model_name]["projection"],

            columns=["PC1","PC2","PC3"]

        )

        # ==============================================================
        # 2D PCA
        # ==============================================================

        plt.figure(figsize=(9,7))

        plt.scatter(

            original_projection["PC1"],
            original_projection["PC2"],

            s=15,

            alpha=0.55,

            label="Original"

        )

        plt.scatter(

            synthetic_projection["PC1"],
            synthetic_projection["PC2"],

            s=15,

            alpha=0.55,

            label=model_name

        )

        plt.xlabel("Principal Component 1")

        plt.ylabel("Principal Component 2")

        plt.title(

            f"{dataset_name}\nOriginal vs {model_name}"

        )

        plt.legend()

        plt.tight_layout()

        filename = (

            PCA_2D_DIR /

            f"{dataset_name.replace(' ','_')}_{model_name.replace(' ','_')}_2D.png"

        )

        plt.savefig(

            filename,

            dpi=300,

            bbox_inches="tight"

        )

        plt.close()

        # ==============================================================
        # 3D PCA
        # ==============================================================

        fig = plt.figure(figsize=(9,7))

        ax = fig.add_subplot(

            111,

            projection="3d"

        )

        ax.scatter(

            original_projection["PC1"],

            original_projection["PC2"],

            original_projection["PC3"],

            s=12,

            alpha=0.55,

            label="Original"

        )

        ax.scatter(

            synthetic_projection["PC1"],

            synthetic_projection["PC2"],

            synthetic_projection["PC3"],

            s=12,

            alpha=0.55,

            label=model_name

        )

        ax.set_xlabel("PC1")

        ax.set_ylabel("PC2")

        ax.set_zlabel("PC3")

        ax.set_title(

            f"{dataset_name}\nOriginal vs {model_name}"

        )

        ax.legend()

        plt.tight_layout()

        filename = (

            PCA_3D_DIR /

            f"{dataset_name.replace(' ','_')}_{model_name.replace(' ','_')}_3D.png"

        )

        plt.savefig(

            filename,

            dpi=300,

            bbox_inches="tight"

        )

        plt.close()

        figure_count += 2

        print(

            f"✓ {model_name}"

        )

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("PCA Projection Summary")
print("=" * 80)

print(f"Datasets              : {len(original_datasets)}")

print(f"Models                : {len(MODELS)}")

print(f"2D Figures            : {len(original_datasets)*len(MODELS)}")

print(f"3D Figures            : {len(original_datasets)*len(MODELS)}")

print(f"Total Figures         : {figure_count}")

print(f"\n2D Directory")

print(PCA_2D_DIR)

print(f"\n3D Directory")

print(PCA_3D_DIR)

print("\n" + "=" * 80)
print("PCA Projection Figures Generated Successfully")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.7 : Mutual Information Analysis
# =============================================================================

print("=" * 80)
print("SECTION 8.8.7 : MUTUAL INFORMATION ANALYSIS")
print("=" * 80)

# =============================================================================
# Helper Function
# =============================================================================

def compute_mutual_information_matrix(df):

    """
    Computes a symmetric pairwise mutual information matrix.
    """

    df = get_numeric_dataframe(df)

    n_features = df.shape[1]

    columns = df.columns

    mi_matrix = np.zeros((n_features, n_features))

    for i in range(n_features):

        for j in range(i, n_features):

            if i == j:

                mi = 1.0

            else:

                mi = mutual_info_regression(

                    df.iloc[:, [i]],

                    df.iloc[:, j],

                    random_state=SEED

                )[0]

            mi_matrix[i, j] = mi
            mi_matrix[j, i] = mi

    return pd.DataFrame(

        mi_matrix,

        index=columns,

        columns=columns

    )


# =============================================================================
# Containers
# =============================================================================

mi_results = {}

mi_summary = []

# =============================================================================
# Compute Mutual Information
# =============================================================================

for dataset_name in original_datasets.keys():

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    mi_results[dataset_name] = {}

    # -------------------------------------------------------------------------
    # Original Dataset
    # -------------------------------------------------------------------------

    original_df = get_numeric_dataframe(

        original_datasets[dataset_name]["train"]

    )

    original_mi = compute_mutual_information_matrix(

        original_df

    )

    mi_results[dataset_name]["Original"] = original_mi

    # -------------------------------------------------------------------------
    # Synthetic Models
    # -------------------------------------------------------------------------

    for model_name in MODELS:

        if model_name not in synthetic_datasets[dataset_name]:

            continue

        synthetic_df = get_numeric_dataframe(

            synthetic_datasets[dataset_name][model_name]

        )

        synthetic_mi = compute_mutual_information_matrix(

            synthetic_df

        )

        mi_results[dataset_name][model_name] = synthetic_mi

        difference = (

            synthetic_mi.values -

            original_mi.values

        )

        mae = np.mean(

            np.abs(difference)

        )

        rmse = np.sqrt(

            np.mean(

                difference ** 2

            )

        )

        frobenius = np.linalg.norm(

            difference,

            ord="fro"

        )

        maximum = np.max(

            np.abs(difference)

        )

        mi_summary.append({

            "Dataset": dataset_name,

            "Model": model_name,

            "MAE": mae,

            "RMSE": rmse,

            "Frobenius": frobenius,

            "Maximum Error": maximum

        })

        print(

            f"{model_name:<25}"

            f"MAE={mae:.6f}   "

            f"RMSE={rmse:.6f}"

        )

# =============================================================================
# Summary Table
# =============================================================================

mi_summary = pd.DataFrame(

    mi_summary

)

mi_summary = mi_summary.sort_values(

    [

        "Dataset",

        "MAE",

        "RMSE"

    ]

).reset_index(

    drop=True

)

# =============================================================================
# Overall Ranking
# =============================================================================

overall_mi_summary = (

    mi_summary

    .groupby("Model")

    .agg({

        "MAE":"mean",

        "RMSE":"mean",

        "Frobenius":"mean",

        "Maximum Error":"mean"

    })

)

overall_mi_summary = (

    overall_mi_summary

    .sort_values(

        [

            "MAE",

            "RMSE"

        ]

    )

)

overall_mi_summary.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_mi_summary)+1

    )

)

# =============================================================================
# Best Model Per Dataset
# =============================================================================

best_mi_models = (

    mi_summary

    .sort_values(

        [

            "Dataset",

            "MAE"

        ]

    )

    .groupby("Dataset")

    .first()

    .reset_index()

)

# =============================================================================
# Display Results
# =============================================================================

print("\nDataset-wise Mutual Information Preservation")
print("-" * 80)

display(

    mi_summary.round(6)

)

print("\nOverall Mutual Information Ranking")
print("-" * 80)

display(

    overall_mi_summary.round(6)

)

print("\nBest Model Per Dataset")
print("-" * 80)

display(

    best_mi_models.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(

    f"Datasets Evaluated : {len(mi_results)}"

)

print(

    f"Models Evaluated   : {len(MODELS)}"

)

print(

    f"Summary Rows       : {len(mi_summary)}"

)

print("\n" + "=" * 80)
print("Mutual Information Analysis Completed")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.8 : Multivariate Summary Table
# =============================================================================

print("=" * 80)
print("SECTION 8.8.8 : MULTIVARIATE SUMMARY TABLE")
print("=" * 80)

# =============================================================================
# Merge All Evaluation Results
# =============================================================================

multivariate_summary = (

    covariance_summary

    .merge(

        pca_summary[

            [

                "Dataset",

                "Model",

                "Cosine Similarity",

                "Pearson Correlation"

            ]

        ],

        on=[

            "Dataset",

            "Model"

        ]

    )

    .merge(

        mi_summary[

            [

                "Dataset",

                "Model",

                "MAE",

                "RMSE"

            ]

        ].rename(

            columns={

                "MAE":"MI MAE",

                "RMSE":"MI RMSE"

            }

        ),

        on=[

            "Dataset",

            "Model"

        ]

    )

)

# =============================================================================
# Composite Score
# Lower is Better
# =============================================================================

multivariate_summary["Overall Score"] = (

    multivariate_summary["MAE"]

    +

    multivariate_summary["RMSE"]

    +

    multivariate_summary["Frobenius"]

    +

    multivariate_summary["MI MAE"]

    +

    multivariate_summary["MI RMSE"]

    -

    multivariate_summary["Cosine Similarity"]

    -

    multivariate_summary["Pearson Correlation"]

)

# =============================================================================
# Dataset-wise Ranking
# =============================================================================

multivariate_summary = (

    multivariate_summary

    .sort_values(

        [

            "Dataset",

            "Overall Score"

        ]

    )

)

multivariate_summary["Rank"] = (

    multivariate_summary

    .groupby("Dataset")["Overall Score"]

    .rank(

        method="dense",

        ascending=True

    )

)

multivariate_summary = multivariate_summary.reset_index(

    drop=True

)

# =============================================================================
# Overall Ranking
# =============================================================================

overall_multivariate = (

    multivariate_summary

    .groupby("Model")

    .agg({

        "MAE":"mean",

        "RMSE":"mean",

        "Frobenius":"mean",

        "MI MAE":"mean",

        "MI RMSE":"mean",

        "Cosine Similarity":"mean",

        "Pearson Correlation":"mean",

        "Overall Score":"mean"

    })

)

overall_multivariate = (

    overall_multivariate

    .sort_values(

        "Overall Score"

    )

)

overall_multivariate.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_multivariate)+1

    )

)

# =============================================================================
# Best Model Per Dataset
# =============================================================================

best_models = (

    multivariate_summary

    .sort_values(

        [

            "Dataset",

            "Overall Score"

        ]

    )

    .groupby("Dataset")

    .first()

    .reset_index()

)

# =============================================================================
# Display Tables
# =============================================================================

print("\nDataset-wise Ranking")
print("-" * 80)

display(

    multivariate_summary.round(6)

)

print("\nOverall Ranking")
print("-" * 80)

display(

    overall_multivariate.round(6)

)

print("\nBest Model Per Dataset")
print("-" * 80)

display(

    best_models.round(6)

)

# =============================================================================
# Summary Statistics
# =============================================================================

print("\n" + "=" * 80)
print("Summary Statistics")
print("=" * 80)

print(

    f"Datasets              : {multivariate_summary['Dataset'].nunique()}"

)

print(

    f"Models                : {multivariate_summary['Model'].nunique()}"

)

print(

    f"Evaluations           : {len(multivariate_summary)}"

)

print(

    f"Best Overall Model    : {overall_multivariate.index[0]}"

)

print("\n" + "=" * 80)
print("Multivariate Summary Table Completed")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.9 : Save Results
# =============================================================================

print("=" * 80)
print("SECTION 8.8.9 : SAVE MULTIVARIATE RESULTS")
print("=" * 80)

# =============================================================================
# Create Output Directory
# =============================================================================

CSV_DIR = MULTIVARIATE_DIR / "csv"

CSV_DIR.mkdir(

    parents=True,

    exist_ok=True

)

# =============================================================================
# File Paths
# =============================================================================

covariance_file = (

    CSV_DIR /

    "covariance_summary.csv"

)

pca_file = (

    CSV_DIR /

    "pca_similarity_summary.csv"

)

mi_file = (

    CSV_DIR /

    "mutual_information_summary.csv"

)

multivariate_file = (

    CSV_DIR /

    "multivariate_summary.csv"

)

overall_file = (

    CSV_DIR /

    "overall_multivariate_ranking.csv"

)

best_model_file = (

    CSV_DIR /

    "best_multivariate_models.csv"

)

# =============================================================================
# Save CSV Files
# =============================================================================

covariance_summary.to_csv(

    covariance_file,

    index=False

)

pca_summary.to_csv(

    pca_file,

    index=False

)

mi_summary.to_csv(

    mi_file,

    index=False

)

multivariate_summary.to_csv(

    multivariate_file,

    index=False

)

overall_multivariate.reset_index().to_csv(

    overall_file,

    index=False

)

best_models.to_csv(

    best_model_file,

    index=False

)

# =============================================================================
# Verification
# =============================================================================

saved_files = [

    covariance_file,

    pca_file,

    mi_file,

    multivariate_file,

    overall_file,

    best_model_file

]

print("\nSaved Files")
print("-" * 80)

for file in saved_files:

    if file.exists():

        size_kb = file.stat().st_size / 1024

        print(

            f"✓ {file.name:<40}"

            f"{size_kb:8.2f} KB"

        )

    else:

        print(

            f"✗ {file.name}"

        )

# =============================================================================
# Summary
# =============================================================================

print("\n" + "=" * 80)
print("MULTIVARIATE RESULTS SAVED SUCCESSFULLY")
print("=" * 80)

print(f"Output Directory : {CSV_DIR}")

print(f"Total CSV Files  : {len(saved_files)}")

print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.8 : Multivariate Similarity
# Block 8.8.10 : Final Summary
# =============================================================================

print("=" * 80)
print("SECTION 8.8.10 : FINAL SUMMARY")
print("=" * 80)

# =============================================================================
# Build Publication Summary
# =============================================================================

publication_summary = pd.DataFrame({

    "Evaluation Module": [

        "Covariance Preservation",

        "PCA Similarity",

        "Mutual Information",

        "Overall Multivariate Ranking"

    ],

    "Status": [

        "Completed",

        "Completed",

        "Completed",

        "Completed"

    ],

    "Output": [

        "Covariance Metrics",

        "Explained Variance Analysis",

        "MI Preservation",

        "Composite Ranking"

    ]

})

# =============================================================================
# Best Model Per Dataset
# =============================================================================

print("\n" + "=" * 80)
print("BEST MODEL FOR EACH DATASET")
print("=" * 80)

display(

    best_models.round(6)

)

# =============================================================================
# Overall Best Model
# =============================================================================

overall_best_model = overall_multivariate.reset_index().iloc[0]

print("\n" + "=" * 80)
print("OVERALL BEST MODEL")
print("=" * 80)

print(f"Model               : {overall_best_model['Model']}")
print(f"Overall Rank        : {int(overall_best_model['Overall Rank'])}")
print(f"Average MAE         : {overall_best_model['MAE']:.6f}")
print(f"Average RMSE        : {overall_best_model['RMSE']:.6f}")
print(f"Average Frobenius   : {overall_best_model['Frobenius']:.6f}")
print(f"Average MI MAE      : {overall_best_model['MI MAE']:.6f}")
print(f"Average MI RMSE     : {overall_best_model['MI RMSE']:.6f}")
print(f"Cosine Similarity   : {overall_best_model['Cosine Similarity']:.6f}")
print(f"Pearson Correlation : {overall_best_model['Pearson Correlation']:.6f}")
print(f"Overall Score       : {overall_best_model['Overall Score']:.6f}")

# =============================================================================
# Evaluation Coverage
# =============================================================================

summary_statistics = pd.DataFrame({

    "Metric":[

        "Datasets",

        "Synthetic Models",

        "Covariance Metrics",

        "PCA Similarity",

        "Mutual Information",

        "Overall Rankings"

    ],

    "Value":[

        len(original_datasets),

        len(MODELS),

        len(covariance_summary),

        len(pca_summary),

        len(mi_summary),

        len(overall_multivariate)

    ]

})

print("\n" + "=" * 80)
print("EVALUATION COVERAGE")
print("=" * 80)

display(summary_statistics)

# =============================================================================
# Save Publication Summary
# =============================================================================

publication_summary_file = (

    CSV_DIR /

    "publication_summary.csv"

)

summary_statistics_file = (

    CSV_DIR /

    "evaluation_statistics.csv"

)

publication_summary.to_csv(

    publication_summary_file,

    index=False

)

summary_statistics.to_csv(

    summary_statistics_file,

    index=False

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("FILES GENERATED")
print("=" * 80)

generated_files = [

    "covariance_summary.csv",

    "pca_similarity_summary.csv",

    "mutual_information_summary.csv",

    "multivariate_summary.csv",

    "overall_multivariate_ranking.csv",

    "best_multivariate_models.csv",

    "publication_summary.csv",

    "evaluation_statistics.csv"

]

for file_name in generated_files:

    file_path = CSV_DIR / file_name

    if file_path.exists():

        print(f"✓ {file_name}")

    else:

        print(f"✗ {file_name}")

# =============================================================================
# Completion Report
# =============================================================================

print("\n" + "=" * 80)
print("MULTIVARIATE SIMILARITY EVALUATION COMPLETED")
print("=" * 80)

print(f"Datasets Evaluated        : {len(original_datasets)}")
print(f"Synthetic Models          : {len(MODELS)}")
print(f"Covariance Analysis       : Completed")
print(f"PCA Similarity            : Completed")
print(f"Mutual Information        : Completed")
print(f"Overall Ranking           : Completed")
print(f"Results Directory         : {MULTIVARIATE_DIR}")
print(f"CSV Directory             : {CSV_DIR}")

print("\nNotebook 08 - Section 8.8 completed successfully.")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.1 : Imports & Configuration
# =============================================================================

print("=" * 80)
print("SECTION 8.9.1 : IMPORTS & CONFIGURATION")
print("=" * 80)

# =============================================================================
# Standard Libraries
# =============================================================================

import os
import random
import warnings
from pathlib import Path

# =============================================================================
# Numerical Libraries
# =============================================================================

import numpy as np
import pandas as pd

# =============================================================================
# Visualization Libraries
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib as mpl

# =============================================================================
# Machine Learning
# =============================================================================

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline

# -----------------------------------------------------------------------------
# Classifiers
# -----------------------------------------------------------------------------

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.svm import SVC

from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

# -----------------------------------------------------------------------------
# Evaluation Metrics
# -----------------------------------------------------------------------------

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    balanced_accuracy_score,

    matthews_corrcoef,

    roc_curve,

    precision_recall_curve,

    auc,

    confusion_matrix,

    classification_report

)

# =============================================================================
# Display Options
# =============================================================================

pd.set_option(

    "display.max_columns",

    None

)

pd.set_option(

    "display.width",

    1000

)

pd.set_option(

    "display.float_format",

    lambda x: f"{x:.6f}"

)

# =============================================================================
# Random Seed
# =============================================================================

random.seed(SEED)

np.random.seed(SEED)

# =============================================================================
# Warnings
# =============================================================================

warnings.filterwarnings("ignore")

# =============================================================================
# Output Directories
# =============================================================================

ML_UTILITY_DIR = (

    RESULTS_DIR /

    "evaluation" /

    "machine_learning_utility"

)

CSV_DIR = (

    ML_UTILITY_DIR /

    "csv"

)

FIGURE_DIR = (

    ML_UTILITY_DIR /

    "figures"

)

MODEL_DIR = (

    ML_UTILITY_DIR /

    "trained_models"

)

ROC_DIR = (

    FIGURE_DIR /

    "roc_curves"

)

PR_DIR = (

    FIGURE_DIR /

    "precision_recall"

)

BAR_DIR = (

    FIGURE_DIR /

    "barplots"

)

HEATMAP_DIR = (

    FIGURE_DIR /

    "heatmaps"

)

for directory in [

    ML_UTILITY_DIR,

    CSV_DIR,

    FIGURE_DIR,

    MODEL_DIR,

    ROC_DIR,

    PR_DIR,

    BAR_DIR,

    HEATMAP_DIR

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

# =============================================================================
# Classifiers Used
# =============================================================================

CLASSIFIERS = [

    "Random Forest",

    "XGBoost",

    "LightGBM",

    "Logistic Regression",

    "Support Vector Machine",

    "Neural Network"

]

# =============================================================================
# Datasets
# =============================================================================

DATASETS = [

    "Adult Income",

    "Bank Marketing",

    "Breast Cancer"

]

# =============================================================================
# Synthetic Models
# =============================================================================

MODELS = [

    "Gaussian Multivariate",

    "Gaussian Copula",

    "CTGAN",

    "TVAE",

    "DP-CTGAN",

    "SPP-GAN"

]

# =============================================================================
# Evaluation Metrics
# =============================================================================

UTILITY_METRICS = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC-AUC",

    "Balanced Accuracy",

    "MCC"

]

# =============================================================================
# Environment Verification
# =============================================================================

print("\nEnvironment Verification")
print("-" * 80)

print(f"Python Version        : {os.sys.version.split()[0]}")

print(f"NumPy Version         : {np.__version__}")

print(f"Pandas Version        : {pd.__version__}")

print(f"Datasets             : {len(DATASETS)}")

print(f"Synthetic Models     : {len(MODELS)}")

print(f"Classifiers          : {len(CLASSIFIERS)}")

print(f"Evaluation Metrics   : {len(UTILITY_METRICS)}")

print(f"Random Seed          : {SEED}")

print(f"Device               : {DEVICE}")

print(f"\nOutput Directory")

print(ML_UTILITY_DIR)

print("\nClassification Models")

for i, classifier in enumerate(CLASSIFIERS, 1):

    print(f"{i}. {classifier}")

print("\nEvaluation Metrics")

for metric in UTILITY_METRICS:

    print(f"• {metric}")

print("\n" + "=" * 80)
print("SECTION 8.9.1 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.2 : Helper Functions
# =============================================================================

print("=" * 80)
print("SECTION 8.9.2 : HELPER FUNCTIONS")
print("=" * 80)

# =============================================================================
# Numeric Columns
# =============================================================================

def get_numeric_dataframe(df):

    """
    Keep only numeric columns.
    """

    return df.select_dtypes(include=[np.number]).copy()


# =============================================================================
# Feature / Target Split
# =============================================================================

def split_features_target(df, target_column):

    X = df.drop(columns=[target_column])

    y = df[target_column]

    return X, y


# =============================================================================
# Encode Target
# =============================================================================

def encode_target(y_train, y_test):

    encoder = LabelEncoder()

    y_train = encoder.fit_transform(y_train)

    y_test = encoder.transform(y_test)

    return y_train, y_test, encoder


# =============================================================================
# Missing Value Imputation
# =============================================================================

def impute_features(X_train, X_test):

    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(

        imputer.fit_transform(X_train),

        columns=X_train.columns

    )

    X_test = pd.DataFrame(

        imputer.transform(X_test),

        columns=X_test.columns

    )

    return X_train, X_test, imputer


# =============================================================================
# Feature Scaling
# =============================================================================

def scale_features(X_train, X_test):

    scaler = StandardScaler()

    X_train = pd.DataFrame(

        scaler.fit_transform(X_train),

        columns=X_train.columns

    )

    X_test = pd.DataFrame(

        scaler.transform(X_test),

        columns=X_test.columns

    )

    return X_train, X_test, scaler


# =============================================================================
# Complete Preprocessing
# =============================================================================

def prepare_data(

    train_df,

    test_df,

    target_column

):

    X_train, y_train = split_features_target(

        train_df,

        target_column

    )

    X_test, y_test = split_features_target(

        test_df,

        target_column

    )

    X_train, X_test, imputer = impute_features(

        X_train,

        X_test

    )

    X_train, X_test, scaler = scale_features(

        X_train,

        X_test

    )

    y_train, y_test, encoder = encode_target(

        y_train,

        y_test

    )

    return (

        X_train,

        X_test,

        y_train,

        y_test,

        encoder,

        scaler,

        imputer

    )


# =============================================================================
# Classifier Factory
# =============================================================================

def get_classifier(name):

    if name == "Random Forest":

        return RandomForestClassifier(

            n_estimators=300,

            random_state=SEED,

            n_jobs=-1

        )

    elif name == "XGBoost":

        return XGBClassifier(

            n_estimators=300,

            learning_rate=0.05,

            max_depth=6,

            random_state=SEED,

            eval_metric="logloss"

        )

    elif name == "LightGBM":

        return LGBMClassifier(

            n_estimators=300,

            learning_rate=0.05,

            random_state=SEED,

            verbose=-1

        )

    elif name == "Logistic Regression":

        return LogisticRegression(

            max_iter=5000,

            random_state=SEED

        )

    elif name == "Support Vector Machine":

        return SVC(

            probability=True,

            random_state=SEED

        )

    elif name == "Neural Network":

        return MLPClassifier(

            hidden_layer_sizes=(128,64),

            max_iter=500,

            random_state=SEED

        )

    else:

        raise ValueError(f"Unknown classifier: {name}")


# =============================================================================
# Prediction Probabilities
# =============================================================================

def predict_probability(model, X):

    if hasattr(model, "predict_proba"):

        return model.predict_proba(X)[:,1]

    elif hasattr(model, "decision_function"):

        scores = model.decision_function(X)

        scores = (

            scores -

            scores.min()

        ) / (

            scores.max() -

            scores.min()

        )

        return scores

    else:

        return None


# =============================================================================
# Evaluate Classifier
# =============================================================================

def evaluate_classifier(

    model,

    X_test,

    y_test

):

    predictions = model.predict(X_test)

    probabilities = predict_probability(

        model,

        X_test

    )

    metrics = {

        "Accuracy":

            accuracy_score(

                y_test,

                predictions

            ),

        "Precision":

            precision_score(

                y_test,

                predictions,

                zero_division=0

            ),

        "Recall":

            recall_score(

                y_test,

                predictions,

                zero_division=0

            ),

        "F1 Score":

            f1_score(

                y_test,

                predictions,

                zero_division=0

            ),

        "Balanced Accuracy":

            balanced_accuracy_score(

                y_test,

                predictions

            ),

        "MCC":

            matthews_corrcoef(

                y_test,

                predictions

            )

    }

    if probabilities is not None:

        metrics["ROC-AUC"] = roc_auc_score(

            y_test,

            probabilities

        )

    else:

        metrics["ROC-AUC"] = np.nan

    return (

        metrics,

        predictions,

        probabilities

    )


# =============================================================================
# Convert Results to DataFrame
# =============================================================================

def metrics_to_dataframe(results):

    return pd.DataFrame(results).round(6)


# =============================================================================
# Verification
# =============================================================================

print("\nAvailable Helper Functions")
print("-"*80)

functions = [

    "get_numeric_dataframe()",

    "split_features_target()",

    "encode_target()",

    "impute_features()",

    "scale_features()",

    "prepare_data()",

    "get_classifier()",

    "predict_probability()",

    "evaluate_classifier()",

    "metrics_to_dataframe()"

]

for function in functions:

    print("✓", function)

print("\n" + "=" * 80)
print("SECTION 8.9.2 COMPLETED")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.3 : Prepare Training & Test Data
# =============================================================================

print("=" * 80)
print("SECTION 8.9.3 : PREPARE TRAINING & TEST DATA")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

prepared_datasets = {}

dataset_summary = []

# =============================================================================
# Prepare Datasets
# =============================================================================

for dataset_name in DATASETS:

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    prepared_datasets[dataset_name] = {}

    # -------------------------------------------------------------------------
    # Original Train/Test
    # -------------------------------------------------------------------------

    train_df = original_datasets[dataset_name]["train"].copy()

    test_df = original_datasets[dataset_name]["test"].copy()

    # -------------------------------------------------------------------------
    # Target Column
    # -------------------------------------------------------------------------

    target_column = train_df.columns[-1]

    print(f"Target Column : {target_column}")

    # -------------------------------------------------------------------------
    # Prepare Original Data
    # -------------------------------------------------------------------------

    (
        X_train,
        X_test,
        y_train,
        y_test,
        label_encoder,
        scaler,
        imputer

    ) = prepare_data(

        train_df=train_df,

        test_df=test_df,

        target_column=target_column

    )

    prepared_datasets[dataset_name]["Original"] = {

        "X_train": X_train,

        "X_test": X_test,

        "y_train": y_train,

        "y_test": y_test,

        "target_column": target_column,

        "label_encoder": label_encoder,

        "scaler": scaler,

        "imputer": imputer

    }

    print(f"Original Train Shape : {X_train.shape}")
    print(f"Original Test Shape  : {X_test.shape}")

    # -------------------------------------------------------------------------
    # Prepare Synthetic Datasets
    # -------------------------------------------------------------------------

    for model_name in MODELS:

        if model_name not in synthetic_datasets[dataset_name]:

            continue

        synthetic_df = synthetic_datasets[dataset_name][model_name].copy()

        X_syn = synthetic_df.drop(

            columns=[target_column]

        )

        y_syn = synthetic_df[target_column]

        # -------------------------------------------------------------
        # Use Original Preprocessing
        # -------------------------------------------------------------

        X_syn = pd.DataFrame(

            imputer.transform(X_syn),

            columns=X_syn.columns

        )

        X_syn = pd.DataFrame(

            scaler.transform(X_syn),

            columns=X_syn.columns

        )

        y_syn = label_encoder.transform(y_syn)

        prepared_datasets[dataset_name][model_name] = {

            "X_train": X_syn,

            "y_train": y_syn,

            "X_test": X_test,

            "y_test": y_test,

            "target_column": target_column

        }

        dataset_summary.append({

            "Dataset": dataset_name,

            "Model": model_name,

            "Training Samples": len(X_syn),

            "Testing Samples": len(X_test),

            "Features": X_syn.shape[1]

        })

        print(

            f"{model_name:<25}"

            f"Train={len(X_syn):>6}   "

            f"Features={X_syn.shape[1]}"

        )

# =============================================================================
# Summary Table
# =============================================================================

dataset_summary = pd.DataFrame(

    dataset_summary

)

print("\n" + "=" * 80)
print("Prepared Dataset Summary")
print("=" * 80)

display(

    dataset_summary

)

# =============================================================================
# Verification
# =============================================================================

print("\nVerification")
print("-" * 80)

for dataset_name in DATASETS:

    feature_counts = []

    for model_name in prepared_datasets[dataset_name]:

        feature_counts.append(

            prepared_datasets[dataset_name][model_name]["X_train"].shape[1]

        )

    consistent = len(set(feature_counts)) == 1

    print(

        f"{dataset_name:<20}"

        f"Feature Consistency : {consistent}"

    )

print("\n" + "=" * 80)
print("SECTION 8.9.3 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.4 : Define Downstream Classifiers
# =============================================================================

print("=" * 80)
print("SECTION 8.9.4 : DEFINE DOWNSTREAM CLASSIFIERS")
print("=" * 80)

# =============================================================================
# Downstream Classifiers
# =============================================================================

CLASSIFIERS = {

    # -------------------------------------------------------------------------
    # Logistic Regression
    # -------------------------------------------------------------------------

    "Logistic Regression": LogisticRegression(

        max_iter=5000,

        solver="lbfgs",

        random_state=SEED,

        n_jobs=-1

    ),

    # -------------------------------------------------------------------------
    # Random Forest
    # -------------------------------------------------------------------------

    "Random Forest": RandomForestClassifier(

        n_estimators=300,

        criterion="gini",

        max_depth=None,

        min_samples_split=2,

        min_samples_leaf=1,

        bootstrap=True,

        random_state=SEED,

        n_jobs=-1

    ),

    # -------------------------------------------------------------------------
    # Support Vector Machine
    # -------------------------------------------------------------------------

    "Support Vector Machine": SVC(

        kernel="rbf",

        C=1.0,

        gamma="scale",

        probability=True,

        random_state=SEED

    ),

    # -------------------------------------------------------------------------
    # XGBoost
    # -------------------------------------------------------------------------

    "XGBoost": XGBClassifier(

        n_estimators=300,

        learning_rate=0.05,

        max_depth=6,

        subsample=0.80,

        colsample_bytree=0.80,

        objective="binary:logistic",

        eval_metric="logloss",

        random_state=SEED,

        n_jobs=-1,

        verbosity=0

    ),

    # -------------------------------------------------------------------------
    # LightGBM
    # -------------------------------------------------------------------------

    "LightGBM": LGBMClassifier(

        n_estimators=300,

        learning_rate=0.05,

        num_leaves=31,

        subsample=0.80,

        colsample_bytree=0.80,

        random_state=SEED,

        verbose=-1

    ),

    # -------------------------------------------------------------------------
    # Multi-Layer Perceptron
    # -------------------------------------------------------------------------

    "Neural Network": MLPClassifier(

        hidden_layer_sizes=(128, 64),

        activation="relu",

        solver="adam",

        learning_rate_init=0.001,

        batch_size=128,

        max_iter=500,

        early_stopping=True,

        validation_fraction=0.10,

        random_state=SEED

    )

}

# =============================================================================
# Verification
# =============================================================================

print("\nAvailable Classifiers")
print("-" * 80)

for i, (name, model) in enumerate(CLASSIFIERS.items(), start=1):

    print(f"{i}. {name}")

    print(f"   Class : {model.__class__.__name__}")

print("\n" + "=" * 80)
print("Classifier Summary")
print("=" * 80)

print(f"Total Classifiers : {len(CLASSIFIERS)}")

print(f"Random Seed       : {SEED}")

print("\nClassifiers Used")

for classifier in CLASSIFIERS.keys():

    print(f"✓ {classifier}")

print("\n" + "=" * 80)
print("SECTION 8.9.4 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.5 : Train on Original Data (Reference Baseline)
# =============================================================================

print("=" * 80)
print("SECTION 8.9.5 : ORIGINAL DATA REFERENCE BASELINE")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

baseline_models = {}

baseline_predictions = {}

baseline_results = []

# =============================================================================
# Train Reference Models
# =============================================================================

for dataset_name in DATASETS:

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    baseline_models[dataset_name] = {}
    baseline_predictions[dataset_name] = {}

    X_train = prepared_datasets[dataset_name]["Original"]["X_train"]
    X_test  = prepared_datasets[dataset_name]["Original"]["X_test"]

    y_train = prepared_datasets[dataset_name]["Original"]["y_train"]
    y_test  = prepared_datasets[dataset_name]["Original"]["y_test"]

    for classifier_name, classifier in CLASSIFIERS.items():

        print(f"Training : {classifier_name}")

        # ---------------------------------------------------------------------
        # Train
        # ---------------------------------------------------------------------

        model = classifier

        model.fit(
            X_train,
            y_train
        )

        # ---------------------------------------------------------------------
        # Evaluate
        # ---------------------------------------------------------------------

        metrics, predictions, probabilities = evaluate_classifier(

            model,

            X_test,

            y_test

        )

        # ---------------------------------------------------------------------
        # Save Model
        # ---------------------------------------------------------------------

        baseline_models[dataset_name][classifier_name] = model

        baseline_predictions[dataset_name][classifier_name] = {

            "y_true": y_test,

            "y_pred": predictions,

            "y_prob": probabilities

        }

        # ---------------------------------------------------------------------
        # Save Metrics
        # ---------------------------------------------------------------------

        baseline_results.append({

            "Dataset": dataset_name,

            "Training Data": "Original",

            "Classifier": classifier_name,

            "Accuracy": metrics["Accuracy"],

            "Precision": metrics["Precision"],

            "Recall": metrics["Recall"],

            "F1 Score": metrics["F1 Score"],

            "ROC-AUC": metrics["ROC-AUC"],

            "Balanced Accuracy": metrics["Balanced Accuracy"],

            "MCC": metrics["MCC"]

        })

        print(

            f"Accuracy={metrics['Accuracy']:.4f}   "

            f"F1={metrics['F1 Score']:.4f}"

        )

# =============================================================================
# Results DataFrame
# =============================================================================

baseline_results = pd.DataFrame(

    baseline_results

)

baseline_results = baseline_results.sort_values(

    [

        "Dataset",

        "Accuracy"

    ],

    ascending=[

        True,

        False

    ]

).reset_index(

    drop=True

)

# =============================================================================
# Dataset Ranking
# =============================================================================

baseline_results["Rank"] = (

    baseline_results

    .groupby("Dataset")["Accuracy"]

    .rank(

        method="dense",

        ascending=False

    )

)

# =============================================================================
# Overall Ranking
# =============================================================================

overall_baseline = (

    baseline_results

    .groupby("Classifier")

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean"

    })

)

overall_baseline = (

    overall_baseline

    .sort_values(

        "Accuracy",

        ascending=False

    )

)

overall_baseline.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_baseline)+1

    )

)

# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 80)
print("Dataset-wise Baseline Results")
print("=" * 80)

display(

    baseline_results.round(6)

)

print("\n" + "=" * 80)
print("Overall Baseline Ranking")
print("=" * 80)

display(

    overall_baseline.round(6)

)

# =============================================================================
# Best Baseline Model Per Dataset
# =============================================================================

best_baseline = (

    baseline_results

    .sort_values(

        [

            "Dataset",

            "Accuracy"

        ],

        ascending=[

            True,

            False

        ]

    )

    .groupby("Dataset")

    .first()

    .reset_index()

)

print("\n" + "=" * 80)
print("Best Baseline Classifier")
print("=" * 80)

display(

    best_baseline.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(f"Datasets Evaluated : {len(DATASETS)}")
print(f"Classifiers        : {len(CLASSIFIERS)}")
print(f"Models Trained     : {len(baseline_results)}")

print("\n" + "=" * 80)
print("SECTION 8.9.5 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.6 : Train on Synthetic Data -> Test on Real Data
# =============================================================================

print("=" * 80)
print("SECTION 8.9.6 : TRAIN ON SYNTHETIC DATA -> TEST ON REAL DATA")
print("=" * 80)

from sklearn.base import clone

# =============================================================================
# Containers
# =============================================================================

synthetic_models = {}

synthetic_predictions = {}

synthetic_results = []

# =============================================================================
# Train on Synthetic Data
# =============================================================================

for dataset_name in DATASETS:

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    synthetic_models[dataset_name] = {}
    synthetic_predictions[dataset_name] = {}

    for model_name in MODELS:

        # Skip unavailable models
        if model_name not in prepared_datasets[dataset_name]:
            continue

        print(f"\nSynthetic Model : {model_name}")

        synthetic_models[dataset_name][model_name] = {}
        synthetic_predictions[dataset_name][model_name] = {}

        # --------------------------------------------------------------
        # Synthetic Training Data
        # --------------------------------------------------------------

        X_train = prepared_datasets[dataset_name][model_name]["X_train"]
        y_train = prepared_datasets[dataset_name][model_name]["y_train"]

        # --------------------------------------------------------------
        # Original Test Data
        # --------------------------------------------------------------

        X_test = prepared_datasets[dataset_name]["Original"]["X_test"]
        y_test = prepared_datasets[dataset_name]["Original"]["y_test"]

        # --------------------------------------------------------------
        # Train Every Classifier
        # --------------------------------------------------------------

        for classifier_name, classifier in CLASSIFIERS.items():

            print(f"Training : {classifier_name}")

            model = clone(classifier)

            model.fit(
                X_train,
                y_train
            )

            metrics, predictions, probabilities = evaluate_classifier(

                model,

                X_test,

                y_test

            )

            synthetic_models[dataset_name][model_name][classifier_name] = model

            synthetic_predictions[dataset_name][model_name][classifier_name] = {

                "y_true": y_test,

                "y_pred": predictions,

                "y_prob": probabilities

            }

            synthetic_results.append({

                "Dataset": dataset_name,

                "Synthetic Model": model_name,

                "Classifier": classifier_name,

                "Accuracy": metrics["Accuracy"],

                "Precision": metrics["Precision"],

                "Recall": metrics["Recall"],

                "F1 Score": metrics["F1 Score"],

                "ROC-AUC": metrics["ROC-AUC"],

                "Balanced Accuracy": metrics["Balanced Accuracy"],

                "MCC": metrics["MCC"]

            })

            print(
                f"Accuracy={metrics['Accuracy']:.4f}   "
                f"F1={metrics['F1 Score']:.4f}"
            )

# =============================================================================
# Results DataFrame
# =============================================================================

synthetic_results = pd.DataFrame(synthetic_results)

synthetic_results = synthetic_results.sort_values(

    [

        "Dataset",

        "Synthetic Model",

        "Accuracy"

    ],

    ascending=[

        True,

        True,

        False

    ]

).reset_index(drop=True)

# =============================================================================
# Ranking
# =============================================================================

synthetic_results["Rank"] = (

    synthetic_results

    .groupby(

        [

            "Dataset",

            "Synthetic Model"

        ]

    )["Accuracy"]

    .rank(

        method="dense",

        ascending=False

    )

)

# =============================================================================
# Best Classifier Per Synthetic Model
# =============================================================================

best_synthetic = (

    synthetic_results

    .sort_values(

        [

            "Dataset",

            "Synthetic Model",

            "Accuracy"

        ],

        ascending=[

            True,

            True,

            False

        ]

    )

    .groupby(

        [

            "Dataset",

            "Synthetic Model"

        ]

    )

    .first()

    .reset_index()

)

# =============================================================================
# Overall Ranking Across Synthetic Models
# =============================================================================

overall_synthetic = (

    synthetic_results

    .groupby(

        [

            "Synthetic Model",

            "Classifier"

        ]

    )

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean"

    })

    .reset_index()

)

overall_synthetic = overall_synthetic.sort_values(

    "Accuracy",

    ascending=False

)

# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 80)
print("Synthetic Utility Results")
print("=" * 80)

display(

    synthetic_results.round(6)

)

print("\n" + "=" * 80)
print("Best Classifier For Each Synthetic Model")
print("=" * 80)

display(

    best_synthetic.round(6)

)

print("\n" + "=" * 80)
print("Overall Synthetic Ranking")
print("=" * 80)

display(

    overall_synthetic.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(f"Datasets                 : {len(DATASETS)}")
print(f"Synthetic Models         : {len(MODELS)}")
print(f"Classifiers              : {len(CLASSIFIERS)}")
print(f"Experiments Completed    : {len(synthetic_results)}")

print("\n" + "=" * 80)
print("SECTION 8.9.6 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.7 : Calculate Evaluation Metrics
# =============================================================================

print("=" * 80)
print("SECTION 8.9.7 : CALCULATE EVALUATION METRICS")
print("=" * 80)

# =============================================================================
# Prepare Baseline Results
# =============================================================================

baseline_metrics = baseline_results.copy()

baseline_metrics = baseline_metrics.rename(

    columns={

        "Training Data": "Synthetic Model"

    }

)

baseline_metrics["Synthetic Model"] = "Original"

# =============================================================================
# Merge Original + Synthetic Results
# =============================================================================

utility_results = pd.concat(

    [

        baseline_metrics,

        synthetic_results

    ],

    ignore_index=True

)

# =============================================================================
# Reorder Columns
# =============================================================================

utility_results = utility_results[

    [

        "Dataset",

        "Synthetic Model",

        "Classifier",

        "Accuracy",

        "Precision",

        "Recall",

        "F1 Score",

        "ROC-AUC",

        "Balanced Accuracy",

        "MCC"

    ]

]

# =============================================================================
# Baseline Lookup
# =============================================================================

baseline_lookup = (

    utility_results

    [

        utility_results["Synthetic Model"] == "Original"

    ]

    [

        [

            "Dataset",

            "Classifier",

            "Accuracy",

            "F1 Score",

            "ROC-AUC",

            "Balanced Accuracy",

            "MCC"

        ]

    ]

)

baseline_lookup = baseline_lookup.rename(

    columns={

        "Accuracy":"Baseline Accuracy",

        "F1 Score":"Baseline F1",

        "ROC-AUC":"Baseline ROC",

        "Balanced Accuracy":"Baseline Balanced Accuracy",

        "MCC":"Baseline MCC"

    }

)

# =============================================================================
# Merge Baseline
# =============================================================================

utility_results = utility_results.merge(

    baseline_lookup,

    on=[

        "Dataset",

        "Classifier"

    ],

    how="left"

)

# =============================================================================
# Utility Preservation (%)
# =============================================================================

utility_results["Accuracy Retention (%)"] = (

    utility_results["Accuracy"]

    /

    utility_results["Baseline Accuracy"]

) * 100

utility_results["F1 Retention (%)"] = (

    utility_results["F1 Score"]

    /

    utility_results["Baseline F1"]

) * 100

utility_results["ROC Retention (%)"] = (

    utility_results["ROC-AUC"]

    /

    utility_results["Baseline ROC"]

) * 100

utility_results["Balanced Accuracy Retention (%)"] = (

    utility_results["Balanced Accuracy"]

    /

    utility_results["Baseline Balanced Accuracy"]

) * 100

utility_results["MCC Retention (%)"] = (

    utility_results["MCC"]

    /

    utility_results["Baseline MCC"]

) * 100

# =============================================================================
# Composite Utility Score
# =============================================================================

utility_results["Utility Score"] = (

    utility_results["Accuracy"]

    +

    utility_results["Precision"]

    +

    utility_results["Recall"]

    +

    utility_results["F1 Score"]

    +

    utility_results["ROC-AUC"]

    +

    utility_results["Balanced Accuracy"]

    +

    utility_results["MCC"]

) / 7

# =============================================================================
# Ranking
# =============================================================================

utility_results["Rank"] = (

    utility_results

    .groupby(

        [

            "Dataset",

            "Classifier"

        ]

    )["Utility Score"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# =============================================================================
# Overall Summary
# =============================================================================

utility_summary = (

    utility_results

    .groupby(

        "Synthetic Model"

    )

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean",

        "Utility Score":"mean",

        "Accuracy Retention (%)":"mean",

        "F1 Retention (%)":"mean"

    })

)

utility_summary = utility_summary.sort_values(

    "Utility Score",

    ascending=False

)

utility_summary.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(utility_summary)+1

    )

)

# =============================================================================
# Best Synthetic Generator Per Dataset
# =============================================================================

best_generator = (

    utility_results

    [

        utility_results["Synthetic Model"] != "Original"

    ]

    .groupby(

        [

            "Dataset",

            "Synthetic Model"

        ]

    )["Utility Score"]

    .mean()

    .reset_index()

)

best_generator = (

    best_generator

    .sort_values(

        [

            "Dataset",

            "Utility Score"

        ],

        ascending=[

            True,

            False

        ]

    )

    .groupby("Dataset")

    .first()

    .reset_index()

)

# =============================================================================
# Display
# =============================================================================

print("\nDataset-wise Utility Results")

display(

    utility_results.round(6)

)

print("\nOverall Utility Summary")

display(

    utility_summary.round(6)

)

print("\nBest Synthetic Generator")

display(

    best_generator.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)

print("Verification")

print("=" * 80)

print(f"Total Experiments : {len(utility_results)}")

print(f"Datasets          : {utility_results['Dataset'].nunique()}")

print(f"Generators        : {utility_results['Synthetic Model'].nunique()}")

print(f"Classifiers       : {utility_results['Classifier'].nunique()}")

print("\n" + "=" * 80)

print("SECTION 8.9.7 COMPLETED SUCCESSFULLY")

print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.8 : ROC Curve Comparison
# =============================================================================

print("="*80)
print("SECTION 8.9.8 : ROC CURVE COMPARISON")
print("="*80)

from sklearn.metrics import roc_curve, roc_auc_score

roc_summary = []

# =============================================================================
# Create ROC directory
# =============================================================================

ROC_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Plot ROC Curves
# =============================================================================

for dataset in DATASETS:

    for classifier in CLASSIFIERS.keys():

        plt.figure(figsize=(8,7))

        # ----------------------------------------------------------
        # Baseline
        # ----------------------------------------------------------

        base = baseline_predictions[dataset][classifier]

        if base["y_prob"] is not None:

            fpr, tpr, _ = roc_curve(

                base["y_true"],

                base["y_prob"]

            )

            auc_score = roc_auc_score(

                base["y_true"],

                base["y_prob"]

            )

            roc_summary.append({

                "Dataset":dataset,

                "Model":"Original",

                "Classifier":classifier,

                "ROC-AUC":auc_score

            })

            plt.plot(

                fpr,

                tpr,

                linewidth=3,

                label=f"Original (AUC={auc_score:.3f})"

            )

        # ----------------------------------------------------------
        # Synthetic Models
        # ----------------------------------------------------------

        for model_name in MODELS:

            if model_name not in synthetic_predictions[dataset]:

                continue

            pred = synthetic_predictions[dataset][model_name][classifier]

            if pred["y_prob"] is None:

                continue

            fpr, tpr, _ = roc_curve(

                pred["y_true"],

                pred["y_prob"]

            )

            auc_score = roc_auc_score(

                pred["y_true"],

                pred["y_prob"]

            )

            roc_summary.append({

                "Dataset":dataset,

                "Model":model_name,

                "Classifier":classifier,

                "ROC-AUC":auc_score

            })

            plt.plot(

                fpr,

                tpr,

                linewidth=2,

                label=f"{model_name} ({auc_score:.3f})"

            )

        # ----------------------------------------------------------
        # Random Classifier
        # ----------------------------------------------------------

        plt.plot(

            [0,1],

            [0,1],

            "--",

            color="black",

            linewidth=1,

            label="Random"

        )

        plt.xlabel("False Positive Rate", fontsize=12)

        plt.ylabel("True Positive Rate", fontsize=12)

        plt.title(

            f"{dataset}\n{classifier}",

            fontsize=14,

            fontweight="bold"

        )

        plt.grid(alpha=0.3)

        plt.legend(fontsize=8)

        plt.tight_layout()

        figure_name = (

            ROC_DIR /

            f"{dataset}_{classifier}_ROC.png"

        )

        plt.savefig(

            figure_name,

            dpi=600,

            bbox_inches="tight"

        )

        plt.close()

# =============================================================================
# ROC Summary Table
# =============================================================================

roc_summary = pd.DataFrame(

    roc_summary

)

roc_summary["Rank"] = (

    roc_summary

    .groupby(

        [

            "Dataset",

            "Classifier"

        ]

    )["ROC-AUC"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# =============================================================================
# Overall ROC Ranking
# =============================================================================

overall_roc = (

    roc_summary

    .groupby("Model")

    .agg({

        "ROC-AUC":"mean"

    })

)

overall_roc = overall_roc.sort_values(

    "ROC-AUC",

    ascending=False

)

overall_roc.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_roc)+1

    )

)

# =============================================================================
# Display
# =============================================================================

print("\nROC Summary")

display(

    roc_summary.round(6)

)

print("\nOverall ROC Ranking")

display(

    overall_roc.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n"+"="*80)

print("Verification")

print("="*80)

print(f"ROC Figures Generated : {len(DATASETS)*len(CLASSIFIERS)}")

print(f"Summary Rows          : {len(roc_summary)}")

print(f"Output Folder")

print(ROC_DIR)

print("\n"+"="*80)

print("SECTION 8.9.8 COMPLETED SUCCESSFULLY")

print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.9 : Precision-Recall Curves
# =============================================================================

print("=" * 80)
print("SECTION 8.9.9 : PRECISION-RECALL CURVES")
print("=" * 80)

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

# =============================================================================
# Create Output Directory
# =============================================================================

PR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# Container
# =============================================================================

pr_summary = []

# =============================================================================
# Generate Precision-Recall Curves
# =============================================================================

for dataset in DATASETS:

    for classifier in CLASSIFIERS.keys():

        plt.figure(figsize=(8,7))

        # ---------------------------------------------------------------------
        # Original Baseline
        # ---------------------------------------------------------------------

        base = baseline_predictions[dataset][classifier]

        if base["y_prob"] is not None:

            precision, recall, _ = precision_recall_curve(

                base["y_true"],

                base["y_prob"]

            )

            ap = average_precision_score(

                base["y_true"],

                base["y_prob"]

            )

            pr_summary.append({

                "Dataset":dataset,

                "Model":"Original",

                "Classifier":classifier,

                "Average Precision":ap

            })

            plt.plot(

                recall,

                precision,

                linewidth=3,

                label=f"Original (AP={ap:.3f})"

            )

        # ---------------------------------------------------------------------
        # Synthetic Models
        # ---------------------------------------------------------------------

        for model_name in MODELS:

            if model_name not in synthetic_predictions[dataset]:

                continue

            prediction = synthetic_predictions[dataset][model_name][classifier]

            if prediction["y_prob"] is None:

                continue

            precision, recall, _ = precision_recall_curve(

                prediction["y_true"],

                prediction["y_prob"]

            )

            ap = average_precision_score(

                prediction["y_true"],

                prediction["y_prob"]

            )

            pr_summary.append({

                "Dataset":dataset,

                "Model":model_name,

                "Classifier":classifier,

                "Average Precision":ap

            })

            plt.plot(

                recall,

                precision,

                linewidth=2,

                label=f"{model_name} ({ap:.3f})"

            )

        # ---------------------------------------------------------------------
        # Figure Formatting
        # ---------------------------------------------------------------------

        plt.xlabel(

            "Recall",

            fontsize=12

        )

        plt.ylabel(

            "Precision",

            fontsize=12

        )

        plt.title(

            f"{dataset}\n{classifier}",

            fontsize=14,

            fontweight="bold"

        )

        plt.grid(alpha=0.30)

        plt.legend(

            fontsize=8

        )

        plt.tight_layout()

        filename = (

            PR_DIR /

            f"{dataset}_{classifier}_PR_Curve.png"

        )

        plt.savefig(

            filename,

            dpi=600,

            bbox_inches="tight"

        )

        plt.close()

# =============================================================================
# Summary DataFrame
# =============================================================================

pr_summary = pd.DataFrame(

    pr_summary

)

# =============================================================================
# Ranking
# =============================================================================

pr_summary["Rank"] = (

    pr_summary

    .groupby(

        [

            "Dataset",

            "Classifier"

        ]

    )["Average Precision"]

    .rank(

        ascending=False,

        method="dense"

    )

)

# =============================================================================
# Overall Summary
# =============================================================================

overall_pr = (

    pr_summary

    .groupby("Model")

    .agg({

        "Average Precision":"mean"

    })

)

overall_pr = overall_pr.sort_values(

    "Average Precision",

    ascending=False

)

overall_pr.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(overall_pr)+1

    )

)

# =============================================================================
# Best Generator
# =============================================================================

best_pr = (

    pr_summary

    [

        pr_summary["Model"]!="Original"

    ]

    .groupby(

        [

            "Dataset",

            "Model"

        ]

    )["Average Precision"]

    .mean()

    .reset_index()

)

best_pr = (

    best_pr

    .sort_values(

        [

            "Dataset",

            "Average Precision"

        ],

        ascending=[

            True,

            False

        ]

    )

    .groupby("Dataset")

    .first()

    .reset_index()

)

# =============================================================================
# Display
# =============================================================================

print("\nPrecision-Recall Summary")

display(

    pr_summary.round(6)

)

print("\nOverall Average Precision Ranking")

display(

    overall_pr.round(6)

)

print("\nBest Synthetic Generator")

display(

    best_pr.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)

print("Verification")

print("=" * 80)

print(f"Precision-Recall Figures : {len(DATASETS)*len(CLASSIFIERS)}")

print(f"Summary Rows             : {len(pr_summary)}")

print(f"Output Directory")

print(PR_DIR)

print("\n" + "=" * 80)

print("SECTION 8.9.9 COMPLETED SUCCESSFULLY")

print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.10 : Utility Summary Tables
# =============================================================================

print("=" * 80)
print("SECTION 8.9.10 : UTILITY SUMMARY TABLES")
print("=" * 80)

# =============================================================================
# Table 1 : Original Baseline
# =============================================================================

baseline_table = (

    baseline_results

    .sort_values(

        [

            "Dataset",

            "Accuracy"

        ],

        ascending=[

            True,

            False

        ]

    )

)

# =============================================================================
# Table 2 : Synthetic Results
# =============================================================================

synthetic_table = (

    synthetic_results

    .sort_values(

        [

            "Dataset",

            "Synthetic Model",

            "Accuracy"

        ],

        ascending=[

            True,

            True,

            False

        ]

    )

)

# =============================================================================
# Table 3 : Overall Generator Performance
# =============================================================================

generator_summary = (

    utility_results

    .groupby(

        "Synthetic Model"

    )

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean",

        "Utility Score":"mean"

    })

    .reset_index()

)

generator_summary = generator_summary.sort_values(

    "Utility Score",

    ascending=False

)

generator_summary.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(generator_summary)+1

    )

)

# =============================================================================
# Table 4 : Best Classifier for Each Generator
# =============================================================================

best_classifier_table = (

    utility_results

    .groupby(

        [

            "Dataset",

            "Synthetic Model",

            "Classifier"

        ]

    )

    .agg({

        "Utility Score":"mean"

    })

    .reset_index()

)

best_classifier_table = (

    best_classifier_table

    .sort_values(

        [

            "Dataset",

            "Synthetic Model",

            "Utility Score"

        ],

        ascending=[

            True,

            True,

            False

        ]

    )

)

best_classifier_table = (

    best_classifier_table

    .groupby(

        [

            "Dataset",

            "Synthetic Model"

        ]

    )

    .first()

    .reset_index()

)

# =============================================================================
# Table 5 : Overall Classifier Ranking
# =============================================================================

classifier_summary = (

    utility_results

    .groupby(

        "Classifier"

    )

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean",

        "Utility Score":"mean"

    })

    .reset_index()

)

classifier_summary = classifier_summary.sort_values(

    "Utility Score",

    ascending=False

)

classifier_summary.insert(

    0,

    "Overall Rank",

    range(

        1,

        len(classifier_summary)+1

    )

)

# =============================================================================
# Table 6 : Publication Summary
# =============================================================================

publication_summary = (

    utility_results

    .groupby(

        [

            "Synthetic Model"

        ]

    )

    .agg({

        "Accuracy":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean"

    })

)

publication_summary["Average Score"] = (

    publication_summary.mean(

        axis=1

    )

)

publication_summary = publication_summary.sort_values(

    "Average Score",

    ascending=False

)

publication_summary.insert(

    0,

    "Publication Rank",

    range(

        1,

        len(publication_summary)+1

    )

)

# =============================================================================
# Display
# =============================================================================

print("\n" + "=" * 80)
print("TABLE 1 : ORIGINAL BASELINE")
print("=" * 80)

display(

    baseline_table.round(6)

)

print("\n" + "=" * 80)
print("TABLE 2 : SYNTHETIC RESULTS")
print("=" * 80)

display(

    synthetic_table.round(6)

)

print("\n" + "=" * 80)
print("TABLE 3 : GENERATOR SUMMARY")
print("=" * 80)

display(

    generator_summary.round(6)

)

print("\n" + "=" * 80)
print("TABLE 4 : BEST CLASSIFIER")
print("=" * 80)

display(

    best_classifier_table.round(6)

)

print("\n" + "=" * 80)
print("TABLE 5 : CLASSIFIER SUMMARY")
print("=" * 80)

display(

    classifier_summary.round(6)

)

print("\n" + "=" * 80)
print("TABLE 6 : PUBLICATION SUMMARY")
print("=" * 80)

display(

    publication_summary.round(6)

)

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)
print("Verification")
print("=" * 80)

print(f"Baseline Results          : {len(baseline_table)}")
print(f"Synthetic Results         : {len(synthetic_table)}")
print(f"Generator Summary         : {len(generator_summary)}")
print(f"Best Classifier Records   : {len(best_classifier_table)}")
print(f"Classifier Summary        : {len(classifier_summary)}")
print(f"Publication Summary       : {len(publication_summary)}")

print("\n" + "=" * 80)
print("SECTION 8.9.10 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.11 : Save Results
# =============================================================================

print("=" * 80)
print("SECTION 8.9.11 : SAVE RESULTS")
print("=" * 80)

# =============================================================================
# Create Output Directories
# =============================================================================

UTILITY_DIR = RESULTS_DIR / "evaluation" / "machine_learning_utility"

CSV_DIR = UTILITY_DIR / "csv"

EXCEL_DIR = UTILITY_DIR / "excel"

CSV_DIR.mkdir(

    parents=True,

    exist_ok=True

)

EXCEL_DIR.mkdir(

    parents=True,

    exist_ok=True

)

# =============================================================================
# CSV Files
# =============================================================================

baseline_table.to_csv(

    CSV_DIR / "baseline_results.csv",

    index=False

)

synthetic_table.to_csv(

    CSV_DIR / "synthetic_results.csv",

    index=False

)

utility_results.to_csv(

    CSV_DIR / "utility_results.csv",

    index=False

)

generator_summary.to_csv(

    CSV_DIR / "generator_summary.csv",

    index=False

)

classifier_summary.to_csv(

    CSV_DIR / "classifier_summary.csv",

    index=False

)

best_classifier_table.to_csv(

    CSV_DIR / "best_classifier_table.csv",

    index=False

)

publication_summary.to_csv(

    CSV_DIR / "publication_summary.csv",

    index=False

)

roc_summary.to_csv(

    CSV_DIR / "roc_summary.csv",

    index=False

)

pr_summary.to_csv(

    CSV_DIR / "precision_recall_summary.csv",

    index=False

)

overall_roc.to_csv(

    CSV_DIR / "overall_roc_ranking.csv",

    index=False

)

overall_pr.to_csv(

    CSV_DIR / "overall_precision_recall_ranking.csv",

    index=False

)

# =============================================================================
# Excel Workbook
# =============================================================================

excel_file = (

    EXCEL_DIR /

    "Machine_Learning_Utility.xlsx"

)

with pd.ExcelWriter(

    excel_file,

    engine="openpyxl"

) as writer:

    baseline_table.to_excel(

        writer,

        sheet_name="Baseline",

        index=False

    )

    synthetic_table.to_excel(

        writer,

        sheet_name="Synthetic",

        index=False

    )

    utility_results.to_excel(

        writer,

        sheet_name="Utility Results",

        index=False

    )

    generator_summary.to_excel(

        writer,

        sheet_name="Generator Summary",

        index=False

    )

    classifier_summary.to_excel(

        writer,

        sheet_name="Classifier Summary",

        index=False

    )

    best_classifier_table.to_excel(

        writer,

        sheet_name="Best Classifier",

        index=False

    )

    publication_summary.to_excel(

        writer,

        sheet_name="Publication Summary",

        index=False

    )

    roc_summary.to_excel(

        writer,

        sheet_name="ROC Summary",

        index=False

    )

    pr_summary.to_excel(

        writer,

        sheet_name="PR Summary",

        index=False

    )

    overall_roc.to_excel(

        writer,

        sheet_name="Overall ROC",

        index=False

    )

    overall_pr.to_excel(

        writer,

        sheet_name="Overall PR",

        index=False

    )

# =============================================================================
# Saved Files Summary
# =============================================================================

saved_files = [

    "baseline_results.csv",

    "synthetic_results.csv",

    "utility_results.csv",

    "generator_summary.csv",

    "classifier_summary.csv",

    "best_classifier_table.csv",

    "publication_summary.csv",

    "roc_summary.csv",

    "precision_recall_summary.csv",

    "overall_roc_ranking.csv",

    "overall_precision_recall_ranking.csv",

    "Machine_Learning_Utility.xlsx"

]

print("\nSaved Files")
print("-" * 80)

for file in saved_files:

    print(f"✓ {file}")

# =============================================================================
# Verification
# =============================================================================

print("\n" + "=" * 80)

print("Output Directory")

print("=" * 80)

print(UTILITY_DIR)

print("\nCSV Folder")

print(CSV_DIR)

print("\nExcel Folder")

print(EXCEL_DIR)

print("\nTotal CSV Files :", len(saved_files) - 1)

print("Excel Files     : 1")

print("\n" + "=" * 80)
print("SECTION 8.9.11 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.1 : Imports & Configuration
# =============================================================================

print("=" * 80)
print("SECTION 8.9.1 : MACHINE LEARNING UTILITY")
print("Imports & Configuration")
print("=" * 80)

# =============================================================================
# Standard Library Imports
# =============================================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path

# =============================================================================
# Numerical Libraries
# =============================================================================

import numpy as np
import pandas as pd

# =============================================================================
# Visualization Libraries
# =============================================================================

import matplotlib.pyplot as plt

# =============================================================================
# Machine Learning Utilities
# =============================================================================

from sklearn.base import clone

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

# =============================================================================
# Classification Metrics
# =============================================================================

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    balanced_accuracy_score,

    matthews_corrcoef,

    roc_auc_score,

    confusion_matrix,

    classification_report,

    roc_curve,

    precision_recall_curve,

    average_precision_score

)

# =============================================================================
# Classifiers
# =============================================================================

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.svm import SVC

from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

# =============================================================================
# Progress Bar
# =============================================================================

from tqdm.auto import tqdm

# =============================================================================
# Display Options
# =============================================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", None)

warnings.filterwarnings("ignore")

# =============================================================================
# Random Seed
# =============================================================================

RANDOM_STATE = CONFIG["seed"]

random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

# =============================================================================
# Project Directories
# =============================================================================

RESULTS_DIR = PROJECT_ROOT / "results"

EVALUATION_DIR = RESULTS_DIR / "evaluation"

ML_UTILITY_DIR = EVALUATION_DIR / "machine_learning_utility"

FIGURE_DIR = ML_UTILITY_DIR / "figures"

CSV_DIR = ML_UTILITY_DIR / "csv"

EXCEL_DIR = ML_UTILITY_DIR / "excel"

MODEL_DIR = ML_UTILITY_DIR / "trained_models"

ROC_DIR = FIGURE_DIR / "roc_curves"

PR_DIR = FIGURE_DIR / "precision_recall_curves"

# =============================================================================
# Create Directories
# =============================================================================

directories = [

    RESULTS_DIR,

    EVALUATION_DIR,

    ML_UTILITY_DIR,

    FIGURE_DIR,

    CSV_DIR,

    EXCEL_DIR,

    MODEL_DIR,

    ROC_DIR,

    PR_DIR

]

for directory in directories:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

# =============================================================================
# Dataset Names
# =============================================================================

DATASETS = [

    "Adult Income",

    "Bank Marketing",

    "Breast Cancer"

]

# =============================================================================
# Synthetic Models
# =============================================================================

MODELS = [

    "Gaussian Multivariate",

    "Gaussian Copula",

    "CTGAN",

    "TVAE",

    "DP-CTGAN",

    "Proposed SPP-GAN"

]

# =============================================================================
# Global Containers
# =============================================================================

prepared_datasets = {}

baseline_models = {}

synthetic_models = {}

baseline_predictions = {}

synthetic_predictions = {}

baseline_results = pd.DataFrame()

synthetic_results = pd.DataFrame()

utility_results = pd.DataFrame()

generator_summary = pd.DataFrame()

classifier_summary = pd.DataFrame()

publication_summary = pd.DataFrame()

roc_summary = pd.DataFrame()

pr_summary = pd.DataFrame()

# =============================================================================
# Environment Verification
# =============================================================================

print("\nEnvironment Verification")
print("-" * 80)

print(f"Project Root        : {PROJECT_ROOT}")

print(f"Results Directory   : {RESULTS_DIR}")

print(f"Evaluation Folder   : {EVALUATION_DIR}")

print(f"ML Utility Folder   : {ML_UTILITY_DIR}")

print(f"Figures Folder      : {FIGURE_DIR}")

print(f"CSV Folder          : {CSV_DIR}")

print(f"Excel Folder        : {EXCEL_DIR}")

print(f"Model Folder        : {MODEL_DIR}")

print()

print(f"Datasets            : {len(DATASETS)}")

for dataset in DATASETS:

    print(f"   • {dataset}")

print()

print(f"Synthetic Models    : {len(MODELS)}")

for model in MODELS:

    print(f"   • {model}")

print()

print(f"Random Seed         : {RANDOM_STATE}")

print()

print("=" * 80)

print("SECTION 8.9.1 COMPLETED SUCCESSFULLY")

print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.2 : Helper Functions
# =============================================================================

print("=" * 80)
print("SECTION 8.9.2 : HELPER FUNCTIONS")
print("=" * 80)

# =============================================================================
# Reset Random Seed
# =============================================================================

def reset_random_seed(seed=RANDOM_STATE):

    random.seed(seed)

    np.random.seed(seed)

# =============================================================================
# Create Directory
# =============================================================================

def create_directory(path):

    Path(path).mkdir(

        parents=True,

        exist_ok=True

    )

# =============================================================================
# Encode Target Variable
# =============================================================================

def encode_target(

    train_target,

    test_target

):

    encoder = LabelEncoder()

    y_train = encoder.fit_transform(train_target)

    y_test = encoder.transform(test_target)

    return (

        y_train,

        y_test,

        encoder

    )

# =============================================================================
# Standardize Numerical Features
# =============================================================================

def scale_features(

    train_features,

    test_features

):

    scaler = StandardScaler()

    x_train = scaler.fit_transform(

        train_features

    )

    x_test = scaler.transform(

        test_features

    )

    return (

        x_train,

        x_test,

        scaler

    )

# =============================================================================
# Prepare Dataset
# =============================================================================

def prepare_dataset(

    train_df,

    test_df,

    target_column

):

    x_train = train_df.drop(

        columns=[target_column]

    )

    y_train = train_df[target_column]

    x_test = test_df.drop(

        columns=[target_column]

    )

    y_test = test_df[target_column]

    (

        y_train,

        y_test,

        encoder

    ) = encode_target(

        y_train,

        y_test

    )

    (

        x_train,

        x_test,

        scaler

    ) = scale_features(

        x_train,

        x_test

    )

    return {

        "x_train":x_train,

        "y_train":y_train,

        "x_test":x_test,

        "y_test":y_test,

        "label_encoder":encoder,

        "scaler":scaler

    }

# =============================================================================
# Get Prediction Probabilities
# =============================================================================

def predict_probabilities(

    model,

    x

):

    if hasattr(

        model,

        "predict_proba"

    ):

        return model.predict_proba(

            x

        )[:,1]

    if hasattr(

        model,

        "decision_function"

    ):

        scores = model.decision_function(

            x

        )

        scores = (

            scores -

            scores.min()

        ) / (

            scores.max()

            -

            scores.min()

            +

            1e-12

        )

        return scores

    return None

# =============================================================================
# Calculate Classification Metrics
# =============================================================================

def calculate_metrics(

    y_true,

    y_pred,

    y_prob=None

):

    metrics = {}

    metrics["Accuracy"] = accuracy_score(

        y_true,

        y_pred

    )

    metrics["Precision"] = precision_score(

        y_true,

        y_pred,

        average="weighted",

        zero_division=0

    )

    metrics["Recall"] = recall_score(

        y_true,

        y_pred,

        average="weighted",

        zero_division=0

    )

    metrics["F1 Score"] = f1_score(

        y_true,

        y_pred,

        average="weighted",

        zero_division=0

    )

    metrics["Balanced Accuracy"] = (

        balanced_accuracy_score(

            y_true,

            y_pred

        )

    )

    metrics["MCC"] = matthews_corrcoef(

        y_true,

        y_pred

    )

    if y_prob is not None:

        try:

            metrics["ROC-AUC"] = roc_auc_score(

                y_true,

                y_prob

            )

        except:

            metrics["ROC-AUC"] = np.nan

        try:

            metrics["Average Precision"] = (

                average_precision_score(

                    y_true,

                    y_prob

                )

            )

        except:

            metrics["Average Precision"] = np.nan

    else:

        metrics["ROC-AUC"] = np.nan

        metrics["Average Precision"] = np.nan

    return metrics

# =============================================================================
# Utility Score
# =============================================================================

def compute_utility_score(

    metrics

):

    values = [

        metrics["Accuracy"],

        metrics["Precision"],

        metrics["Recall"],

        metrics["F1 Score"],

        metrics["Balanced Accuracy"],

        metrics["MCC"]

    ]

    values = [

        v

        for v in values

        if not pd.isna(v)

    ]

    return np.mean(values)

# =============================================================================
# Convert Metrics Dictionary to DataFrame Row
# =============================================================================

def metrics_to_dataframe(

    dataset,

    model_name,

    classifier,

    metrics

):

    row = {

        "Dataset":dataset,

        "Synthetic Model":model_name,

        "Classifier":classifier

    }

    row.update(metrics)

    row["Utility Score"] = (

        compute_utility_score(

            metrics

        )

    )

    return row

# =============================================================================
# Print Section Header
# =============================================================================

def print_header(title):

    print("\n")

    print("="*80)

    print(title)

    print("="*80)

# =============================================================================
# Verification
# =============================================================================

helper_functions = [

    "reset_random_seed",

    "create_directory",

    "encode_target",

    "scale_features",

    "prepare_dataset",

    "predict_probabilities",

    "calculate_metrics",

    "compute_utility_score",

    "metrics_to_dataframe",

    "print_header"

]

print("\nRegistered Helper Functions")

print("-"*80)

for func in helper_functions:

    print(f"✓ {func}")

print("\nTotal Helper Functions :", len(helper_functions))

print("\n" + "="*80)
print("SECTION 8.9.2 COMPLETED SUCCESSFULLY")
print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.3 : Prepare Training & Test Data
# =============================================================================

print("=" * 80)
print("SECTION 8.9.3 : PREPARE TRAINING & TEST DATA")
print("=" * 80)

# =============================================================================
# Target Column Mapping
# =============================================================================

TARGET_COLUMNS = {

    "Adult Income": "income",

    "Bank Marketing": "y",

    "Breast Cancer": "diagnosis"

}

# =============================================================================
# Containers
# =============================================================================

prepared_datasets = {}

# =============================================================================
# Prepare Datasets
# =============================================================================

for dataset in DATASETS:

    print("\n" + "-" * 80)
    print(f"Dataset : {dataset}")
    print("-" * 80)

    target_column = TARGET_COLUMNS[dataset]

    # -------------------------------------------------------------------------
    # Load Real Data
    # -------------------------------------------------------------------------

    real_train = original_datasets[dataset]["train"].copy()

    real_test = original_datasets[dataset]["test"].copy()

    # -------------------------------------------------------------------------
    # Baseline (Real Train -> Real Test)
    # -------------------------------------------------------------------------

    baseline = prepare_dataset(

        train_df=real_train,

        test_df=real_test,

        target_column=target_column

    )

    # -------------------------------------------------------------------------
    # Store Baseline
    # -------------------------------------------------------------------------

    prepared_datasets[dataset] = {

        "baseline": baseline,

        "synthetic": {}

    }

    # -------------------------------------------------------------------------
    # Prepare Synthetic Training Sets
    # -------------------------------------------------------------------------

    for model_name in MODELS:

        synthetic_train = synthetic_datasets[dataset][model_name].copy()

        # -------------------------------------------------------------
        # Ensure Same Feature Order
        # -------------------------------------------------------------

        synthetic_train = synthetic_train[real_train.columns]

        synthetic = prepare_dataset(

            train_df=synthetic_train,

            test_df=real_test,

            target_column=target_column

        )

        prepared_datasets[dataset]["synthetic"][model_name] = synthetic

    # -------------------------------------------------------------------------
    # Verification
    # -------------------------------------------------------------------------

    print(f"Target Column      : {target_column}")

    print(f"Real Train Shape   : {real_train.shape}")

    print(f"Real Test Shape    : {real_test.shape}")

    print(f"Synthetic Models   : {len(MODELS)}")

# =============================================================================
# Final Verification
# =============================================================================

print("\n" + "=" * 80)
print("Prepared Dataset Summary")
print("=" * 80)

for dataset in DATASETS:

    baseline = prepared_datasets[dataset]["baseline"]

    print(f"\n{dataset}")

    print(f"Baseline Train : {baseline['x_train'].shape}")

    print(f"Baseline Test  : {baseline['x_test'].shape}")

    print("Synthetic Training Sets")

    for model_name in MODELS:

        x_train = prepared_datasets[dataset]["synthetic"][model_name]["x_train"]

        print(f"  • {model_name:<25} {x_train.shape}")

print("\n" + "=" * 80)
print("SECTION 8.9.3 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.4 : Define Downstream Classifiers
# =============================================================================

print("=" * 80)
print("SECTION 8.9.4 : DEFINE DOWNSTREAM CLASSIFIERS")
print("=" * 80)

# =============================================================================
# Logistic Regression
# =============================================================================

logistic_regression = LogisticRegression(

    random_state=RANDOM_STATE,

    max_iter=1000,

    solver="lbfgs",

    n_jobs=-1

)

# =============================================================================
# Random Forest
# =============================================================================

random_forest = RandomForestClassifier(

    n_estimators=300,

    max_depth=None,

    min_samples_split=2,

    min_samples_leaf=1,

    bootstrap=True,

    random_state=RANDOM_STATE,

    n_jobs=-1

)

# =============================================================================
# XGBoost
# =============================================================================

xgboost = XGBClassifier(

    n_estimators=300,

    learning_rate=0.05,

    max_depth=6,

    subsample=0.80,

    colsample_bytree=0.80,

    objective="binary:logistic",

    eval_metric="logloss",

    random_state=RANDOM_STATE,

    n_jobs=-1,

    tree_method="hist",

    verbosity=0

)

# =============================================================================
# Classifier Dictionary
# =============================================================================

CLASSIFIERS = {

    "Logistic Regression": logistic_regression,

    "Random Forest": random_forest,

    "XGBoost": xgboost

}

# =============================================================================
# Display Configuration
# =============================================================================

print("\nDefined Downstream Classifiers")
print("-" * 80)

for name, model in CLASSIFIERS.items():

    print(f"✓ {name}")

print("\nTotal Classifiers :", len(CLASSIFIERS))

print("\nClassifier Details")
print("-" * 80)

for name, model in CLASSIFIERS.items():

    print(f"\n{name}")

    print(model)

# =============================================================================
# Verification
# =============================================================================

assert len(CLASSIFIERS) == 3, "Expected exactly 3 classifiers."

print("\n" + "=" * 80)
print("SECTION 8.9.4 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.5 : Train on Original Data (Reference Baseline)
# =============================================================================

print("=" * 80)
print("SECTION 8.9.5 : TRAIN ON ORIGINAL DATA (REFERENCE BASELINE)")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

baseline_models = {}

baseline_predictions = {}

baseline_results_list = []

# =============================================================================
# Train Classifiers on Original Data
# =============================================================================

for dataset_name in DATASETS:

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    data = prepared_datasets[dataset_name]["baseline"]

    x_train = data["x_train"]
    y_train = data["y_train"]

    x_test = data["x_test"]
    y_test = data["y_test"]

    baseline_models[dataset_name] = {}
    baseline_predictions[dataset_name] = {}

    for classifier_name, classifier in CLASSIFIERS.items():

        print(f"Training : {classifier_name}")

        # -------------------------------------------------------------
        # Clone classifier
        # -------------------------------------------------------------

        model = clone(classifier)

        # -------------------------------------------------------------
        # Train
        # -------------------------------------------------------------

        model.fit(

            x_train,

            y_train

        )

        # -------------------------------------------------------------
        # Predict
        # -------------------------------------------------------------

        y_pred = model.predict(

            x_test

        )

        y_prob = predict_probabilities(

            model,

            x_test

        )

        # -------------------------------------------------------------
        # Metrics
        # -------------------------------------------------------------

        metrics = calculate_metrics(

            y_true=y_test,

            y_pred=y_pred,

            y_prob=y_prob

        )

        # -------------------------------------------------------------
        # Store Results
        # -------------------------------------------------------------

        row = metrics_to_dataframe(

            dataset=dataset_name,

            model_name="Original Data",

            classifier=classifier_name,

            metrics=metrics

        )

        baseline_results_list.append(row)

        baseline_models[dataset_name][classifier_name] = model

        baseline_predictions[dataset_name][classifier_name] = {

            "y_true": y_test,

            "y_pred": y_pred,

            "y_prob": y_prob

        }

        print(
            f"Accuracy={metrics['Accuracy']:.4f} | "
            f"F1={metrics['F1 Score']:.4f} | "
            f"ROC-AUC={metrics['ROC-AUC']:.4f}"
        )

# =============================================================================
# Create Baseline Results DataFrame
# =============================================================================

baseline_results = pd.DataFrame(

    baseline_results_list

)

# =============================================================================
# Sort Results
# =============================================================================

baseline_results = baseline_results.sort_values(

    [

        "Dataset",

        "Classifier"

    ]

).reset_index(

    drop=True

)

# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 80)
print("REFERENCE BASELINE RESULTS")
print("=" * 80)

display(

    baseline_results.round(4)

)

# =============================================================================
# Average Performance
# =============================================================================

baseline_summary = (

    baseline_results

    .groupby("Classifier")

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean",

        "Utility Score":"mean"

    })

    .round(4)

)

print("\n" + "=" * 80)
print("AVERAGE BASELINE PERFORMANCE")
print("=" * 80)

display(

    baseline_summary

)

# =============================================================================
# Verification
# =============================================================================

expected_results = len(DATASETS) * len(CLASSIFIERS)

print("\nVerification")
print("-" * 80)

print(f"Datasets              : {len(DATASETS)}")

print(f"Classifiers           : {len(CLASSIFIERS)}")

print(f"Expected Experiments  : {expected_results}")

print(f"Completed Experiments : {len(baseline_results)}")

assert len(baseline_results) == expected_results

print("\n✓ All baseline models trained successfully.")

print("\n" + "=" * 80)
print("SECTION 8.9.5 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.6 : Train on Synthetic → Test on Real (TSTR)
# =============================================================================

print("=" * 80)
print("SECTION 8.9.6 : TRAIN ON SYNTHETIC → TEST ON REAL")
print("TSTR PROTOCOL")
print("=" * 80)

# =============================================================================
# Containers
# =============================================================================

synthetic_models = {}

synthetic_predictions = {}

synthetic_results_list = []

# =============================================================================
# TSTR Evaluation
# =============================================================================

for dataset_name in DATASETS:

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset_name}")
    print("=" * 80)

    synthetic_models[dataset_name] = {}

    synthetic_predictions[dataset_name] = {}

    # -------------------------------------------------------------------------
    # Evaluate Every Synthetic Generator
    # -------------------------------------------------------------------------

    for generator_name in MODELS:

        print(f"\nSynthetic Generator : {generator_name}")

        synthetic_models[dataset_name][generator_name] = {}

        synthetic_predictions[dataset_name][generator_name] = {}

        data = prepared_datasets[dataset_name]["synthetic"][generator_name]

        x_train = data["x_train"]

        y_train = data["y_train"]

        x_test = data["x_test"]

        y_test = data["y_test"]

        # ---------------------------------------------------------------------
        # Evaluate Every Classifier
        # ---------------------------------------------------------------------

        for classifier_name, classifier in CLASSIFIERS.items():

            print(f"   Training : {classifier_name}")

            model = clone(classifier)

            model.fit(

                x_train,

                y_train

            )

            y_pred = model.predict(

                x_test

            )

            y_prob = predict_probabilities(

                model,

                x_test

            )

            metrics = calculate_metrics(

                y_true=y_test,

                y_pred=y_pred,

                y_prob=y_prob

            )

            row = metrics_to_dataframe(

                dataset=dataset_name,

                model_name=generator_name,

                classifier=classifier_name,

                metrics=metrics

            )

            synthetic_results_list.append(row)

            synthetic_models[dataset_name][generator_name][classifier_name] = model

            synthetic_predictions[dataset_name][generator_name][classifier_name] = {

                "y_true": y_test,

                "y_pred": y_pred,

                "y_prob": y_prob

            }

            print(

                f"      Accuracy={metrics['Accuracy']:.4f} | "

                f"F1={metrics['F1 Score']:.4f} | "

                f"ROC-AUC={metrics['ROC-AUC']:.4f}"

            )

# =============================================================================
# Create Results DataFrame
# =============================================================================

synthetic_results = pd.DataFrame(

    synthetic_results_list

)

# =============================================================================
# Sort Results
# =============================================================================

synthetic_results = (

    synthetic_results

    .sort_values(

        [

            "Dataset",

            "Synthetic Model",

            "Classifier"

        ]

    )

    .reset_index(

        drop=True

    )

)

# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 80)
print("TSTR RESULTS")
print("=" * 80)

display(

    synthetic_results.round(4)

)

# =============================================================================
# Generator Summary
# =============================================================================

generator_summary = (

    synthetic_results

    .groupby(

        "Synthetic Model"

    )

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean",

        "Utility Score":"mean"

    })

    .sort_values(

        "Utility Score",

        ascending=False

    )

    .reset_index()

)

print("\n" + "=" * 80)
print("GENERATOR PERFORMANCE")
print("=" * 80)

display(

    generator_summary.round(4)

)

# =============================================================================
# Classifier Summary
# =============================================================================

classifier_summary = (

    synthetic_results

    .groupby(

        "Classifier"

    )

    .agg({

        "Accuracy":"mean",

        "Precision":"mean",

        "Recall":"mean",

        "F1 Score":"mean",

        "ROC-AUC":"mean",

        "Balanced Accuracy":"mean",

        "MCC":"mean",

        "Utility Score":"mean"

    })

    .sort_values(

        "Utility Score",

        ascending=False

    )

    .reset_index()

)

print("\n" + "=" * 80)
print("CLASSIFIER PERFORMANCE")
print("=" * 80)

display(

    classifier_summary.round(4)

)

# =============================================================================
# Verification
# =============================================================================

expected = (

    len(DATASETS)

    *

    len(MODELS)

    *

    len(CLASSIFIERS)

)

print("\nVerification")
print("-" * 80)

print(f"Datasets              : {len(DATASETS)}")

print(f"Synthetic Models      : {len(MODELS)}")

print(f"Classifiers           : {len(CLASSIFIERS)}")

print(f"Expected Experiments  : {expected}")

print(f"Completed Experiments : {len(synthetic_results)}")

assert len(synthetic_results) == expected

print("\n✓ All TSTR experiments completed successfully.")

print("\n" + "=" * 80)
print("SECTION 8.9.6 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.7 : Calculate Utility Metrics
# =============================================================================

print("=" * 80)
print("SECTION 8.9.7 : CALCULATE UTILITY METRICS")
print("=" * 80)

# =============================================================================
# Combine Baseline + Synthetic Results
# =============================================================================

utility_results = pd.concat(

    [

        baseline_results,

        synthetic_results

    ],

    ignore_index=True

)

# =============================================================================
# Utility Metrics
# =============================================================================

UTILITY_METRICS = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC-AUC",

    "Balanced Accuracy",

    "MCC",

    "Utility Score"

]

# =============================================================================
# Overall Statistics
# =============================================================================

overall_statistics = (

    utility_results

    [

        UTILITY_METRICS

    ]

    .describe()

    .T

)

print("\nOverall Utility Statistics")

print("-"*80)

display(

    overall_statistics.round(4)

)

# =============================================================================
# Dataset-wise Summary
# =============================================================================

dataset_summary = (

    utility_results

    .groupby(

        "Dataset"

    )

    [

        UTILITY_METRICS

    ]

    .mean()

    .reset_index()

)

print("\nDataset-wise Average Utility")

print("-"*80)

display(

    dataset_summary.round(4)

)

# =============================================================================
# Generator-wise Summary
# =============================================================================

generator_summary = (

    utility_results

    .groupby(

        "Synthetic Model"

    )

    [

        UTILITY_METRICS

    ]

    .mean()

    .sort_values(

        "Utility Score",

        ascending=False

    )

    .reset_index()

)

print("\nGenerator Performance")

print("-"*80)

display(

    generator_summary.round(4)

)

# =============================================================================
# Classifier Summary
# =============================================================================

classifier_summary = (

    utility_results

    .groupby(

        "Classifier"

    )

    [

        UTILITY_METRICS

    ]

    .mean()

    .sort_values(

        "Utility Score",

        ascending=False

    )

    .reset_index()

)

print("\nClassifier Performance")

print("-"*80)

display(

    classifier_summary.round(4)

)

# =============================================================================
# Best Generator Per Dataset
# =============================================================================

best_generator_dataset = (

    utility_results

    .groupby(

        [

            "Dataset",

            "Synthetic Model"

        ]

    )["Utility Score"]

    .mean()

    .reset_index()

)

best_generator_dataset = (

    best_generator_dataset

    .sort_values(

        [

            "Dataset",

            "Utility Score"

        ],

        ascending=[

            True,

            False

        ]

    )

    .groupby(

        "Dataset"

    )

    .first()

    .reset_index()

)

print("\nBest Generator Per Dataset")

print("-"*80)

display(

    best_generator_dataset.round(4)

)

# =============================================================================
# Best Classifier Per Dataset
# =============================================================================

best_classifier_dataset = (

    utility_results

    .groupby(

        [

            "Dataset",

            "Classifier"

        ]

    )["Utility Score"]

    .mean()

    .reset_index()

)

best_classifier_dataset = (

    best_classifier_dataset

    .sort_values(

        [

            "Dataset",

            "Utility Score"

        ],

        ascending=[

            True,

            False

        ]

    )

    .groupby(

        "Dataset"

    )

    .first()

    .reset_index()

)

print("\nBest Classifier Per Dataset")

print("-"*80)

display(

    best_classifier_dataset.round(4)

)

# =============================================================================
# Utility Ranking
# =============================================================================

utility_ranking = (

    generator_summary

    [

        [

            "Synthetic Model",

            "Utility Score"

        ]

    ]

    .copy()

)

utility_ranking.insert(

    0,

    "Rank",

    np.arange(

        1,

        len(

            utility_ranking

        ) + 1

    )

)

print("\nOverall Generator Ranking")

print("-"*80)

display(

    utility_ranking

)

# =============================================================================
# Best Overall Model
# =============================================================================

best_model = utility_ranking.iloc[0]

print("\nBest Synthetic Generator")

print("-"*80)

print(

    f"Model         : {best_model['Synthetic Model']}"

)

print(

    f"Utility Score : {best_model['Utility Score']:.4f}"

)

# =============================================================================
# Verification
# =============================================================================

expected_rows = (

    len(DATASETS)

    *

    (

        len(MODELS)

        +

        1

    )

    *

    len(CLASSIFIERS)

)

print("\nVerification")

print("-"*80)

print(

    f"Total Utility Records : {len(utility_results)}"

)

print(

    f"Expected Records      : {expected_rows}"

)

assert len(utility_results) == expected_rows

print("\n✓ Utility metrics calculated successfully.")

print("\n"+"="*80)

print("SECTION 8.9.7 COMPLETED SUCCESSFULLY")

print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.8 : ROC Curve Comparison
# =============================================================================

print("=" * 80)
print("SECTION 8.9.8 : ROC CURVE COMPARISON")
print("=" * 80)

roc_summary = []

# =============================================================================
# ROC Curves
# =============================================================================

for dataset in DATASETS:

    print(f"\nGenerating ROC Curves : {dataset}")

    for classifier in CLASSIFIERS.keys():

        plt.figure(figsize=(8, 6))

        # ---------------------------------------------------------------------
        # Original Baseline
        # ---------------------------------------------------------------------

        baseline = baseline_predictions[dataset][classifier]

        if baseline["y_prob"] is not None:

            fpr, tpr, _ = roc_curve(

                baseline["y_true"],
                baseline["y_prob"]
            )

            auc = roc_auc_score(

                baseline["y_true"],
                baseline["y_prob"]
            )

            plt.plot(

                fpr,
                tpr,
                linewidth=3,
                label=f"Original (AUC={auc:.3f})"

            )

            roc_summary.append({

                "Dataset": dataset,
                "Classifier": classifier,
                "Synthetic Model": "Original Data",
                "ROC-AUC": auc

            })

        # ---------------------------------------------------------------------
        # Synthetic Models
        # ---------------------------------------------------------------------

        for generator in MODELS:

            prediction = synthetic_predictions[dataset][generator][classifier]

            if prediction["y_prob"] is None:
                continue

            fpr, tpr, _ = roc_curve(

                prediction["y_true"],
                prediction["y_prob"]

            )

            auc = roc_auc_score(

                prediction["y_true"],
                prediction["y_prob"]

            )

            plt.plot(

                fpr,
                tpr,
                linewidth=2,
                label=f"{generator} ({auc:.3f})"

            )

            roc_summary.append({

                "Dataset": dataset,
                "Classifier": classifier,
                "Synthetic Model": generator,
                "ROC-AUC": auc

            })

        # ---------------------------------------------------------------------
        # Random Classifier
        # ---------------------------------------------------------------------

        plt.plot(

            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=2,
            color="black"

        )

        plt.xlabel("False Positive Rate", fontsize=12)

        plt.ylabel("True Positive Rate", fontsize=12)

        plt.title(

            f"{dataset}\n{classifier}",

            fontsize=13,

            weight="bold"

        )

        plt.grid(alpha=0.3)

        plt.legend(fontsize=8)

        plt.tight_layout()

        save_path = ROC_DIR / f"{dataset}_{classifier}_ROC.png"

        plt.savefig(

            save_path,

            dpi=600,

            bbox_inches="tight"

        )

        plt.show()

        plt.close()

# =============================================================================
# ROC Summary
# =============================================================================

roc_summary = pd.DataFrame(roc_summary)

roc_summary = (

    roc_summary

    .sort_values(

        [

            "Dataset",

            "Classifier",

            "ROC-AUC"

        ],

        ascending=[

            True,

            True,

            False

        ]

    )

    .reset_index(drop=True)

)

print("\nROC Summary")

print("-" * 80)

display(

    roc_summary.round(4)

)

# =============================================================================
# Average ROC-AUC
# =============================================================================

roc_average = (

    roc_summary

    .groupby(

        "Synthetic Model"

    )["ROC-AUC"]

    .mean()

    .sort_values(

        ascending=False

    )

    .reset_index()

)

print("\nAverage ROC-AUC")

print("-" * 80)

display(

    roc_average.round(4)

)

# =============================================================================
# Verification
# =============================================================================

expected = len(DATASETS) * len(CLASSIFIERS) * (len(MODELS) + 1)

print("\nVerification")

print("-" * 80)

print(f"Expected ROC Results : {expected}")

print(f"Generated Results    : {len(roc_summary)}")

assert len(roc_summary) == expected

print("\n✓ ROC curve comparison completed successfully.")

print("\n" + "=" * 80)
print("SECTION 8.9.8 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.9 : Precision–Recall Curves
# =============================================================================

print("=" * 80)
print("SECTION 8.9.9 : PRECISION–RECALL CURVES")
print("=" * 80)

pr_summary = []

# =============================================================================
# Precision-Recall Curves
# =============================================================================

for dataset in DATASETS:

    print(f"\nGenerating Precision–Recall Curves : {dataset}")

    for classifier in CLASSIFIERS.keys():

        plt.figure(figsize=(8, 6))

        # ---------------------------------------------------------------------
        # Original Baseline
        # ---------------------------------------------------------------------

        baseline = baseline_predictions[dataset][classifier]

        if baseline["y_prob"] is not None:

            precision, recall, _ = precision_recall_curve(

                baseline["y_true"],
                baseline["y_prob"]

            )

            ap = average_precision_score(

                baseline["y_true"],
                baseline["y_prob"]

            )

            plt.plot(

                recall,
                precision,
                linewidth=3,
                label=f"Original (AP={ap:.3f})"

            )

            pr_summary.append({

                "Dataset": dataset,
                "Classifier": classifier,
                "Synthetic Model": "Original Data",
                "Average Precision": ap

            })

        # ---------------------------------------------------------------------
        # Synthetic Models
        # ---------------------------------------------------------------------

        for generator in MODELS:

            prediction = synthetic_predictions[dataset][generator][classifier]

            if prediction["y_prob"] is None:
                continue

            precision, recall, _ = precision_recall_curve(

                prediction["y_true"],
                prediction["y_prob"]

            )

            ap = average_precision_score(

                prediction["y_true"],
                prediction["y_prob"]

            )

            plt.plot(

                recall,
                precision,
                linewidth=2,
                label=f"{generator} (AP={ap:.3f})"

            )

            pr_summary.append({

                "Dataset": dataset,
                "Classifier": classifier,
                "Synthetic Model": generator,
                "Average Precision": ap

            })

        # ---------------------------------------------------------------------
        # Formatting
        # ---------------------------------------------------------------------

        plt.xlabel("Recall", fontsize=12)

        plt.ylabel("Precision", fontsize=12)

        plt.title(

            f"{dataset}\n{classifier}",

            fontsize=13,

            weight="bold"

        )

        plt.grid(alpha=0.3)

        plt.legend(fontsize=8)

        plt.tight_layout()

        save_path = PR_DIR / f"{dataset}_{classifier}_PR.png"

        plt.savefig(

            save_path,

            dpi=600,

            bbox_inches="tight"

        )

        plt.show()

        plt.close()

# =============================================================================
# Create Summary DataFrame
# =============================================================================

pr_summary = pd.DataFrame(pr_summary)

pr_summary = (

    pr_summary

    .sort_values(

        [

            "Dataset",

            "Classifier",

            "Average Precision"

        ],

        ascending=[

            True,

            True,

            False

        ]

    )

    .reset_index(drop=True)

)

# =============================================================================
# Display Summary
# =============================================================================

print("\nPrecision–Recall Summary")

print("-" * 80)

display(

    pr_summary.round(4)

)

# =============================================================================
# Average Precision Ranking
# =============================================================================

pr_average = (

    pr_summary

    .groupby(

        "Synthetic Model"

    )["Average Precision"]

    .mean()

    .sort_values(

        ascending=False

    )

    .reset_index()

)

print("\nAverage Precision Ranking")

print("-" * 80)

display(

    pr_average.round(4)

)

# =============================================================================
# Verification
# =============================================================================

expected = len(DATASETS) * len(CLASSIFIERS) * (len(MODELS) + 1)

print("\nVerification")

print("-" * 80)

print(f"Expected Results : {expected}")

print(f"Generated Results: {len(pr_summary)}")

assert len(pr_summary) == expected

print("\n✓ Precision–Recall curve comparison completed successfully.")

print("\n" + "=" * 80)
print("SECTION 8.9.9 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.10 : Utility Summary Tables
# =============================================================================

print("=" * 80)
print("SECTION 8.9.10 : UTILITY SUMMARY TABLES")
print("=" * 80)

# =============================================================================
# Metrics
# =============================================================================

UTILITY_METRICS = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC-AUC",

    "Balanced Accuracy",

    "MCC",

    "Utility Score"

]

# =============================================================================
# Table 1 : Overall Generator Performance
# =============================================================================

generator_summary = (

    utility_results

    .groupby(

        "Synthetic Model"

    )[UTILITY_METRICS]

    .mean()

    .sort_values(

        "Utility Score",

        ascending=False

    )

    .reset_index()

)

generator_summary.insert(

    0,

    "Rank",

    np.arange(

        1,

        len(generator_summary)+1

    )

)

# =============================================================================
# Table 2 : Overall Classifier Performance
# =============================================================================

classifier_summary = (

    utility_results

    .groupby(

        "Classifier"

    )[UTILITY_METRICS]

    .mean()

    .sort_values(

        "Utility Score",

        ascending=False

    )

    .reset_index()

)

classifier_summary.insert(

    0,

    "Rank",

    np.arange(

        1,

        len(classifier_summary)+1

    )

)

# =============================================================================
# Table 3 : Dataset × Generator
# =============================================================================

dataset_generator_summary = (

    utility_results

    .groupby(

        [

            "Dataset",

            "Synthetic Model"

        ]

    )[UTILITY_METRICS]

    .mean()

    .reset_index()

)

# =============================================================================
# Table 4 : Dataset × Classifier
# =============================================================================

dataset_classifier_summary = (

    utility_results

    .groupby(

        [

            "Dataset",

            "Classifier"

        ]

    )[UTILITY_METRICS]

    .mean()

    .reset_index()

)

# =============================================================================
# Table 5 : ROC Summary
# =============================================================================

roc_table = (

    roc_summary

    .groupby(

        "Synthetic Model"

    )["ROC-AUC"]

    .mean()

    .sort_values(

        ascending=False

    )

    .reset_index()

)

# =============================================================================
# Table 6 : Precision–Recall Summary
# =============================================================================

pr_table = (

    pr_summary

    .groupby(

        "Synthetic Model"

    )["Average Precision"]

    .mean()

    .sort_values(

        ascending=False

    )

    .reset_index()

)

# =============================================================================
# Table 7 : Publication Summary
# =============================================================================

publication_summary = (

    generator_summary

    .merge(

        roc_table,

        on="Synthetic Model"

    )

    .merge(

        pr_table,

        on="Synthetic Model"

    )

)

publication_summary = publication_summary.sort_values(

    "Utility Score",

    ascending=False

)

publication_summary.reset_index(

    drop=True,

    inplace=True

)

# =============================================================================
# Display Tables
# =============================================================================

print("\n")

print("="*80)

print("TABLE 1 : GENERATOR PERFORMANCE")

print("="*80)

display(

    generator_summary.round(4)

)

print("\n")

print("="*80)

print("TABLE 2 : CLASSIFIER PERFORMANCE")

print("="*80)

display(

    classifier_summary.round(4)

)

print("\n")

print("="*80)

print("TABLE 3 : DATASET × GENERATOR")

print("="*80)

display(

    dataset_generator_summary.round(4)

)

print("\n")

print("="*80)

print("TABLE 4 : DATASET × CLASSIFIER")

print("="*80)

display(

    dataset_classifier_summary.round(4)

)

print("\n")

print("="*80)

print("TABLE 5 : ROC SUMMARY")

print("="*80)

display(

    roc_table.round(4)

)

print("\n")

print("="*80)

print("TABLE 6 : PRECISION–RECALL SUMMARY")

print("="*80)

display(

    pr_table.round(4)

)

print("\n")

print("="*80)

print("TABLE 7 : PUBLICATION SUMMARY")

print("="*80)

display(

    publication_summary.round(4)

)

# =============================================================================
# Best Model
# =============================================================================

best_model = publication_summary.iloc[0]

print("\n")

print("="*80)

print("BEST SYNTHETIC DATA GENERATOR")

print("="*80)

print(

    f"Model          : {best_model['Synthetic Model']}"

)

print(

    f"Utility Score  : {best_model['Utility Score']:.4f}"

)

print(

    f"ROC-AUC        : {best_model['ROC-AUC']:.4f}"

)

print(

    f"Average Precision : {best_model['Average Precision']:.4f}"

)

# =============================================================================
# Verification
# =============================================================================

print("\n")

print("="*80)

print("VERIFICATION")

print("="*80)

print(

    f"Generator Summary Rows : {len(generator_summary)}"

)

print(

    f"Classifier Summary Rows : {len(classifier_summary)}"

)

print(

    f"Publication Summary Rows : {len(publication_summary)}"

)

print("\n✓ Utility summary tables generated successfully.")

print("\n")

print("="*80)

print("SECTION 8.9.10 COMPLETED SUCCESSFULLY")

print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.11 : Save Results
# =============================================================================

print("=" * 80)
print("SECTION 8.9.11 : SAVE RESULTS")
print("=" * 80)

# =============================================================================
# Create Output Directories
# =============================================================================

CSV_DIR.mkdir(

    parents=True,

    exist_ok=True

)

EXCEL_DIR.mkdir(

    parents=True,

    exist_ok=True

)

# =============================================================================
# CSV Files
# =============================================================================

print("\nSaving CSV Files...")

baseline_results.to_csv(

    CSV_DIR / "baseline_results.csv",

    index=False

)

synthetic_results.to_csv(

    CSV_DIR / "synthetic_results.csv",

    index=False

)

utility_results.to_csv(

    CSV_DIR / "utility_results.csv",

    index=False

)

generator_summary.to_csv(

    CSV_DIR / "generator_summary.csv",

    index=False

)

classifier_summary.to_csv(

    CSV_DIR / "classifier_summary.csv",

    index=False

)

dataset_generator_summary.to_csv(

    CSV_DIR / "dataset_generator_summary.csv",

    index=False

)

dataset_classifier_summary.to_csv(

    CSV_DIR / "dataset_classifier_summary.csv",

    index=False

)

publication_summary.to_csv(

    CSV_DIR / "publication_summary.csv",

    index=False

)

roc_summary.to_csv(

    CSV_DIR / "roc_summary.csv",

    index=False

)

roc_table.to_csv(

    CSV_DIR / "roc_average.csv",

    index=False

)

pr_summary.to_csv(

    CSV_DIR / "precision_recall_summary.csv",

    index=False

)

pr_table.to_csv(

    CSV_DIR / "precision_recall_average.csv",

    index=False

)

print("✓ CSV files saved.")

# =============================================================================
# Excel Workbook
# =============================================================================

excel_file = EXCEL_DIR / "Machine_Learning_Utility.xlsx"

print("\nCreating Excel Workbook...")

with pd.ExcelWriter(

    excel_file,

    engine="openpyxl"

) as writer:

    baseline_results.to_excel(

        writer,

        sheet_name="Baseline",

        index=False

    )

    synthetic_results.to_excel(

        writer,

        sheet_name="Synthetic",

        index=False

    )

    utility_results.to_excel(

        writer,

        sheet_name="Utility",

        index=False

    )

    generator_summary.to_excel(

        writer,

        sheet_name="Generator Summary",

        index=False

    )

    classifier_summary.to_excel(

        writer,

        sheet_name="Classifier Summary",

        index=False

    )

    dataset_generator_summary.to_excel(

        writer,

        sheet_name="Dataset Generator",

        index=False

    )

    dataset_classifier_summary.to_excel(

        writer,

        sheet_name="Dataset Classifier",

        index=False

    )

    roc_summary.to_excel(

        writer,

        sheet_name="ROC Summary",

        index=False

    )

    roc_table.to_excel(

        writer,

        sheet_name="ROC Average",

        index=False

    )

    pr_summary.to_excel(

        writer,

        sheet_name="PR Summary",

        index=False

    )

    pr_table.to_excel(

        writer,

        sheet_name="PR Average",

        index=False

    )

    publication_summary.to_excel(

        writer,

        sheet_name="Publication Summary",

        index=False

    )

print("✓ Excel workbook saved.")

# =============================================================================
# Saved Files Summary
# =============================================================================

saved_files = [

    "baseline_results.csv",

    "synthetic_results.csv",

    "utility_results.csv",

    "generator_summary.csv",

    "classifier_summary.csv",

    "dataset_generator_summary.csv",

    "dataset_classifier_summary.csv",

    "publication_summary.csv",

    "roc_summary.csv",

    "roc_average.csv",

    "precision_recall_summary.csv",

    "precision_recall_average.csv",

    "Machine_Learning_Utility.xlsx"

]

print("\n")

print("=" * 80)

print("FILES SAVED")

print("=" * 80)

for file in saved_files:

    print(f"✓ {file}")

# =============================================================================
# Output Locations
# =============================================================================

print("\n")

print("=" * 80)

print("OUTPUT DIRECTORIES")

print("=" * 80)

print(f"CSV Files      : {CSV_DIR}")

print(f"Excel Workbook : {excel_file}")

print(f"ROC Figures    : {ROC_DIR}")

print(f"PR Figures     : {PR_DIR}")

# =============================================================================
# Verification
# =============================================================================

print("\n")

print("=" * 80)

print("VERIFICATION")

print("=" * 80)

csv_count = len(list(CSV_DIR.glob("*.csv")))

excel_count = len(list(EXCEL_DIR.glob("*.xlsx")))

print(f"CSV Files Created     : {csv_count}")

print(f"Excel Files Created   : {excel_count}")

assert csv_count >= 12

assert excel_count >= 1

print("\n✓ All results successfully saved.")

print("\n")

print("=" * 80)

print("SECTION 8.9.11 COMPLETED SUCCESSFULLY")

print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.9 : Machine Learning Utility
# Block 8.9.12 : Final Summary
# =============================================================================

print("=" * 80)
print("SECTION 8.9.12 : FINAL SUMMARY")
print("=" * 80)

# =============================================================================
# Experiment Statistics
# =============================================================================

num_datasets = len(DATASETS)

num_generators = len(MODELS)

num_classifiers = len(CLASSIFIERS)

num_baseline = len(baseline_results)

num_synthetic = len(synthetic_results)

total_experiments = len(utility_results)

# =============================================================================
# Average Utility Metrics
# =============================================================================

avg_metrics = utility_results[[
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "Balanced Accuracy",
    "MCC",
    "Utility Score"
]].mean()

# =============================================================================
# Best Generator
# =============================================================================

best_generator = generator_summary.iloc[0]

# =============================================================================
# Best Classifier
# =============================================================================

best_classifier = classifier_summary.iloc[0]

# =============================================================================
# Best Generator Per Dataset
# =============================================================================

dataset_best = (

    utility_results

    .groupby(

        [

            "Dataset",

            "Synthetic Model"

        ]

    )["Utility Score"]

    .mean()

    .reset_index()

)

dataset_best = (

    dataset_best

    .sort_values(

        [

            "Dataset",

            "Utility Score"

        ],

        ascending=[

            True,

            False

        ]

    )

    .groupby("Dataset")

    .first()

    .reset_index()

)

# =============================================================================
# Overall Evaluation Summary
# =============================================================================

print("\nOVERALL EVALUATION SUMMARY")
print("-" * 80)

print(f"Datasets Evaluated             : {num_datasets}")

print(f"Synthetic Generators           : {num_generators}")

print(f"Downstream Classifiers         : {num_classifiers}")

print(f"Baseline Experiments           : {num_baseline}")

print(f"Synthetic Experiments          : {num_synthetic}")

print(f"Total Experiments              : {total_experiments}")

# =============================================================================
# Average Performance
# =============================================================================

print("\nAVERAGE MACHINE LEARNING UTILITY")
print("-" * 80)

for metric in avg_metrics.index:

    print(f"{metric:<25}: {avg_metrics[metric]:.4f}")

# =============================================================================
# Best Generator
# =============================================================================

print("\nBEST SYNTHETIC DATA GENERATOR")
print("-" * 80)

print(f"Rank               : {int(best_generator['Rank'])}")

print(f"Generator          : {best_generator['Synthetic Model']}")

print(f"Utility Score      : {best_generator['Utility Score']:.4f}")

print(f"Accuracy           : {best_generator['Accuracy']:.4f}")

print(f"F1 Score           : {best_generator['F1 Score']:.4f}")

print(f"ROC-AUC            : {best_generator['ROC-AUC']:.4f}")

print(f"Balanced Accuracy  : {best_generator['Balanced Accuracy']:.4f}")

print(f"MCC                : {best_generator['MCC']:.4f}")

# =============================================================================
# Best Classifier
# =============================================================================

print("\nBEST DOWNSTREAM CLASSIFIER")
print("-" * 80)

print(f"Rank               : {int(best_classifier['Rank'])}")

print(f"Classifier         : {best_classifier['Classifier']}")

print(f"Utility Score      : {best_classifier['Utility Score']:.4f}")

print(f"Accuracy           : {best_classifier['Accuracy']:.4f}")

print(f"F1 Score           : {best_classifier['F1 Score']:.4f}")

print(f"ROC-AUC            : {best_classifier['ROC-AUC']:.4f}")

# =============================================================================
# Dataset-wise Best Generator
# =============================================================================

print("\nBEST GENERATOR FOR EACH DATASET")
print("-" * 80)

display(

    dataset_best.round(4)

)

# =============================================================================
# Final Rankings
# =============================================================================

print("\nOVERALL SYNTHETIC GENERATOR RANKING")
print("-" * 80)

display(

    generator_summary.round(4)

)

print("\nOVERALL CLASSIFIER RANKING")
print("-" * 80)

display(

    classifier_summary.round(4)

)

# =============================================================================
# Generated Outputs
# =============================================================================

print("\nGENERATED OUTPUTS")
print("-" * 80)

print("CSV Files")

print(" • baseline_results.csv")

print(" • synthetic_results.csv")

print(" • utility_results.csv")

print(" • generator_summary.csv")

print(" • classifier_summary.csv")

print(" • dataset_generator_summary.csv")

print(" • dataset_classifier_summary.csv")

print(" • publication_summary.csv")

print(" • roc_summary.csv")

print(" • roc_average.csv")

print(" • precision_recall_summary.csv")

print(" • precision_recall_average.csv")

print()

print("Excel Workbook")

print(" • Machine_Learning_Utility.xlsx")

print()

print("Figures")

print(" • ROC Curves")

print(" • Precision–Recall Curves")

# =============================================================================
# Completion Summary
# =============================================================================

print("\nCOMPLETED TASKS")
print("-" * 80)

completed_tasks = [

    "Reference Baseline Evaluation",

    "Train on Synthetic → Test on Real (TSTR)",

    "Utility Metric Calculation",

    "Accuracy Evaluation",

    "Precision Evaluation",

    "Recall Evaluation",

    "F1 Score Evaluation",

    "ROC-AUC Evaluation",

    "Balanced Accuracy Evaluation",

    "Matthews Correlation Coefficient",

    "ROC Curve Comparison",

    "Precision–Recall Curve Comparison",

    "Utility Summary Tables",

    "CSV Export",

    "Excel Export",

    "Publication-Ready Figures"

]

for task in completed_tasks:

    print(f"✓ {task}")

# =============================================================================
# Final Message
# =============================================================================

print("\n" + "=" * 80)

print("MACHINE LEARNING UTILITY EVALUATION COMPLETED SUCCESSFULLY")

print("=" * 80)

print("""
Section 8.9 has been completed successfully.

Outputs generated include:

• Reference baseline evaluation
• TSTR evaluation
• 54 synthetic experiments
• 9 baseline experiments
• Utility metrics
• ROC analysis
• Precision–Recall analysis
• Publication-ready summary tables
• CSV exports
• Excel workbook
• High-resolution figures

The generated results are now ready for:

→ Section 8.10 : Privacy Evaluation
→ Section 8.11 : Statistical Significance Testing
→ Section 8.12 : Overall Model Ranking
""")

print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10 : Privacy Evaluation
# Block 8.10.1 : Imports & Configuration
# =============================================================================

print("=" * 80)
print("SECTION 8.10.1 : PRIVACY EVALUATION")
print("Imports & Configuration")
print("=" * 80)

# =============================================================================
# Standard Library Imports
# =============================================================================

import os
import gc
import random
import warnings
from pathlib import Path

# =============================================================================
# Numerical Libraries
# =============================================================================

import numpy as np
import pandas as pd

# =============================================================================
# Visualization Libraries
# =============================================================================

import matplotlib.pyplot as plt

# =============================================================================
# Machine Learning
# =============================================================================

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    confusion_matrix,

    roc_curve,

    auc

)

from sklearn.neighbors import (

    NearestNeighbors

)

from sklearn.metrics.pairwise import (

    euclidean_distances,

    cosine_distances,

    manhattan_distances

)

from sklearn.preprocessing import (

    StandardScaler

)

# =============================================================================
# Statistics
# =============================================================================

from scipy.spatial.distance import cdist

from scipy.stats import entropy

# =============================================================================
# Progress Bar
# =============================================================================

from tqdm.auto import tqdm

# =============================================================================
# Display Options
# =============================================================================

pd.set_option("display.max_columns", None)

pd.set_option("display.max_rows", 200)

pd.set_option("display.width", 180)

pd.set_option("display.max_colwidth", None)

warnings.filterwarnings("ignore")

# =============================================================================
# Random Seed
# =============================================================================

RANDOM_STATE = CONFIG["seed"]

random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

# =============================================================================
# Privacy Evaluation Directories
# =============================================================================

PRIVACY_DIR = RESULTS_DIR / "evaluation" / "privacy"

PRIVACY_CSV_DIR = PRIVACY_DIR / "csv"

PRIVACY_EXCEL_DIR = PRIVACY_DIR / "excel"

PRIVACY_FIGURE_DIR = PRIVACY_DIR / "figures"

# =============================================================================
# Create Directories
# =============================================================================

directories = [

    PRIVACY_DIR,

    PRIVACY_CSV_DIR,

    PRIVACY_EXCEL_DIR,

    PRIVACY_FIGURE_DIR

]

for directory in directories:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

# =============================================================================
# Dataset Names
# =============================================================================

DATASETS = [

    "Adult Income",

    "Bank Marketing",

    "Breast Cancer"

]

# =============================================================================
# Synthetic Generators
# =============================================================================

MODELS = [

    "Gaussian Multivariate",

    "Gaussian Copula",

    "CTGAN",

    "TVAE",

    "DP-CTGAN",

    "Proposed SPP-GAN"

]

# =============================================================================
# Global Containers
# =============================================================================

membership_results = pd.DataFrame()

attribute_results = pd.DataFrame()

reidentification_results = pd.DataFrame()

distance_results = pd.DataFrame()

privacy_scores = pd.DataFrame()

privacy_summary = pd.DataFrame()

privacy_rankings = pd.DataFrame()

privacy_figures = {}

# =============================================================================
# Privacy Configuration
# =============================================================================

PRIVACY_CONFIG = {

    "nearest_neighbors": 1,

    "distance_metric": "euclidean",

    "random_state": RANDOM_STATE,

    "similarity_threshold": 0.05,

    "privacy_score_scale": 100

}

# =============================================================================
# Environment Verification
# =============================================================================

print("\nEnvironment Verification")

print("-" * 80)

print(f"Project Root              : {PROJECT_ROOT}")

print(f"Results Directory         : {RESULTS_DIR}")

print(f"Privacy Directory         : {PRIVACY_DIR}")

print(f"CSV Directory             : {PRIVACY_CSV_DIR}")

print(f"Excel Directory           : {PRIVACY_EXCEL_DIR}")

print(f"Figures Directory         : {PRIVACY_FIGURE_DIR}")

print()

print(f"Datasets                  : {len(DATASETS)}")

for dataset in DATASETS:

    print(f"   • {dataset}")

print()

print(f"Synthetic Generators      : {len(MODELS)}")

for model in MODELS:

    print(f"   • {model}")

print()

print("Privacy Configuration")

print("-" * 80)

for key, value in PRIVACY_CONFIG.items():

    print(f"{key:<25}: {value}")

print()

print(f"Random Seed              : {RANDOM_STATE}")

print()

print("=" * 80)

print("SECTION 8.10.1 COMPLETED SUCCESSFULLY")

print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10 : Privacy Evaluation
# Block 8.10.2 : Helper Functions
# =============================================================================

print("=" * 80)
print("SECTION 8.10.2 : HELPER FUNCTIONS")
print("=" * 80)

# =============================================================================
# Numeric Feature Selection
# =============================================================================

def get_numeric_columns(df):

    """
    Return numeric columns only.
    """

    return df.select_dtypes(include=[np.number]).columns.tolist()


# =============================================================================
# Distance Calculations
# =============================================================================

def compute_distance_matrix(real_df,
                            synthetic_df,
                            metric="euclidean"):

    """
    Compute pairwise distance matrix between
    real and synthetic datasets.
    """

    real_num = real_df[get_numeric_columns(real_df)]

    synthetic_num = synthetic_df[get_numeric_columns(synthetic_df)]

    if metric == "euclidean":

        return euclidean_distances(real_num,
                                   synthetic_num)

    elif metric == "manhattan":

        return manhattan_distances(real_num,
                                   synthetic_num)

    elif metric == "cosine":

        return cosine_distances(real_num,
                                synthetic_num)

    else:

        raise ValueError(f"Unsupported metric: {metric}")


# =============================================================================
# Nearest Neighbor Search
# =============================================================================

def nearest_neighbor_distance(real_df,
                              synthetic_df):

    """
    Compute nearest-neighbor distance
    from synthetic to real samples.
    """

    real_num = real_df[get_numeric_columns(real_df)]

    synthetic_num = synthetic_df[get_numeric_columns(synthetic_df)]

    nn = NearestNeighbors(

        n_neighbors=1,

        metric="euclidean"

    )

    nn.fit(real_num)

    distances, indices = nn.kneighbors(

        synthetic_num

    )

    return distances.flatten(), indices.flatten()


# =============================================================================
# Similarity Score
# =============================================================================

def similarity_score(real_df,
                     synthetic_df):

    """
    Convert nearest-neighbor distance into
    similarity score (higher = more similar).
    """

    distances, _ = nearest_neighbor_distance(

        real_df,

        synthetic_df

    )

    similarity = 1.0 / (1.0 + distances)

    return {

        "Mean Similarity": similarity.mean(),

        "Median Similarity": np.median(similarity),

        "Max Similarity": similarity.max(),

        "Min Similarity": similarity.min()

    }


# =============================================================================
# Membership Disclosure Rate
# =============================================================================

def disclosure_rate(distances,
                    threshold=0.05):

    """
    Percentage of synthetic records
    that are extremely close to real data.
    """

    return np.mean(

        distances <= threshold

    )


# =============================================================================
# Re-identification Risk
# =============================================================================

def reidentification_risk(distances,
                          threshold=0.05):

    """
    Estimate re-identification probability.
    """

    return {

        "Risk": np.mean(distances < threshold),

        "Safe": np.mean(distances >= threshold)

    }


# =============================================================================
# Composite Privacy Score
# =============================================================================

def privacy_score(

        membership_attack,

        attribute_disclosure,

        reidentification

):

    """
    Compute overall privacy score.

    Higher = better privacy.
    """

    score = (

        (1.0 - membership_attack) * 0.40 +

        (1.0 - attribute_disclosure) * 0.30 +

        (1.0 - reidentification) * 0.30

    )

    return score * 100


# =============================================================================
# Summary Statistics
# =============================================================================

def distance_statistics(distances):

    """
    Descriptive statistics of distances.
    """

    return {

        "Mean": np.mean(distances),

        "Median": np.median(distances),

        "Std": np.std(distances),

        "Min": np.min(distances),

        "Max": np.max(distances)

    }


# =============================================================================
# Histogram Helper
# =============================================================================

def plot_distance_histogram(

        distances,

        title,

        save_path=None

):

    plt.figure(figsize=(8,5))

    plt.hist(

        distances,

        bins=40,

        edgecolor="black"

    )

    plt.title(title)

    plt.xlabel("Distance")

    plt.ylabel("Frequency")

    plt.grid(alpha=0.30)

    plt.tight_layout()

    if save_path is not None:

        plt.savefig(

            save_path,

            dpi=600,

            bbox_inches="tight"

        )

    plt.show()

    plt.close()


# =============================================================================
# Boxplot Helper
# =============================================================================

def plot_distance_boxplot(

        values,

        labels,

        title,

        save_path=None

):

    plt.figure(figsize=(10,6))

    plt.boxplot(

        values,

        labels=labels,

        showfliers=False

    )

    plt.title(title)

    plt.ylabel("Distance")

    plt.grid(alpha=0.30)

    plt.tight_layout()

    if save_path is not None:

        plt.savefig(

            save_path,

            dpi=600,

            bbox_inches="tight"

        )

    plt.show()

    plt.close()


# =============================================================================
# Ranking Helper
# =============================================================================

def rank_models(

        dataframe,

        metric,

        ascending=False

):

    """
    Rank generators based on metric.
    """

    ranked = dataframe.copy()

    ranked = ranked.sort_values(

        metric,

        ascending=ascending

    ).reset_index(drop=True)

    ranked.insert(

        0,

        "Rank",

        np.arange(

            1,

            len(ranked)+1

        )

    )

    return ranked


# =============================================================================
# Verification
# =============================================================================

print("\nAvailable Helper Functions")

print("-" * 80)

helpers = [

    "get_numeric_columns()",

    "compute_distance_matrix()",

    "nearest_neighbor_distance()",

    "similarity_score()",

    "disclosure_rate()",

    "reidentification_risk()",

    "privacy_score()",

    "distance_statistics()",

    "plot_distance_histogram()",

    "plot_distance_boxplot()",

    "rank_models()"

]

for h in helpers:

    print(f"✓ {h}")

print("\nTotal Helper Functions :", len(helpers))

print("\n" + "=" * 80)
print("SECTION 8.10.2 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.3 : Membership Inference Attack (MIA)
# =============================================================================

print("="*80)
print("SECTION 8.10.3 : MEMBERSHIP INFERENCE ATTACK")
print("="*80)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)

mia_results = []

# -----------------------------------------------------------------------------
# Evaluate Each Dataset and Synthetic Generator
# -----------------------------------------------------------------------------

for dataset in DATASETS:

    print(f"\nDataset : {dataset}")

    real_train = original_datasets[dataset]["train"].copy()
    real_test  = original_datasets[dataset]["test"].copy()

    for generator in MODELS:

        synthetic = synthetic_datasets[dataset][generator].copy()

        # -------------------------------------------------------------
        # Numeric Features Only
        # -------------------------------------------------------------

        cols = get_numeric_columns(real_train)

        scaler = StandardScaler()

        train_x = scaler.fit_transform(real_train[cols])

        test_x = scaler.transform(real_test[cols])

        syn_x = scaler.transform(synthetic[cols])

        # -------------------------------------------------------------
        # Distance to Synthetic Records
        # -------------------------------------------------------------

        nn = NearestNeighbors(n_neighbors=1)

        nn.fit(syn_x)

        train_dist, _ = nn.kneighbors(train_x)

        test_dist, _ = nn.kneighbors(test_x)

        # -------------------------------------------------------------
        # Attack Prediction
        # Smaller distance → predicted member
        # -------------------------------------------------------------

        scores = np.concatenate([

            -train_dist.flatten(),

            -test_dist.flatten()

        ])

        labels = np.concatenate([

            np.ones(len(train_dist)),

            np.zeros(len(test_dist))

        ])

        threshold = np.median(scores)

        predictions = (scores >= threshold).astype(int)

        # -------------------------------------------------------------
        # Metrics
        # -------------------------------------------------------------

        acc = accuracy_score(labels, predictions)

        pre = precision_score(labels, predictions)

        rec = recall_score(labels, predictions)

        f1 = f1_score(labels, predictions)

        auc_score = roc_auc_score(labels, scores)

        success = predictions.mean()

        mia_results.append({

            "Dataset": dataset,

            "Synthetic Model": generator,

            "Attack Accuracy": acc,

            "Attack Precision": pre,

            "Attack Recall": rec,

            "Attack F1": f1,

            "Attack ROC-AUC": auc_score,

            "Attack Success Rate": success

        })

        # -------------------------------------------------------------
        # ROC Curve
        # -------------------------------------------------------------

        fpr, tpr, _ = roc_curve(labels, scores)

        plt.figure(figsize=(6,5))

        plt.plot(fpr, tpr,
                 linewidth=2,
                 label=f"AUC = {auc_score:.3f}")

        plt.plot([0,1],[0,1],'k--')

        plt.xlabel("False Positive Rate")

        plt.ylabel("True Positive Rate")

        plt.title(f"{dataset}\n{generator}")

        plt.legend()

        plt.grid(alpha=0.3)

        plt.tight_layout()

        plt.savefig(

            PRIVACY_FIGURE_DIR /

            f"{dataset}_{generator}_MIA_ROC.png",

            dpi=600

        )

        plt.show()

        plt.close()

        # -------------------------------------------------------------
        # Confusion Matrix
        # -------------------------------------------------------------

        cm = confusion_matrix(labels, predictions)

        disp = ConfusionMatrixDisplay(cm)

        disp.plot(cmap="Blues")

        plt.title(f"{dataset}\n{generator}")

        plt.tight_layout()

        plt.savefig(

            PRIVACY_FIGURE_DIR /

            f"{dataset}_{generator}_MIA_CM.png",

            dpi=600

        )

        plt.show()

        plt.close()

# =============================================================================
# Summary Table
# =============================================================================

membership_results = pd.DataFrame(mia_results)

membership_results = membership_results.sort_values(

    "Attack ROC-AUC"

).reset_index(drop=True)

print("\nMembership Inference Summary")

display(

    membership_results.round(4)

)

# =============================================================================
# Average Results
# =============================================================================

membership_summary = (

    membership_results

    .groupby(

        "Synthetic Model"

    )[

        [

            "Attack Accuracy",

            "Attack Precision",

            "Attack Recall",

            "Attack F1",

            "Attack ROC-AUC",

            "Attack Success Rate"

        ]

    ]

    .mean()

    .sort_values(

        "Attack ROC-AUC"

    )

)

print("\nAverage Membership Inference Performance")

display(

    membership_summary.round(4)

)

print("\n✓ Membership Inference Attack Evaluation Completed")

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.4 : Attribute Disclosure Risk
# =============================================================================

print("="*80)
print("SECTION 8.10.4 : ATTRIBUTE DISCLOSURE RISK")
print("="*80)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

attribute_results = []

# -----------------------------------------------------------------------------
# Attribute Disclosure Evaluation
# -----------------------------------------------------------------------------

for dataset in DATASETS:

    print(f"\nDataset : {dataset}")

    real_train = original_datasets[dataset]["train"].copy()

    real_test = original_datasets[dataset]["test"].copy()

    # -------------------------------------------------------------------------
    # Sensitive Attribute
    # -------------------------------------------------------------------------

    sensitive_col = real_train.columns[-1]

    feature_cols = [

        c for c in real_train.columns

        if c != sensitive_col

    ]

    x_test = real_test[feature_cols]

    y_test = real_test[sensitive_col]

    for generator in MODELS:

        synthetic = synthetic_datasets[dataset][generator].copy()

        x_train = synthetic[feature_cols]

        y_train = synthetic[sensitive_col]

        # ---------------------------------------------------------------------
        # Train Attack Model
        # ---------------------------------------------------------------------

        attack_model = RandomForestClassifier(

            n_estimators=200,

            random_state=RANDOM_STATE,

            n_jobs=-1

        )

        attack_model.fit(

            x_train,

            y_train

        )

        y_pred = attack_model.predict(

            x_test

        )

        acc = accuracy_score(

            y_test,

            y_pred

        )

        pre = precision_score(

            y_test,

            y_pred,

            average="weighted",

            zero_division=0

        )

        rec = recall_score(

            y_test,

            y_pred,

            average="weighted",

            zero_division=0

        )

        f1 = f1_score(

            y_test,

            y_pred,

            average="weighted",

            zero_division=0

        )

        disclosure_rate = acc

        attribute_results.append({

            "Dataset": dataset,

            "Synthetic Model": generator,

            "Disclosure Accuracy": acc,

            "Disclosure Precision": pre,

            "Disclosure Recall": rec,

            "Disclosure F1": f1,

            "Disclosure Rate": disclosure_rate

        })

# =============================================================================
# Summary Tables
# =============================================================================

attribute_results = pd.DataFrame(attribute_results)

print("\nAttribute Disclosure Results")

display(

    attribute_results.round(4)

)

# -----------------------------------------------------------------------------
# Generator Comparison
# -----------------------------------------------------------------------------

attribute_summary = (

    attribute_results

    .groupby(

        "Synthetic Model"

    )[

        [

            "Disclosure Accuracy",

            "Disclosure Precision",

            "Disclosure Recall",

            "Disclosure F1",

            "Disclosure Rate"

        ]

    ]

    .mean()

    .sort_values(

        "Disclosure Rate"

    )

    .reset_index()

)

print("\nGenerator Comparison")

display(

    attribute_summary.round(4)

)

# -----------------------------------------------------------------------------
# Dataset-wise Comparison
# -----------------------------------------------------------------------------

dataset_summary = (

    attribute_results

    .pivot_table(

        index="Dataset",

        columns="Synthetic Model",

        values="Disclosure Rate"

    )

)

print("\nDataset-wise Disclosure Rate")

display(

    dataset_summary.round(4)

)

print("\n✓ Attribute Disclosure Risk Evaluation Completed")

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.5 : Re-identification Risk
# =============================================================================

print("="*80)
print("SECTION 8.10.5 : RE-IDENTIFICATION RISK")
print("="*80)

from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde

reid_results = []

# -----------------------------------------------------------------------------
# Evaluate Each Dataset & Generator
# -----------------------------------------------------------------------------

for dataset in DATASETS:

    print(f"\nDataset : {dataset}")

    real = original_datasets[dataset]["train"].copy()

    cols = get_numeric_columns(real)

    real_x = StandardScaler().fit_transform(real[cols])

    for generator in MODELS:

        synthetic = synthetic_datasets[dataset][generator].copy()

        syn_x = StandardScaler().fit_transform(synthetic[cols])

        # -------------------------------------------------------------
        # Top-1 Nearest Neighbor
        # -------------------------------------------------------------

        nn = NearestNeighbors(

            n_neighbors=1,

            metric="euclidean"

        )

        nn.fit(real_x)

        distances, indices = nn.kneighbors(syn_x)

        distances = distances.flatten()

        # -------------------------------------------------------------
        # Privacy Metrics
        # -------------------------------------------------------------

        exact_match = np.mean(distances == 0)

        top1_match = np.mean(distances < 0.05)

        linkage_rate = np.mean(distances < 0.10)

        identity_probability = np.mean(

            1.0 / (1.0 + distances)

        )

        reid_results.append({

            "Dataset": dataset,

            "Synthetic Model": generator,

            "Exact Match Rate": exact_match,

            "Top-1 NN Match": top1_match,

            "Record Linkage Rate": linkage_rate,

            "Identity Disclosure Probability": identity_probability

        })

        # -------------------------------------------------------------
        # Histogram
        # -------------------------------------------------------------

        plt.figure(figsize=(6,4))

        plt.hist(

            distances,

            bins=40,

            edgecolor="black"

        )

        plt.title(f"{dataset}\n{generator}")

        plt.xlabel("Nearest Neighbor Distance")

        plt.ylabel("Frequency")

        plt.grid(alpha=0.30)

        plt.tight_layout()

        plt.savefig(

            PRIVACY_FIGURE_DIR /

            f"{dataset}_{generator}_Histogram.png",

            dpi=600

        )

        plt.show()

        plt.close()

        # -------------------------------------------------------------
        # Density Plot
        # -------------------------------------------------------------

        density = gaussian_kde(distances)

        x = np.linspace(

            distances.min(),

            distances.max(),

            300

        )

        plt.figure(figsize=(6,4))

        plt.plot(

            x,

            density(x),

            linewidth=2

        )

        plt.fill_between(

            x,

            density(x),

            alpha=0.30

        )

        plt.title(f"{dataset}\n{generator}")

        plt.xlabel("Nearest Neighbor Distance")

        plt.ylabel("Density")

        plt.grid(alpha=0.30)

        plt.tight_layout()

        plt.savefig(

            PRIVACY_FIGURE_DIR /

            f"{dataset}_{generator}_Density.png",

            dpi=600

        )

        plt.show()

        plt.close()

# =============================================================================
# Summary Tables
# =============================================================================

reidentification_results = pd.DataFrame(reid_results)

print("\nRe-identification Results")

display(

    reidentification_results.round(4)

)

# -----------------------------------------------------------------------------
# Generator Comparison
# -----------------------------------------------------------------------------

reidentification_summary = (

    reidentification_results

    .groupby(

        "Synthetic Model"

    )[

        [

            "Exact Match Rate",

            "Top-1 NN Match",

            "Record Linkage Rate",

            "Identity Disclosure Probability"

        ]

    ]

    .mean()

    .sort_values(

        "Identity Disclosure Probability"

    )

    .reset_index()

)

print("\nGenerator Comparison")

display(

    reidentification_summary.round(4)

)

print("\n✓ Re-identification Risk Evaluation Completed")

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.6 : Distance-Based Privacy Metrics
# =============================================================================

print("="*80)
print("SECTION 8.10.6 : DISTANCE-BASED PRIVACY METRICS")
print("="*80)

distance_results = []

# -----------------------------------------------------------------------------
# Evaluate Distance Metrics
# -----------------------------------------------------------------------------

for dataset in DATASETS:

    print(f"\nDataset : {dataset}")

    real = original_datasets[dataset]["train"].copy()

    cols = get_numeric_columns(real)

    scaler = StandardScaler()

    real_x = scaler.fit_transform(real[cols])

    for generator in MODELS:

        synthetic = synthetic_datasets[dataset][generator].copy()

        syn_x = scaler.transform(synthetic[cols])

        # -------------------------------------------------------------
        # Distance Matrices
        # -------------------------------------------------------------

        euclidean = euclidean_distances(syn_x, real_x)

        manhattan = manhattan_distances(syn_x, real_x)

        cosine = cosine_distances(syn_x, real_x)

        # -------------------------------------------------------------
        # Nearest Neighbor Distance
        # -------------------------------------------------------------

        nn = NearestNeighbors(

            n_neighbors=1,

            metric="euclidean"

        )

        nn.fit(real_x)

        nn_dist, _ = nn.kneighbors(syn_x)

        nn_dist = nn_dist.flatten()

        # -------------------------------------------------------------
        # Mean Metrics
        # -------------------------------------------------------------

        distance_results.append({

            "Dataset": dataset,

            "Synthetic Model": generator,

            "Nearest Neighbor Distance": nn_dist.mean(),

            "Mean Euclidean Distance": euclidean.mean(),

            "Mean Manhattan Distance": manhattan.mean(),

            "Mean Cosine Distance": cosine.mean()

        })

        # -------------------------------------------------------------
        # Histogram
        # -------------------------------------------------------------

        plt.figure(figsize=(6,4))

        plt.hist(

            nn_dist,

            bins=35,

            edgecolor="black"

        )

        plt.title(f"{dataset}\n{generator}")

        plt.xlabel("Nearest Neighbor Distance")

        plt.ylabel("Frequency")

        plt.grid(alpha=0.30)

        plt.tight_layout()

        plt.savefig(

            PRIVACY_FIGURE_DIR /

            f"{dataset}_{generator}_Distance_Histogram.png",

            dpi=600

        )

        plt.show()

        plt.close()

# =============================================================================
# Summary Table
# =============================================================================

distance_results = pd.DataFrame(distance_results)

print("\nDistance Metrics")

display(

    distance_results.round(4)

)

# -----------------------------------------------------------------------------
# Generator Comparison
# -----------------------------------------------------------------------------

distance_summary = (

    distance_results

    .groupby(

        "Synthetic Model"

    )[

        [

            "Nearest Neighbor Distance",

            "Mean Euclidean Distance",

            "Mean Manhattan Distance",

            "Mean Cosine Distance"

        ]

    ]

    .mean()

    .sort_values(

        "Nearest Neighbor Distance",

        ascending=False

    )

    .reset_index()

)

print("\nGenerator Comparison")

display(

    distance_summary.round(4)

)

# -----------------------------------------------------------------------------
# Boxplots
# -----------------------------------------------------------------------------

metrics = [

    "Nearest Neighbor Distance",

    "Mean Euclidean Distance",

    "Mean Manhattan Distance",

    "Mean Cosine Distance"

]

for metric in metrics:

    plt.figure(figsize=(8,5))

    distance_results.boxplot(

        column=metric,

        by="Synthetic Model",

        rot=30,

        grid=False

    )

    plt.title(metric)

    plt.suptitle("")

    plt.ylabel(metric)

    plt.tight_layout()

    plt.savefig(

        PRIVACY_FIGURE_DIR /

        f"{metric.replace(' ','_')}_Boxplot.png",

        dpi=600

    )

    plt.show()

    plt.close()

# =============================================================================
# Summary Statistics
# =============================================================================

summary_statistics = (

    distance_results

    .describe()

    .T

)

print("\nSummary Statistics")

display(

    summary_statistics.round(4)

)

print("\n✓ Distance-Based Privacy Metrics Completed")

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.7 : Privacy Score Calculation
# =============================================================================

print("="*80)
print("SECTION 8.10.7 : PRIVACY SCORE CALCULATION")
print("="*80)

# -----------------------------------------------------------------------------
# Aggregate Results
# -----------------------------------------------------------------------------

mia = (

    membership_results

    .groupby("Synthetic Model")["Attack Success Rate"]

    .mean()

)

attribute = (

    attribute_results

    .groupby("Synthetic Model")["Disclosure Rate"]

    .mean()

)

reid = (

    reidentification_results

    .groupby("Synthetic Model")["Identity Disclosure Probability"]

    .mean()

)

# -----------------------------------------------------------------------------
# Combine Metrics
# -----------------------------------------------------------------------------

privacy_scores = pd.concat(

    [

        mia,

        attribute,

        reid

    ],

    axis=1

)

privacy_scores.columns = [

    "Membership Attack",

    "Attribute Disclosure",

    "Re-identification"

]

privacy_scores.reset_index(inplace=True)

# -----------------------------------------------------------------------------
# Composite Privacy Score
# -----------------------------------------------------------------------------

privacy_scores["Privacy Score"] = (

    (1 - privacy_scores["Membership Attack"]) * 0.40 +

    (1 - privacy_scores["Attribute Disclosure"]) * 0.30 +

    (1 - privacy_scores["Re-identification"]) * 0.30

) * 100

# -----------------------------------------------------------------------------
# Ranking
# -----------------------------------------------------------------------------

privacy_scores = (

    privacy_scores

    .sort_values(

        "Privacy Score",

        ascending=False

    )

    .reset_index(drop=True)

)

privacy_scores.insert(

    0,

    "Rank",

    np.arange(

        1,

        len(privacy_scores)+1

    )

)

# -----------------------------------------------------------------------------
# Display Results
# -----------------------------------------------------------------------------

print("\nComposite Privacy Scores")

display(

    privacy_scores.round(4)

)

# -----------------------------------------------------------------------------
# Best Privacy Model
# -----------------------------------------------------------------------------

best = privacy_scores.iloc[0]

print("\n" + "="*80)

print("BEST PRIVACY-PRESERVING MODEL")

print("="*80)

print(f"Rank               : {best['Rank']}")

print(f"Generator          : {best['Synthetic Model']}")

print(f"Privacy Score      : {best['Privacy Score']:.2f}/100")

print(f"Membership Attack  : {best['Membership Attack']:.4f}")

print(f"Attribute Risk     : {best['Attribute Disclosure']:.4f}")

print(f"Re-ID Risk         : {best['Re-identification']:.4f}")

# -----------------------------------------------------------------------------
# Privacy Ranking Plot
# -----------------------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.bar(

    privacy_scores["Synthetic Model"],

    privacy_scores["Privacy Score"]

)

plt.xticks(rotation=30)

plt.ylabel("Privacy Score")

plt.title("Composite Privacy Ranking")

plt.grid(axis="y", alpha=0.30)

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR /

    "Privacy_Score_Ranking.png",

    dpi=600

)

plt.show()

print("\n✓ Privacy Score Calculation Completed")

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.8 : Privacy Comparison Tables
# =============================================================================

print("="*80)
print("SECTION 8.10.8 : PRIVACY COMPARISON TABLES")
print("="*80)

# =============================================================================
# Table 1 : Membership Inference Attack
# =============================================================================

membership_table = (

    membership_results

    .groupby("Synthetic Model")[

        [

            "Attack Accuracy",

            "Attack Precision",

            "Attack Recall",

            "Attack F1",

            "Attack ROC-AUC",

            "Attack Success Rate"

        ]

    ]

    .mean()

    .sort_values(

        "Attack Success Rate"

    )

    .reset_index()

)

# =============================================================================
# Table 2 : Attribute Disclosure
# =============================================================================

attribute_table = (

    attribute_results

    .groupby("Synthetic Model")[

        [

            "Disclosure Accuracy",

            "Disclosure Precision",

            "Disclosure Recall",

            "Disclosure F1",

            "Disclosure Rate"

        ]

    ]

    .mean()

    .sort_values(

        "Disclosure Rate"

    )

    .reset_index()

)

# =============================================================================
# Table 3 : Re-identification Risk
# =============================================================================

reidentification_table = (

    reidentification_results

    .groupby("Synthetic Model")[

        [

            "Exact Match Rate",

            "Top-1 NN Match",

            "Record Linkage Rate",

            "Identity Disclosure Probability"

        ]

    ]

    .mean()

    .sort_values(

        "Identity Disclosure Probability"

    )

    .reset_index()

)

# =============================================================================
# Table 4 : Distance Metrics
# =============================================================================

distance_table = (

    distance_results

    .groupby("Synthetic Model")[

        [

            "Nearest Neighbor Distance",

            "Mean Euclidean Distance",

            "Mean Manhattan Distance",

            "Mean Cosine Distance"

        ]

    ]

    .mean()

    .sort_values(

        "Nearest Neighbor Distance",

        ascending=False

    )

    .reset_index()

)

# =============================================================================
# Table 5 : Privacy Score Ranking
# =============================================================================

privacy_ranking = privacy_scores.copy()

# =============================================================================
# Display Tables
# =============================================================================

tables = {

    "Membership Attack": membership_table,

    "Attribute Disclosure": attribute_table,

    "Re-identification": reidentification_table,

    "Distance Metrics": distance_table,

    "Privacy Score Ranking": privacy_ranking

}

for title, table in tables.items():

    print("\n" + "="*80)

    print(title.upper())

    print("="*80)

    display(

        table.round(4)

    )

# =============================================================================
# Overall Privacy Leader
# =============================================================================

best_model = privacy_ranking.iloc[0]

print("\n" + "="*80)

print("OVERALL BEST PRIVACY-PRESERVING MODEL")

print("="*80)

print(f"Generator      : {best_model['Synthetic Model']}")

print(f"Privacy Score  : {best_model['Privacy Score']:.2f}/100")

print(f"Overall Rank   : {int(best_model['Rank'])}")

print("\n✓ Privacy comparison tables generated successfully.")

print("="*80)
print("SECTION 8.10.8 COMPLETED SUCCESSFULLY")
print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.9 : Privacy Visualizations
# =============================================================================

print("="*80)
print("SECTION 8.10.9 : PRIVACY VISUALIZATIONS")
print("="*80)

# =============================================================================
# 1. Privacy Score Ranking (Bar Chart)
# =============================================================================

plt.figure(figsize=(8,5))

ranking = privacy_scores.sort_values(
    "Privacy Score",
    ascending=False
)

plt.bar(
    ranking["Synthetic Model"],
    ranking["Privacy Score"]
)

plt.xticks(rotation=30)

plt.ylabel("Privacy Score")

plt.title("Overall Privacy Ranking")

plt.grid(axis="y", alpha=0.30)

plt.tight_layout()

plt.savefig(
    PRIVACY_FIGURE_DIR/"Privacy_Ranking.png",
    dpi=600
)

plt.show()

# =============================================================================
# 2. Membership Attack Comparison
# =============================================================================

membership_table.set_index(

    "Synthetic Model"

)[

    "Attack Success Rate"

].plot(

    kind="bar",

    figsize=(8,5)

)

plt.ylabel("Attack Success Rate")

plt.title("Membership Inference Attack")

plt.grid(axis="y", alpha=0.30)

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR/

    "Membership_Attack.png",

    dpi=600

)

plt.show()

# =============================================================================
# 3. Attribute Disclosure
# =============================================================================

attribute_table.set_index(

    "Synthetic Model"

)[

    "Disclosure Rate"

].plot(

    kind="bar",

    figsize=(8,5)

)

plt.ylabel("Disclosure Rate")

plt.title("Attribute Disclosure Risk")

plt.grid(axis="y", alpha=0.30)

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR/

    "Attribute_Disclosure.png",

    dpi=600

)

plt.show()

# =============================================================================
# 4. Re-identification Risk
# =============================================================================

reidentification_table.set_index(

    "Synthetic Model"

)[

    "Identity Disclosure Probability"

].plot(

    kind="bar",

    figsize=(8,5)

)

plt.ylabel("Identity Disclosure Probability")

plt.title("Re-identification Risk")

plt.grid(axis="y", alpha=0.30)

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR/

    "Reidentification_Risk.png",

    dpi=600

)

plt.show()

# =============================================================================
# 5. Distance Metrics Boxplots
# =============================================================================

distance_results.boxplot(

    column=[

        "Nearest Neighbor Distance",

        "Mean Euclidean Distance",

        "Mean Manhattan Distance",

        "Mean Cosine Distance"

    ],

    figsize=(10,6)

)

plt.title("Distance-Based Privacy Metrics")

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR/

    "Distance_Boxplots.png",

    dpi=600

)

plt.show()

# =============================================================================
# 6. Privacy Heatmap
# =============================================================================

heatmap_data = privacy_scores.set_index(

    "Synthetic Model"

)[

    [

        "Membership Attack",

        "Attribute Disclosure",

        "Re-identification",

        "Privacy Score"

    ]

]

plt.figure(figsize=(8,5))

plt.imshow(

    heatmap_data,

    aspect="auto"

)

plt.xticks(

    range(len(heatmap_data.columns)),

    heatmap_data.columns,

    rotation=30

)

plt.yticks(

    range(len(heatmap_data.index)),

    heatmap_data.index

)

plt.colorbar(label="Value")

plt.title("Privacy Metrics Heatmap")

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR/

    "Privacy_Heatmap.png",

    dpi=600

)

plt.show()

# =============================================================================
# 7. Radar Chart
# =============================================================================

metrics = [

    "Membership Attack",

    "Attribute Disclosure",

    "Re-identification"

]

angles = np.linspace(

    0,

    2*np.pi,

    len(metrics),

    endpoint=False

)

angles = np.concatenate(

    (angles,[angles[0]])

)

plt.figure(figsize=(8,8))

ax = plt.subplot(111, polar=True)

for _, row in privacy_scores.iterrows():

    values = row[metrics].values

    values = np.concatenate((values,[values[0]]))

    ax.plot(

        angles,

        values,

        linewidth=2,

        label=row["Synthetic Model"]

    )

ax.set_xticks(

    angles[:-1]

)

ax.set_xticklabels(metrics)

ax.set_title("Privacy Comparison Radar Chart")

ax.legend(

    bbox_to_anchor=(1.35,1.05),

    fontsize=8

)

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR/

    "Privacy_Radar.png",

    dpi=600

)

plt.show()

# =============================================================================
# 8. Histogram of Privacy Scores
# =============================================================================

plt.figure(figsize=(7,5))

plt.hist(

    privacy_scores["Privacy Score"],

    bins=8,

    edgecolor="black"

)

plt.xlabel("Privacy Score")

plt.ylabel("Frequency")

plt.title("Distribution of Privacy Scores")

plt.grid(alpha=0.30)

plt.tight_layout()

plt.savefig(

    PRIVACY_FIGURE_DIR/

    "PrivacyScore_Histogram.png",

    dpi=600

)

plt.show()

# =============================================================================
# Completed
# =============================================================================

print("\n✓ Privacy visualizations generated successfully.")

print("Figures saved to:")

print(PRIVACY_FIGURE_DIR)

print("="*80)

print("SECTION 8.10.9 COMPLETED SUCCESSFULLY")

print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.10 : Save Results
# =============================================================================

print("="*80)
print("SECTION 8.10.10 : SAVE PRIVACY RESULTS")
print("="*80)

# =============================================================================
# Create Output Directories
# =============================================================================

PRIVACY_CSV_DIR.mkdir(parents=True, exist_ok=True)
PRIVACY_EXCEL_DIR.mkdir(parents=True, exist_ok=True)
PRIVACY_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Save CSV Files
# =============================================================================

membership_results.to_csv(
    PRIVACY_CSV_DIR/"membership_attack.csv",
    index=False
)

attribute_results.to_csv(
    PRIVACY_CSV_DIR/"attribute_disclosure.csv",
    index=False
)

reidentification_results.to_csv(
    PRIVACY_CSV_DIR/"reidentification.csv",
    index=False
)

distance_results.to_csv(
    PRIVACY_CSV_DIR/"distance_metrics.csv",
    index=False
)

privacy_scores.to_csv(
    PRIVACY_CSV_DIR/"privacy_scores.csv",
    index=False
)

membership_table.to_csv(
    PRIVACY_CSV_DIR/"membership_summary.csv",
    index=False
)

attribute_table.to_csv(
    PRIVACY_CSV_DIR/"attribute_summary.csv",
    index=False
)

reidentification_table.to_csv(
    PRIVACY_CSV_DIR/"reidentification_summary.csv",
    index=False
)

distance_table.to_csv(
    PRIVACY_CSV_DIR/"distance_summary.csv",
    index=False
)

privacy_ranking.to_csv(
    PRIVACY_CSV_DIR/"privacy_ranking.csv",
    index=False
)

print("✓ CSV files saved.")

# =============================================================================
# Save Excel Workbook
# =============================================================================

excel_file = PRIVACY_EXCEL_DIR/"Privacy_Evaluation.xlsx"

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    membership_results.to_excel(
        writer,
        sheet_name="Membership Attack",
        index=False
    )

    attribute_results.to_excel(
        writer,
        sheet_name="Attribute Disclosure",
        index=False
    )

    reidentification_results.to_excel(
        writer,
        sheet_name="Reidentification",
        index=False
    )

    distance_results.to_excel(
        writer,
        sheet_name="Distance Metrics",
        index=False
    )

    privacy_scores.to_excel(
        writer,
        sheet_name="Privacy Scores",
        index=False
    )

    membership_table.to_excel(
        writer,
        sheet_name="Membership Summary",
        index=False
    )

    attribute_table.to_excel(
        writer,
        sheet_name="Attribute Summary",
        index=False
    )

    reidentification_table.to_excel(
        writer,
        sheet_name="ReID Summary",
        index=False
    )

    distance_table.to_excel(
        writer,
        sheet_name="Distance Summary",
        index=False
    )

    privacy_ranking.to_excel(
        writer,
        sheet_name="Privacy Ranking",
        index=False
    )

print("✓ Excel workbook saved.")

# =============================================================================
# Summary of Saved Files
# =============================================================================

print("\nSaved CSV Files")

csv_files = sorted(PRIVACY_CSV_DIR.glob("*.csv"))

for file in csv_files:
    print(f"✓ {file.name}")

print("\nExcel Workbook")

print(f"✓ {excel_file.name}")

print("\nFigures")

figure_files = sorted(PRIVACY_FIGURE_DIR.glob("*.png"))

for file in figure_files:
    print(f"✓ {file.name}")

# =============================================================================
# Verification
# =============================================================================

print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

print(f"CSV Files      : {len(csv_files)}")

print(f"Figures        : {len(figure_files)}")

print(f"Excel Workbook : {'Available' if excel_file.exists() else 'Missing'}")

print("\nOutput Directory")

print(PRIVACY_DIR)

print("\n✓ Privacy evaluation results saved successfully.")

print("\n" + "="*80)
print("SECTION 8.10.10 COMPLETED SUCCESSFULLY")
print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.10.11 : Final Summary
# =============================================================================

print("=" * 80)
print("SECTION 8.10.11 : FINAL PRIVACY SUMMARY")
print("=" * 80)

# =============================================================================
# Best Overall Privacy Model
# =============================================================================

best_privacy = privacy_scores.loc[
    privacy_scores["Privacy Score"].idxmax()
]

best_disclosure = attribute_table.loc[
    attribute_table["Disclosure Rate"].idxmin()
]

best_reid = reidentification_table.loc[
    reidentification_table["Identity Disclosure Probability"].idxmin()
]

# =============================================================================
# Publication Summary Table
# =============================================================================

privacy_summary = pd.DataFrame({

    "Criterion": [

        "Best Privacy-Preserving Model",

        "Highest Privacy Score",

        "Lowest Disclosure Risk",

        "Lowest Re-identification Risk"

    ],

    "Generator": [

        best_privacy["Synthetic Model"],

        best_privacy["Synthetic Model"],

        best_disclosure["Synthetic Model"],

        best_reid["Synthetic Model"]

    ],

    "Value": [

        f"{best_privacy['Privacy Score']:.2f}",

        f"{best_privacy['Privacy Score']:.2f}",

        f"{best_disclosure['Disclosure Rate']:.4f}",

        f"{best_reid['Identity Disclosure Probability']:.4f}"

    ]

})

# =============================================================================
# Display Summary
# =============================================================================

print("\nPublication Summary\n")

display(privacy_summary)

print("\n" + "=" * 80)
print("FINAL CONCLUSIONS")
print("=" * 80)

print(f"• Best Privacy-Preserving Model : {best_privacy['Synthetic Model']}")

print(f"• Highest Privacy Score         : {best_privacy['Privacy Score']:.2f}/100")

print(f"• Lowest Disclosure Risk        : "
      f"{best_disclosure['Synthetic Model']} "
      f"({best_disclosure['Disclosure Rate']:.4f})")

print(f"• Lowest Re-identification Risk : "
      f"{best_reid['Synthetic Model']} "
      f"({best_reid['Identity Disclosure Probability']:.4f})")

print("\nPublication Conclusion")
print("-" * 80)

print(
    f"The privacy evaluation demonstrates that "
    f"{best_privacy['Synthetic Model']} achieved the highest overall "
    f"privacy score ({best_privacy['Privacy Score']:.2f}/100). "
    f"It consistently showed strong resistance against membership inference, "
    f"attribute disclosure, and re-identification attacks, indicating superior "
    f"privacy preservation compared with the baseline synthetic data generators. "
    f"These findings support its suitability for privacy-preserving synthetic "
    f"tabular data generation while maintaining compatibility with downstream "
    f"machine learning applications."
)

print("\n" + "=" * 80)
print("PRIVACY EVALUATION COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.11 : Statistical Significance Testing
# =============================================================================

print("="*80)
print("SECTION 8.11 : STATISTICAL SIGNIFICANCE TESTING")
print("="*80)

from scipy.stats import (
    ttest_rel,
    wilcoxon,
    friedmanchisquare
)

# =============================================================================
# Prepare Evaluation Scores
# =============================================================================

# Utility Score
utility_scores = utility_results.pivot_table(
    index="Dataset",
    columns="Synthetic Model",
    values="Utility Score"
)

# Privacy Score
privacy_scores_test = privacy_scores.set_index(
    "Synthetic Model"
)["Privacy Score"]

# Fidelity Score (from Section 8.8)
fidelity_scores = multivariate_summary.set_index(
    "Synthetic Model"
)["Overall Score"]

# =============================================================================
# Paired t-Test (Utility)
# =============================================================================

print("\nPaired t-Test (Utility Score)")
print("-"*80)

models = utility_scores.columns.tolist()

ttest_results = []

for i in range(len(models)):

    for j in range(i+1, len(models)):

        stat, p = ttest_rel(

            utility_scores[models[i]],

            utility_scores[models[j]]

        )

        ttest_results.append({

            "Model A": models[i],

            "Model B": models[j],

            "t-statistic": stat,

            "p-value": p

        })

ttest_results = pd.DataFrame(ttest_results)

display(ttest_results.round(4))

# =============================================================================
# Wilcoxon Signed-Rank Test
# =============================================================================

print("\nWilcoxon Signed-Rank Test")
print("-"*80)

wilcoxon_results = []

for i in range(len(models)):

    for j in range(i+1, len(models)):

        try:

            stat, p = wilcoxon(

                utility_scores[models[i]],

                utility_scores[models[j]]

            )

        except:

            stat, p = np.nan, np.nan

        wilcoxon_results.append({

            "Model A": models[i],

            "Model B": models[j],

            "Statistic": stat,

            "p-value": p

        })

wilcoxon_results = pd.DataFrame(wilcoxon_results)

display(wilcoxon_results.round(4))

# =============================================================================
# Friedman Test
# =============================================================================

print("\nFriedman Test")
print("-"*80)

friedman_stat, friedman_p = friedmanchisquare(

    *[utility_scores[col] for col in utility_scores.columns]

)

friedman_summary = pd.DataFrame({

    "Statistic":[friedman_stat],

    "p-value":[friedman_p]

})

display(friedman_summary.round(4))

# =============================================================================
# Interpretation
# =============================================================================

alpha = 0.05

print("\nInterpretation")
print("-"*80)

if friedman_p < alpha:

    print("✓ Significant differences exist between synthetic generators (p < 0.05).")

else:

    print("✓ No statistically significant difference detected (p ≥ 0.05).")

# =============================================================================
# Summary Table
# =============================================================================

statistics_summary = pd.DataFrame({

    "Test":[

        "Paired t-Test",

        "Wilcoxon",

        "Friedman"

    ],

    "Purpose":[

        "Pairwise mean comparison",

        "Pairwise non-parametric comparison",

        "Overall comparison"

    ]

})

print("\nStatistical Tests Used")

display(statistics_summary)

print("\n✓ Statistical significance testing completed.")

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.12 : Overall Model Ranking
# =============================================================================

print("="*80)
print("SECTION 8.12 : OVERALL MODEL RANKING")
print("="*80)

# =============================================================================
# Weight Configuration
# =============================================================================

FIDELITY_WEIGHT = 0.40
UTILITY_WEIGHT  = 0.30
PRIVACY_WEIGHT  = 0.30

# =============================================================================
# Prepare Scores
# =============================================================================

# Utility Score
utility_score = (

    utility_results

    .groupby("Synthetic Model")["Utility Score"]

    .mean()

    .reset_index()

)

# Privacy Score
privacy_score = privacy_scores[

    ["Synthetic Model", "Privacy Score"]

].copy()

# Fidelity Score
# (Generated in Section 8.8)

fidelity_score = multivariate_summary[

    ["Synthetic Model", "Overall Score"]

].copy()

fidelity_score.rename(

    columns={

        "Overall Score":"Fidelity Score"

    },

    inplace=True

)

# =============================================================================
# Merge Scores
# =============================================================================

overall_ranking = (

    fidelity_score

    .merge(

        utility_score,

        on="Synthetic Model"

    )

    .merge(

        privacy_score,

        on="Synthetic Model"

    )

)

# =============================================================================
# Normalize Scores (0–100)
# =============================================================================

overall_ranking["Utility Score"] *= 100

if overall_ranking["Fidelity Score"].max() <= 1:

    overall_ranking["Fidelity Score"] *= 100

# =============================================================================
# Overall Score
# =============================================================================

overall_ranking["Overall Score"] = (

      FIDELITY_WEIGHT * overall_ranking["Fidelity Score"]

    + UTILITY_WEIGHT  * overall_ranking["Utility Score"]

    + PRIVACY_WEIGHT  * overall_ranking["Privacy Score"]

)

# =============================================================================
# Ranking
# =============================================================================

overall_ranking = (

    overall_ranking

    .sort_values(

        "Overall Score",

        ascending=False

    )

    .reset_index(drop=True)

)

overall_ranking.insert(

    0,

    "Rank",

    np.arange(

        1,

        len(overall_ranking)+1

    )

)

# =============================================================================
# Display Ranking
# =============================================================================

print("\nFINAL MODEL RANKING")

display(

    overall_ranking.round(2)

)

# =============================================================================
# Best Model
# =============================================================================

best = overall_ranking.iloc[0]

print("\n" + "="*80)

print("BEST OVERALL SYNTHETIC DATA GENERATOR")

print("="*80)

print(f"Rank            : {best['Rank']}")

print(f"Model           : {best['Synthetic Model']}")

print(f"Overall Score   : {best['Overall Score']:.2f}/100")

print(f"Fidelity Score  : {best['Fidelity Score']:.2f}")

print(f"Utility Score   : {best['Utility Score']:.2f}")

print(f"Privacy Score   : {best['Privacy Score']:.2f}")

# =============================================================================
# Ranking Figure
# =============================================================================

plt.figure(figsize=(9,5))

plt.bar(

    overall_ranking["Synthetic Model"],

    overall_ranking["Overall Score"]

)

plt.xticks(rotation=30)

plt.ylabel("Overall Score")

plt.title("Overall Ranking of Synthetic Data Generators")

plt.grid(axis="y", alpha=0.30)

plt.tight_layout()

plt.savefig(

    RESULTS_DIR /

    "evaluation" /

    "overall_model_ranking.png",

    dpi=600,

    bbox_inches="tight"

)

plt.show()

# =============================================================================
# Summary Statistics
# =============================================================================

print("\nSUMMARY")

print("-"*80)

print(f"Models Evaluated : {len(overall_ranking)}")

print(f"Top Ranked Model : {best['Synthetic Model']}")

print(f"Highest Score    : {best['Overall Score']:.2f}")

print("\nWeighting Scheme")

print(f"Fidelity : {FIDELITY_WEIGHT*100:.0f}%")

print(f"Utility  : {UTILITY_WEIGHT*100:.0f}%")

print(f"Privacy  : {PRIVACY_WEIGHT*100:.0f}%")

print("\n✓ Overall model ranking completed successfully.")

print("="*80)
print("SECTION 8.12 COMPLETED SUCCESSFULLY")
print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.13 : Publication Figures
# =============================================================================

print("="*80)
print("SECTION 8.13 : PUBLICATION FIGURES")
print("="*80)

PUBLICATION_DIR = RESULTS_DIR / "publication_figures"
PUBLICATION_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Figure 1 : Overall Ranking
# =============================================================================

plt.figure(figsize=(9,5))

plt.bar(
    overall_ranking["Synthetic Model"],
    overall_ranking["Overall Score"]
)

plt.xticks(rotation=30)

plt.ylabel("Overall Score")

plt.title("Overall Ranking of Synthetic Data Generators")

plt.grid(axis="y", alpha=.3)

plt.tight_layout()

plt.savefig(
    PUBLICATION_DIR/"Figure1_OverallRanking.png",
    dpi=600
)

plt.show()

# =============================================================================
# Figure 2 : Fidelity
# =============================================================================

plt.figure(figsize=(8,5))

plt.bar(
    fidelity_score["Synthetic Model"],
    fidelity_score["Fidelity Score"]
)

plt.xticks(rotation=30)

plt.ylabel("Fidelity Score")

plt.title("Statistical Fidelity Comparison")

plt.grid(axis="y", alpha=.3)

plt.tight_layout()

plt.savefig(
    PUBLICATION_DIR/"Figure2_Fidelity.png",
    dpi=600
)

plt.show()

# =============================================================================
# Figure 3 : Utility
# =============================================================================

plt.figure(figsize=(8,5))

utility_plot = utility_score.sort_values(
    "Utility Score",
    ascending=False
)

plt.bar(

    utility_plot["Synthetic Model"],

    utility_plot["Utility Score"]

)

plt.xticks(rotation=30)

plt.ylabel("Utility Score")

plt.title("Machine Learning Utility")

plt.grid(axis="y", alpha=.3)

plt.tight_layout()

plt.savefig(

    PUBLICATION_DIR/"Figure3_Utility.png",

    dpi=600

)

plt.show()

# =============================================================================
# Figure 4 : Privacy
# =============================================================================

plt.figure(figsize=(8,5))

privacy_plot = privacy_scores.sort_values(

    "Privacy Score",

    ascending=False

)

plt.bar(

    privacy_plot["Synthetic Model"],

    privacy_plot["Privacy Score"]

)

plt.xticks(rotation=30)

plt.ylabel("Privacy Score")

plt.title("Privacy Preservation Comparison")

plt.grid(axis="y", alpha=.3)

plt.tight_layout()

plt.savefig(

    PUBLICATION_DIR/"Figure4_Privacy.png",

    dpi=600

)

plt.show()

# =============================================================================
# Figure 5 : Radar Chart
# =============================================================================

metrics = [

    "Fidelity Score",

    "Utility Score",

    "Privacy Score"

]

radar = overall_ranking.copy()

angles = np.linspace(

    0,

    2*np.pi,

    len(metrics),

    endpoint=False

)

angles = np.concatenate((angles,[angles[0]]))

fig = plt.figure(figsize=(8,8))

ax = plt.subplot(111, polar=True)

for _, row in radar.iterrows():

    values = [

        row["Fidelity Score"],

        row["Utility Score"],

        row["Privacy Score"]

    ]

    values.append(values[0])

    ax.plot(

        angles,

        values,

        linewidth=2,

        label=row["Synthetic Model"]

    )

ax.set_xticks(angles[:-1])

ax.set_xticklabels(metrics)

ax.set_title("Overall Model Comparison")

ax.legend(

    bbox_to_anchor=(1.35,1.05),

    fontsize=8

)

plt.tight_layout()

plt.savefig(

    PUBLICATION_DIR/"Figure5_RadarChart.png",

    dpi=600

)

plt.show()

# =============================================================================
# Figure 6 : Heatmap
# =============================================================================

heat = overall_ranking.set_index(

    "Synthetic Model"

)[

    [

        "Fidelity Score",

        "Utility Score",

        "Privacy Score",

        "Overall Score"

    ]

]

plt.figure(figsize=(8,5))

plt.imshow(

    heat,

    aspect="auto"

)

plt.xticks(

    range(len(heat.columns)),

    heat.columns,

    rotation=25

)

plt.yticks(

    range(len(heat.index)),

    heat.index

)

plt.colorbar(label="Score")

plt.title("Overall Evaluation Heatmap")

plt.tight_layout()

plt.savefig(

    PUBLICATION_DIR/"Figure6_Heatmap.png",

    dpi=600

)

plt.show()

# =============================================================================
# Figure 7 : Leaderboard
# =============================================================================

leaderboard = overall_ranking.sort_values(

    "Overall Score",

    ascending=False

)

plt.figure(figsize=(10,6))

plt.barh(

    leaderboard["Synthetic Model"],

    leaderboard["Overall Score"]

)

plt.xlabel("Overall Score")

plt.title("Synthetic Data Generator Leaderboard")

plt.tight_layout()

plt.savefig(

    PUBLICATION_DIR/"Figure7_Leaderboard.png",

    dpi=600

)

plt.show()

# =============================================================================
# Summary
# =============================================================================

print("\nPublication Figures Generated")

figures = sorted(PUBLICATION_DIR.glob("*.png"))

for fig in figures:

    print(f"✓ {fig.name}")

print("\nTotal Figures :", len(figures))

print("\nOutput Folder")

print(PUBLICATION_DIR)

print("\n✓ Publication figures completed successfully.")

print("="*80)
print("SECTION 8.13 COMPLETED SUCCESSFULLY")
print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.14 : Publication Tables
# =============================================================================

print("="*80)
print("SECTION 8.14 : PUBLICATION TABLES")
print("="*80)

PUBLICATION_TABLE_DIR = RESULTS_DIR / "publication_tables"
PUBLICATION_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Table 1 : Statistical Fidelity
# =============================================================================

table1 = fidelity_score.copy()

table1 = table1.sort_values(

    "Fidelity Score",

    ascending=False

)

# =============================================================================
# Table 2 : Machine Learning Utility
# =============================================================================

table2 = utility_score.copy()

table2 = table2.sort_values(

    "Utility Score",

    ascending=False

)

# =============================================================================
# Table 3 : Privacy Evaluation
# =============================================================================

table3 = privacy_scores.copy()

table3 = table3.sort_values(

    "Privacy Score",

    ascending=False

)

# =============================================================================
# Table 4 : Overall Ranking
# =============================================================================

table4 = overall_ranking.copy()

# =============================================================================
# Table 5 : Statistical Significance
# =============================================================================

table5 = friedman_summary.copy()

# =============================================================================
# Display Tables
# =============================================================================

tables = {

    "Table 1 : Fidelity Comparison"        : table1,

    "Table 2 : Utility Comparison"         : table2,

    "Table 3 : Privacy Comparison"         : table3,

    "Table 4 : Overall Model Ranking"      : table4,

    "Table 5 : Statistical Significance"   : table5

}

for title, df in tables.items():

    print("\n" + "="*80)

    print(title)

    print("="*80)

    display(df.round(4))

# =============================================================================
# Save Individual CSV Files
# =============================================================================

table1.to_csv(

    PUBLICATION_TABLE_DIR /

    "Table1_Fidelity.csv",

    index=False

)

table2.to_csv(

    PUBLICATION_TABLE_DIR /

    "Table2_Utility.csv",

    index=False

)

table3.to_csv(

    PUBLICATION_TABLE_DIR /

    "Table3_Privacy.csv",

    index=False

)

table4.to_csv(

    PUBLICATION_TABLE_DIR /

    "Table4_OverallRanking.csv",

    index=False

)

table5.to_csv(

    PUBLICATION_TABLE_DIR /

    "Table5_Statistics.csv",

    index=False

)

# =============================================================================
# Save Excel Workbook
# =============================================================================

publication_excel = PUBLICATION_TABLE_DIR / "Publication_Tables.xlsx"

with pd.ExcelWriter(

    publication_excel,

    engine="openpyxl"

) as writer:

    table1.to_excel(

        writer,

        sheet_name="Table1_Fidelity",

        index=False

    )

    table2.to_excel(

        writer,

        sheet_name="Table2_Utility",

        index=False

    )

    table3.to_excel(

        writer,

        sheet_name="Table3_Privacy",

        index=False

    )

    table4.to_excel(

        writer,

        sheet_name="Table4_Ranking",

        index=False

    )

    table5.to_excel(

        writer,

        sheet_name="Table5_Statistics",

        index=False

    )

# =============================================================================
# Summary
# =============================================================================

print("\nPublication Tables Generated")

files = sorted(PUBLICATION_TABLE_DIR.glob("*"))

for file in files:

    print(f"✓ {file.name}")

print("\nTotal Files :", len(files))

print("\nOutput Directory")

print(PUBLICATION_TABLE_DIR)

print("\n✓ Publication tables generated successfully.")

print("="*80)
print("SECTION 8.14 COMPLETED SUCCESSFULLY")
print("="*80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.15 : Export Complete Evaluation
# =============================================================================

print("=" * 80)
print("SECTION 8.15 : EXPORT COMPLETE EVALUATION")
print("=" * 80)

# =============================================================================
# Output Directory
# =============================================================================

EXPORT_DIR = RESULTS_DIR / "complete_evaluation"

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# Export Excel Workbook
# =============================================================================

excel_path = EXPORT_DIR / "Complete_Evaluation.xlsx"

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    # -------------------------------------------------------------------------
    # Descriptive Statistics
    # -------------------------------------------------------------------------
    if "descriptive_statistics" in globals():
        descriptive_statistics.to_excel(
            writer,
            sheet_name="Descriptive Statistics",
            index=False
        )

    # -------------------------------------------------------------------------
    # Distribution Similarity
    # -------------------------------------------------------------------------
    if "distribution_results" in globals():
        distribution_results.to_excel(
            writer,
            sheet_name="Distribution Similarity",
            index=False
        )

    # -------------------------------------------------------------------------
    # Correlation Preservation
    # -------------------------------------------------------------------------
    if "correlation_summary" in globals():
        correlation_summary.to_excel(
            writer,
            sheet_name="Correlation",
            index=False
        )

    # -------------------------------------------------------------------------
    # Multivariate Similarity
    # -------------------------------------------------------------------------
    if "multivariate_summary" in globals():
        multivariate_summary.to_excel(
            writer,
            sheet_name="Multivariate",
            index=False
        )

    # -------------------------------------------------------------------------
    # Machine Learning Utility
    # -------------------------------------------------------------------------
    utility_results.to_excel(
        writer,
        sheet_name="Utility",
        index=False
    )

    generator_summary.to_excel(
        writer,
        sheet_name="Utility Ranking",
        index=False
    )

    # -------------------------------------------------------------------------
    # Privacy Evaluation
    # -------------------------------------------------------------------------
    membership_results.to_excel(
        writer,
        sheet_name="Membership Attack",
        index=False
    )

    attribute_results.to_excel(
        writer,
        sheet_name="Attribute Disclosure",
        index=False
    )

    reidentification_results.to_excel(
        writer,
        sheet_name="Reidentification",
        index=False
    )

    distance_results.to_excel(
        writer,
        sheet_name="Distance Metrics",
        index=False
    )

    privacy_scores.to_excel(
        writer,
        sheet_name="Privacy Scores",
        index=False
    )

    # -------------------------------------------------------------------------
    # Statistical Tests
    # -------------------------------------------------------------------------
    ttest_results.to_excel(
        writer,
        sheet_name="Paired t-Test",
        index=False
    )

    wilcoxon_results.to_excel(
        writer,
        sheet_name="Wilcoxon",
        index=False
    )

    friedman_summary.to_excel(
        writer,
        sheet_name="Friedman",
        index=False
    )

    # -------------------------------------------------------------------------
    # Overall Ranking
    # -------------------------------------------------------------------------
    overall_ranking.to_excel(
        writer,
        sheet_name="Overall Ranking",
        index=False
    )

print("✓ Excel workbook exported.")

# =============================================================================
# Export Major CSV Files
# =============================================================================

csv_tables = {

    "utility_results.csv": utility_results,
    "privacy_scores.csv": privacy_scores,
    "overall_ranking.csv": overall_ranking,
    "membership_results.csv": membership_results,
    "attribute_results.csv": attribute_results,
    "reidentification_results.csv": reidentification_results,
    "distance_results.csv": distance_results,
    "ttest_results.csv": ttest_results,
    "wilcoxon_results.csv": wilcoxon_results,
    "friedman_results.csv": friedman_summary

}

for filename, dataframe in csv_tables.items():

    dataframe.to_csv(
        EXPORT_DIR / filename,
        index=False
    )

print("✓ CSV files exported.")

# =============================================================================
# Export Figure Inventory
# =============================================================================

figure_files = []

for folder in [

    RESULTS_DIR / "evaluation",
    RESULTS_DIR / "publication_figures"

]:

    if folder.exists():

        figure_files.extend(folder.rglob("*.png"))

figure_inventory = pd.DataFrame({

    "Figure": [f.name for f in figure_files],

    "Location": [str(f) for f in figure_files]

})

figure_inventory.to_csv(

    EXPORT_DIR / "Figure_Inventory.csv",

    index=False

)

# =============================================================================
# Export Summary
# =============================================================================

print("\nFILES CREATED")
print("-" * 80)

for file in sorted(EXPORT_DIR.iterdir()):

    print(f"✓ {file.name}")

print("\nExport Location")

print(EXPORT_DIR)

print("\nTotal Files :", len(list(EXPORT_DIR.iterdir())))

print("\n✓ Complete evaluation exported successfully.")

print("=" * 80)
print("SECTION 8.15 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# =============================================================================
# Notebook 08 : Comparative Evaluation
# Section 8.16 : Final Notebook Summary
# =============================================================================

print("=" * 90)
print("NOTEBOOK 08 : COMPARATIVE EVALUATION")
print("FINAL SUMMARY")
print("=" * 90)

# =============================================================================
# Best Models
# =============================================================================

best_fidelity = fidelity_score.loc[
    fidelity_score["Fidelity Score"].idxmax()
]

best_utility = utility_score.loc[
    utility_score["Utility Score"].idxmax()
]

best_privacy = privacy_scores.loc[
    privacy_scores["Privacy Score"].idxmax()
]

best_overall = overall_ranking.iloc[0]

# =============================================================================
# Final Summary Table
# =============================================================================

final_summary = pd.DataFrame({

    "Evaluation Category":[

        "Statistical Fidelity",

        "Machine Learning Utility",

        "Privacy Preservation",

        "Overall Performance"

    ],

    "Best Generator":[

        best_fidelity["Synthetic Model"],

        best_utility["Synthetic Model"],

        best_privacy["Synthetic Model"],

        best_overall["Synthetic Model"]

    ],

    "Score":[

        round(best_fidelity["Fidelity Score"],2),

        round(best_utility["Utility Score"],2),

        round(best_privacy["Privacy Score"],2),

        round(best_overall["Overall Score"],2)

    ]

})

print("\nFINAL EVALUATION SUMMARY\n")

display(final_summary)

# =============================================================================
# Publication Conclusion
# =============================================================================

print("\n" + "="*90)
print("PUBLICATION-READY CONCLUSION")
print("="*90)

print(
    f"""
The comparative evaluation investigated six synthetic tabular data generators:
Gaussian Multivariate, Gaussian Copula, CTGAN, TVAE, DP-CTGAN, and the proposed
SPP-GAN framework.

The evaluation considered three major perspectives:

• Statistical Fidelity
• Machine Learning Utility
• Privacy Preservation

The overall ranking integrated these dimensions using a weighted scoring
framework consisting of:

   • Fidelity  : 40%
   • Utility   : 30%
   • Privacy   : 30%

Based on the comprehensive evaluation, the highest overall score was achieved by

>>> {best_overall['Synthetic Model']}

Overall Score : {best_overall['Overall Score']:.2f}/100

The results indicate that this generator provides the best trade-off between

✓ Data Fidelity
✓ Downstream Machine Learning Utility
✓ Privacy Preservation

making it the most suitable model for privacy-preserving synthetic tabular data
generation among the evaluated approaches.
"""
)

# =============================================================================
# Save Final Summary
# =============================================================================

summary_file = EXPORT_DIR / "Final_Notebook_Summary.csv"

final_summary.to_csv(
    summary_file,
    index=False
)

# =============================================================================
# Completion
# =============================================================================

print("\n" + "="*90)
print("NOTEBOOK 08 COMPLETED SUCCESSFULLY")
print("="*90)

print(f"""
Datasets Evaluated          : {len(DATASETS)}

Synthetic Generators        : {len(MODELS)}

Best Fidelity Model         : {best_fidelity['Synthetic Model']}

Best Utility Model          : {best_utility['Synthetic Model']}

Best Privacy Model          : {best_privacy['Synthetic Model']}

Best Overall Model          : {best_overall['Synthetic Model']}

Overall Score               : {best_overall['Overall Score']:.2f}/100

Final Summary Saved To

{summary_file}
""")

print("="*90)
print("END OF NOTEBOOK 08")
print("="*90)